# Step 1: Data Preprocessing and Exploratory Behavioral Analysis

In [ ]:
# -*- coding: utf-8 -*-
"""
=============================================================================
STEP 1: Data Preprocessing and Exploratory Behavioral Analysis
=============================================================================
Pipeline Position: ENTRY POINT (no upstream dependencies)
Downstream Consumers: Step 2a (reads 'hddm_data_unfair.csv' and
                      'data_fingerprint.json')
=============================================================================
"""

import os
import json
import hashlib
from datetime import datetime
import logging
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patches as mpatches
from matplotlib.ticker import FuncFormatter, PercentFormatter

# =============================================================================
# GLOBAL CONSTANTS & AESTHETICS
# =============================================================================
sns.set_theme(style="ticks", palette="colorblind")
plt.rcParams.update({
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 10,
    'axes.labelsize': 12,
    'axes.titlesize': 12,
    'axes.titleweight': 'bold',
    'legend.frameon': False,
    'pdf.fonttype': 42
})

ACC_COLOR = "#004D40"
REJ_COLOR = "#4A148C"
ACC_ALPHA = 0.52
REJ_ALPHA = 0.75

# Reaction Time (RT) boundaries in milliseconds.
# Minimum RT set to 300ms to accommodate complex social cognition
# processing time in the Ultimatum Game paradigm.
RT_MIN_MS = 300
RT_MAX_MS = 3000

# Behavioral response coding in the original experimental data
RESPONSE_ACCEPT = 1
RESPONSE_REJECT = 2
RESPONSE_NONE = 0

# HDDM response coding boundary mapping:
# Upper boundary (1) = Accept; Lower boundary (0) = Reject
HDDM_ACCEPT = 1
HDDM_REJECT = 0


# =============================================================================
# LOGGING SETUP
# =============================================================================
def setup_logger(log_file: str = "step1_data_preparation.log") -> logging.Logger:
    """Initialize logging configuration for process tracking."""
    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s %(levelname)s: %(message)s',
        handlers=[
            logging.FileHandler(log_file, encoding='utf-8'),
            logging.StreamHandler()
        ]
    )
    return logging.getLogger(__name__)


# =============================================================================
# DATA FINGERPRINTING & LINEAGE
# =============================================================================
def compute_file_hash(filepath: str) -> str:
    """
    Computes the SHA-256 cryptographic hash of a specified file.
    Returns an empty string if the file is inaccessible.
    """
    if not os.path.exists(filepath):
        return ""
    sha256_hash = hashlib.sha256()
    with open(filepath, "rb") as f:
        for byte_block in iter(lambda: f.read(4096), b""):
            sha256_hash.update(byte_block)
    return sha256_hash.hexdigest()


def generate_data_fingerprint(
    df: pd.DataFrame,
    source_filepath: str,
    output_filepath: str,
    logger: logging.Logger
) -> None:
    """
    Extracts structural metadata and computes cryptographic hashes for
    both the source data and the final analytical matrix. Serializes the
    artifact to a JSON file for downstream pipeline validation.
    """
    logger.info("--- Generating Cryptographic Data Fingerprint ---")

    source_hash = compute_file_hash(source_filepath)
    output_hash = compute_file_hash(output_filepath)

    fingerprint = {
        "data_file_name": os.path.basename(output_filepath),
        "data_file_hash": output_hash,
        "source_file_hash": source_hash,
        "n_rows": int(len(df)),
        "n_subjects": int(df['subj_idx'].nunique()),
        "emotion_levels": sorted(df['emotion'].unique().tolist()),
        "response_levels": sorted(df['response'].unique().tolist()),
        "rt_min_sec": float(df['rt'].min()),
        "rt_max_sec": float(df['rt'].max()),
        "created_at_utc": datetime.utcnow().isoformat() + "Z",
        "preprocessing_signature": {
            "task_scope": "unfair_only",
            "emotion_recode": "enj->rew",
            "exclusion_rule_rt_min_ms": RT_MIN_MS,
            "exclusion_rule_rt_max_ms": RT_MAX_MS,
            "response_mapping": "Accept=1, Reject=0"
        }
    }

    fingerprint_path = os.path.join(
        os.path.dirname(output_filepath) or ".", "data_fingerprint.json"
    )
    with open(fingerprint_path, 'w', encoding='utf-8') as f:
        json.dump(fingerprint, f, indent=4)

    logger.info(f"Data fingerprint serialized to '{fingerprint_path}'.")
    logger.info(f"Target Lineage Hash: {output_hash[:16]}...")


# =============================================================================
# MODULE 1: RESPONSE CODING AUDIT
# =============================================================================
def audit_response_coding(
    df: pd.DataFrame, logger: logging.Logger
) -> pd.DataFrame:
    """
    Strictly audits the mapping between original button presses and
    HDDM boundary coding. Enforces bijective mapping:
      Original Accept (1) -> HDDM Upper Boundary (1)
      Original Reject (2) -> HDDM Lower Boundary (0)
    """
    logger.info("--- Executing Response Coding Audit ---")

    orig_accept = (df['reaction'] == RESPONSE_ACCEPT).sum()
    orig_reject = (df['reaction'] == RESPONSE_REJECT).sum()

    response_mapping = {RESPONSE_ACCEPT: HDDM_ACCEPT,
                        RESPONSE_REJECT: HDDM_REJECT}
    df['response_hddm'] = df['reaction'].map(response_mapping)

    hddm_accept = (df['response_hddm'] == HDDM_ACCEPT).sum()
    hddm_reject = (df['response_hddm'] == HDDM_REJECT).sum()

    if orig_accept != hddm_accept or orig_reject != hddm_reject:
        raise ValueError("CRITICAL: Response mapping mismatch detected!")

    if df['response_hddm'].isnull().any():
        unmapped = df.loc[df['response_hddm'].isnull(), 'reaction'].unique()
        raise ValueError(f"Unmapped response values detected: {unmapped}")

    # Explicit binary validation
    unique_resp = set(df['response_hddm'].unique())
    if not unique_resp.issubset({0, 1}):
        raise ValueError(
            f"HDDM response column contains non-binary values: {unique_resp}"
        )

    audit_data = [{
        'Original_Response': 'Accept (1)', 'Original_Count': orig_accept,
        'HDDM_Boundary': 'Upper (1)', 'HDDM_Count': hddm_accept
    }, {
        'Original_Response': 'Reject (2)', 'Original_Count': orig_reject,
        'HDDM_Boundary': 'Lower (0)', 'HDDM_Count': hddm_reject
    }]

    pd.DataFrame(audit_data).to_csv("response_coding_audit.csv", index=False)
    logger.info(
        "Response coding audit passed and exported to "
        "'response_coding_audit.csv'."
    )
    return df


# =============================================================================
# MODULE 2: FAIR-CEILING DIAGNOSTICS & ROBUSTNESS CHECKS
# =============================================================================
def generate_fair_ceiling_diagnostics(
    df_valid: pd.DataFrame, logger: logging.Logger
):
    """
    Calculates rejection rates across Fair and Unfair conditions to
    justify targeting Unfair trials exclusively. Ceiling effects in
    Fair offers (near 100% acceptance) would violate DDM variance
    assumptions and cause MCMC convergence failure.
    """
    logger.info("--- Generating Fair-Ceiling Diagnostics ---")

    df_valid = df_valid.copy()
    df_valid['Condition_Type'] = np.where(
        df_valid['Offers_You'] <= 2, 'Unfair (9:1, 8:2)',
        np.where(
            df_valid['Offers_You'] >= 4, 'Fair (5:5, 6:4)', 'Intermediate'
        )
    )

    df_target = df_valid[
        df_valid['Condition_Type'].isin(
            ['Unfair (9:1, 8:2)', 'Fair (5:5, 6:4)']
        )
    ]

    summary = df_target.groupby(['Condition_Type', 'emotion']).apply(
        lambda x: pd.Series({
            'Total_Trials': len(x),
            'Rejection_Rate': (x['reaction'] == RESPONSE_REJECT).mean(),
            'RT_Mean': x['RT'].mean() / 1000.0
        })
    ).reset_index()

    summary.to_csv("fair_ceiling_diagnostics.csv", index=False)
    logger.info(
        "Fair-ceiling behavior summary exported to "
        "'fair_ceiling_diagnostics.csv'."
    )

    plt.figure(figsize=(8, 5))
    sns.barplot(
        data=summary, x='emotion', y='Rejection_Rate',
        hue='Condition_Type',
        palette=['#4A148C', '#900C3F'], alpha=0.85
    )
    plt.title("Empirical Rejection Rates: Fair vs. Unfair Offers", pad=15)
    plt.ylabel("Probability of Rejection")
    plt.xlabel("Emotion Condition")
    plt.gca().yaxis.set_major_formatter(PercentFormatter(1.0))
    plt.ylim(0, 1.05)
    plt.legend(title="Offer Type", loc='upper left',
               bbox_to_anchor=(1, 1))
    plt.tight_layout()
    plt.savefig("fair_unfair_rejection_rate.pdf", dpi=300)
    plt.close()


def generate_offer_ratio_robustness(
    df_unfair: pd.DataFrame, logger: logging.Logger
):
    """
    Validates whether 9:1 and 8:2 ratios behave similarly enough
    to merge into a single 'unfair' category for HDDM estimation.
    """
    logger.info("--- Generating 9:1 vs 8:2 Robustness Check ---")

    summary = df_unfair.groupby(['Offers_You', 'emotion']).apply(
        lambda x: pd.Series({
            'Total_Trials': len(x),
            'Rejection_Rate': (x['reaction'] == RESPONSE_REJECT).mean(),
            'RT_Mean': x['RT'].mean() / 1000.0
        })
    ).reset_index()

    summary['Offer_Ratio'] = summary['Offers_You'].map({1: '9:1', 2: '8:2'})
    summary.to_csv("unfair_offer_ratio_behavior_summary.csv", index=False)
    logger.info(
        "Offer ratio robustness check exported to "
        "'unfair_offer_ratio_behavior_summary.csv'."
    )


# =============================================================================
# MODULE 3: PER-SUBJECT TRIAL COUNT AUDIT
# =============================================================================
def audit_trial_counts(
    hddm_df: pd.DataFrame, logger: logging.Logger
) -> pd.DataFrame:
    """
    Generates a subject x condition trial count matrix to verify that
    each experimental cell meets the 30-40 trial minimum required for
    stable four-parameter DDM estimation without across-trial variability
    (Lerche & Voss, 2016, Behavior Research Methods).

    Flags subjects with fewer than 20 trials in any condition as
    candidates for exclusion in sensitivity analyses.
    """
    logger.info("--- Auditing Per-Subject Trial Counts ---")

    trial_counts = hddm_df.groupby(
        ['subj_idx', 'emotion']
    ).size().reset_index(name='n_trials')

    pivot = trial_counts.pivot(
        index='subj_idx', columns='emotion', values='n_trials'
    ).fillna(0).astype(int)

    pivot['min_across_conditions'] = pivot.min(axis=1)
    pivot['total_trials'] = pivot.drop(
        columns='min_across_conditions'
    ).sum(axis=1)

    # Flag subjects below minimum threshold
    MINIMUM_TRIALS_PER_CONDITION = 20
    pivot['below_threshold'] = (
        pivot['min_across_conditions'] < MINIMUM_TRIALS_PER_CONDITION
    )

    n_flagged = pivot['below_threshold'].sum()
    if n_flagged > 0:
        logger.warning(
            f"  {n_flagged} subject(s) have fewer than "
            f"{MINIMUM_TRIALS_PER_CONDITION} trials in at least one "
            f"condition. Consider exclusion in sensitivity analyses."
        )
    else:
        logger.info(
            f"  All subjects meet the minimum trial threshold "
            f"({MINIMUM_TRIALS_PER_CONDITION} trials/condition)."
        )

    pivot.to_csv("subject_trial_count_audit.csv")
    logger.info("Per-subject trial counts exported to "
                "'subject_trial_count_audit.csv'.")

    # Summary statistics
    logger.info(f"  Trial count range: "
                f"{int(pivot['min_across_conditions'].min())} - "
                f"{int(pivot['min_across_conditions'].max())} "
                f"(min across conditions per subject)")
    logger.info(f"  Grand mean trials per cell: "
                f"{trial_counts['n_trials'].mean():.1f}")

    return trial_counts


# =============================================================================
# DATA LOADING, FILTERING AND HDDM INGESTION
# =============================================================================
def load_and_filter_data(
    filepath: str, logger: logging.Logger
) -> pd.DataFrame:
    """
    Loads raw data, applies exclusion criteria, and standardizes labels.
    Exclusion pipeline:
      1. Standardize emotion labels ('enj' -> 'rew')
      2. Remove omitted responses (reaction == 0)
      3. Remove fast RTs (< 300ms)
      4. Remove slow RTs (> 3000ms)
      5. Select unfair offers only (Offers_You in {1, 2})
    """
    logger.info(f"Loading raw data from {filepath}")

    df = pd.read_csv(filepath)
    logger.info(f"Initial raw data dimensions: {df.shape}")

    exclusion_log = [{'Stage': 'Total_Initial_Trials', 'Count': len(df)}]

    # 1. Standardize emotion legacy labels: 'enj' -> 'rew'
    df['emotion'] = (
        df['emotion'].astype(str).str.strip().replace({'enj': 'rew'})
    )

    # 2. Missing responses
    omitted_mask = df['reaction'] == RESPONSE_NONE
    df_valid_resp = df[~omitted_mask]
    exclusion_log.append({
        'Stage': 'Omitted_Responses', 'Count': omitted_mask.sum()
    })

    # 3. RT boundaries (lower)
    fast_mask = df_valid_resp['RT'] < RT_MIN_MS
    df_valid_rt_low = df_valid_resp[~fast_mask]
    exclusion_log.append({
        'Stage': 'RT_Too_Fast', 'Count': fast_mask.sum()
    })

    # 4. RT boundaries (upper)
    slow_mask = df_valid_rt_low['RT'] > RT_MAX_MS
    df_valid_rt = df_valid_rt_low[~slow_mask]
    exclusion_log.append({
        'Stage': 'RT_Too_Slow', 'Count': slow_mask.sum()
    })

    # Trigger diagnostic before filtering fairness
    generate_fair_ceiling_diagnostics(df_valid_rt, logger)

    # 5. Target Unfair Condition Selection (Offers_You == 1 or 2)
    fair_mask = ~df_valid_rt['Offers_You'].isin([1, 2])
    df_unfair = df_valid_rt[~fair_mask].copy()
    exclusion_log.append({
        'Stage': 'Non_Unfair_Offers_Excluded', 'Count': fair_mask.sum()
    })
    exclusion_log.append({
        'Stage': 'Final_Retained_Unfair_Trials', 'Count': len(df_unfair)
    })

    pd.DataFrame(exclusion_log).to_csv(
        "exclusion_summary_flow.csv", index=False
    )
    logger.info(f"Retained trials (Unfair conditions only): {len(df_unfair)}")

    df_unfair = audit_response_coding(df_unfair, logger)
    generate_offer_ratio_robustness(df_unfair, logger)

    return df_unfair


def prepare_hddm_data(
    df: pd.DataFrame, logger: logging.Logger
) -> pd.DataFrame:
    """
    Constructs the canonical data matrix required for HDDM estimation.
    Column schema:
      subj_idx  - Integer subject identifier (HDDM requirement)
      rt        - Reaction time in seconds (HDDM requirement)
      response  - Binary {0, 1} boundary coding (HDDM requirement)
      emotion   - Experimental condition factor
      offer_amount - Offer magnitude (retained for potential covariates)
    """
    df = df.copy()

    unique_ids = df['participant_id'].unique()
    id_map = {orig_id: idx for idx, orig_id in enumerate(unique_ids)}
    df['subj_idx'] = df['participant_id'].map(id_map)

    pd.DataFrame(
        list(id_map.items()),
        columns=['Original_participant_id', 'HDDM_subj_idx']
    ).to_csv('subject_mapping.csv', index=False)

    hddm_df = pd.DataFrame({
        'subj_idx': df['subj_idx'],
        'rt': df['RT'] / 1000.0,
        'response': df['response_hddm'],
        'emotion': df['emotion'],
        'offer_amount': df['Offers_You']
    })

    return hddm_df


# =============================================================================
# PIPELINE EXECUTION
# =============================================================================
def main():
    logger = setup_logger()
    logger.info("=" * 60)
    logger.info("HDDM DATA PREPARATION PIPELINE INITIATED")
    logger.info("=" * 60)

    try:
        input_file = 'trials.csv'
        output_file = 'hddm_data_unfair.csv'

        # Core data processing
        df_unfair = load_and_filter_data(input_file, logger)
        hddm_df = prepare_hddm_data(df_unfair, logger)

        # Trial count adequacy audit
        audit_trial_counts(hddm_df, logger)

        # Export final matrix
        hddm_df.to_csv(output_file, index=False)
        logger.info(f"Final analytical dataset committed to {output_file}")

        # Lineage integration
        generate_data_fingerprint(
            hddm_df, input_file, output_file, logger
        )

        # Verify emotion categories
        emotions_present = hddm_df['emotion'].unique()
        logger.info(f"Emotions preserved for modeling: {emotions_present}")

    except Exception as e:
        logger.error(f"Fatal error encountered: {e}")
        raise


main()

# Step 2a: Global Configuration

In [ ]:
# -*- coding: utf-8 -*-
"""
=============================================================================
STEP 2a: Global Configuration & Cryptographic Lineage (ArviZ-Centric SSOT)
=============================================================================
Pipeline Position: Reads 'data_fingerprint.json' from Step 1.
                   Generates 'hddm_config.py' consumed by Steps 2b-6.

Downstream Consumers: Every subsequent Step imports from hddm_config.py:
  - CFG object (all hyperparameters)
  - load_active_lineage_state() (lineage validation)
  - validate_artifact_lineage() (artifact hash checking)
  - identify_winning_model() (Step 5-6 model routing)
  - load_existing_manifest() (Step 2b checkpoint resume)
  - append_manifest_record() (Step 2b checkpoint persistence)
  - check_memory_headroom() (Step 2b memory safety)

Methodological Basis:
  - Four-parameter DDM (v, a, t, z) without across-trial variability
    (sv, st, sz), per Lerche & Voss (2016) and Boehm et al. (2018)
    recommendations for per-condition trial counts of 30-40.
  - group_only_regressors=True: Treatment contrast coefficients are
    estimated at the group level (fixed effects). Subject-level
    variability is captured by hierarchical DDM base parameters.
    keep_regressor_trace=False: Trial-level regressor traces are
    discarded after sampling to prevent memory exhaustion during
    InferenceData conversion. Group-level posteriors (Intercepts
    and Treatment contrasts) are fully preserved.
    (Wiecki, Sofer & Frank, 2013, Frontiers in Human Neuroscience)
  - Stratified convergence criteria (Vehtari et al., 2021):
    focal parameters require strict ESS, nuisance parameters relaxed.

=============================================================================
"""

import os
import re
import json
import hashlib
from datetime import datetime
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Any, Union


# =============================================================================
# SHARED UTILITY: Lineage Validation (used by all downstream Steps)
# =============================================================================
def load_active_lineage_state(paths: Dict[str, Path], logger=None) -> dict:
    """
    Loads the active pipeline lineage state from the configuration
    fingerprint generated by Step 2a. Returns the dictionary containing
    config_hash, data_hash, and pipeline_hash.

    This function replaces the non-existent 'validate_pipeline_lineage'
    referenced in the original pipeline. All downstream Steps now
    import and call this function consistently.
    """
    fingerprint_path = paths['manifests'] / "config_fingerprint.json"
    if not fingerprint_path.exists():
        raise FileNotFoundError(
            "Missing configuration fingerprint. Execute Step 2a first."
        )

    with open(fingerprint_path, 'r', encoding='utf-8') as f:
        lineage = json.load(f)

    if logger:
        logger.info(
            f"Active Lineage loaded: "
            f"{lineage.get('pipeline_hash', '')[:16]}..."
        )

    return lineage


def validate_artifact_lineage(
    artifact_df, active_lineage: dict, logger=None
) -> str:
    """
    Strictly validates an artifact's lineage against the active
    environment. Checks for the existence of lineage columns and
    enforces hash equality.
    """
    required_fields = ["config_hash", "data_hash", "pipeline_hash"]
    missing = [
        col for col in required_fields
        if col not in artifact_df.columns
    ]

    if missing:
        raise KeyError(
            f"Strict Contract Failure: Artifact missing "
            f"lineage fields {missing}"
        )

    artifact_hash = str(artifact_df['pipeline_hash'].iloc[0])

    if artifact_hash != active_lineage['pipeline_hash']:
        raise RuntimeError(
            f"CRITICAL LINEAGE MISMATCH! Artifact does not belong "
            f"to the current environment.\n"
            f"Artifact Hash: {artifact_hash}\n"
            f"Active Hash: {active_lineage['pipeline_hash']}\n"
            f"Please re-run the pipeline from the out-of-sync step."
        )

    if logger:
        logger.info(
            "Artifact cryptographic lineage validated successfully."
        )

    return artifact_hash


def identify_winning_model(paths: Dict[str, Path]) -> str:
    """
    Parses the Step 4 audit trail to extract the winning model name.
    Strict contract: Demands explicit Is_Winner == True.
    """
    import pandas as pd

    audit_path = paths['audit'] / "final_model_selection_audit.csv"
    if not audit_path.exists():
        raise FileNotFoundError(
            "No audit manifest found. Ensure Step 4 completed."
        )

    df = pd.read_csv(audit_path)
    if df.empty:
        raise RuntimeError("Audit log is empty.")

    if 'Is_Winner' not in df.columns:
        raise KeyError(
            "Strict Contract Failure: 'Is_Winner' column missing "
            "in audit file."
        )

    mask = (
        df["Is_Winner"].astype(str).str.strip().str.lower() == "true"
    )
    winners = df[mask]

    if winners.empty:
        raise ValueError(
            "Strict Contract Failure: No explicit winner flagged."
        )

    if len(winners) > 1:
        raise ValueError(
            "Strict Contract Failure: Multiple winners flagged."
        )

    return str(winners.iloc[0]["model_name"]).lower()


# =============================================================================
# CORE CONFIGURATION DATACLASS
# =============================================================================
@dataclass
class HDDMConfig:
    """
    Analytical configuration schema for the dockerHDDM ArviZ-centric
    pipeline. Manages MCMC hyperparameters, convergence criteria,
    model architecture definitions, and PPC adequacy thresholds.
    """

    # -------------------------------------------------------------------------
    # 1. OPERATIONAL MODE & ROUTING
    # -------------------------------------------------------------------------
    run_mode: str = 'final'    # debug or final
    base_dir: Path = Path(os.getcwd())

    # -------------------------------------------------------------------------
    # 2. STRUCTURAL MODELING CONSTRAINTS
    # -------------------------------------------------------------------------
    # Four-parameter DDM: v, a, t are estimated by default in HDDM;
    # z (starting point bias) requires explicit inclusion via include.
    # sv, st, sz excluded per Lerche & Voss (2016) given 30-40
    # trials per condition.
    #
    # NAMING CLARIFICATION (single source of truth):
    # include_params specifies parameters beyond v, a, t that HDDM
    # does not estimate by default and must be explicitly requested.
    # In Step 2b, full_include=['v','a','t','z'] is the complete
    # parameter set passed to HDDMRegressor(include=...). This is
    # NOT redundant with include_params: full_include is a
    # dockerHDDM compatibility requirement (all 4 parameters must
    # be explicitly listed as regression nodes to avoid
    # AttributeError in wiener_multi_like), while include_params
    # documents which parameters are non-default additions.
    include_params: List[str] = field(
        default_factory=lambda: ['z']
    )

    # group_only_regressors: when True (the HDDMRegressor software default;
    # Pan et al., 2025), the treatment-contrast (emotion) coefficients are
    # estimated at the group level, while the DDM base parameters (v, a, t, z
    # intercepts) retain their subject-level hierarchical (random-intercept)
    # structure; setting it to False would additionally estimate subject-level
    # random slopes for the contrasts. Group-level estimation suits the present
    # design: with N = 30 and the available trials per cell, subject-specific
    # emotion slopes are not stably estimable. Because a hierarchical DDM
    # already partitions between-subject variance across four latent dimensions
    # (v, a, t, z intercepts) plus the RT distribution, much of the
    # choice-level heterogeneity that a single-outcome logistic GLMM must
    # absorb into random slopes is here carried by these random intercepts and
    # the RT constraints; the resulting group-level drift effects are
    # cross-validated against the behavioral GLMM, which does include
    # by-subject emotion slopes (Wiecki, Sofer & Frank, 2013; Pan et al., 2025).
    group_only_regressors: bool = True

    keep_regressor_trace: bool = False
    p_outlier: float = 0.05
    use_informative_priors: bool = True
    baseline_condition: str = 'neu'

    # -------------------------------------------------------------------------
    # 3. STRATIFIED MCMC DIAGNOSTIC THRESHOLDS (Vehtari et al., 2021)
    # -------------------------------------------------------------------------
    # Focal Parameters (Group-level Intercepts and Treatment contrasts):
    rhat_focal: float = 1.01
    ess_bulk_focal: float = 1000.0
    ess_tail_focal: float = 500.0

    # Nuisance Parameters (Subject-level deviations, transformed scales):
    rhat_nuisance: float = 1.05
    ess_bulk_nuisance: float = 100.0
    ess_tail_nuisance: float = 200.0

    # -------------------------------------------------------------------------
    # 4. EXPERIMENTAL DESIGN & VISUAL STANDARDS
    #    (Excluded from cryptographic hash computation)
    # -------------------------------------------------------------------------
    emotion_order: List[str] = field(
        default_factory=lambda: ['neu', 'rew', 'aff', 'dom', 'dis']
    )

    display_labels: Dict[str, str] = field(
        default_factory=lambda: {
            'neu': 'Neutral', 'rew': 'Reward',
            'aff': 'Affiliative', 'dom': 'Dominance',
            'dis': 'Disgust'
        }
    )

    # Okabe-Ito inspired colorblind-friendly palette
    colors: Dict[str, str] = field(
        default_factory=lambda: {
            'neu': "#8491B4", 'rew': "#3C5488",
            'aff': "#91D1C2", 'dom': "#F39B7F",
            'dis': "#E64B35"
        }
    )

    # -------------------------------------------------------------------------
    # 5. BASE MCMC HYPERPARAMETERS (dockerHDDM ArviZ-centric)
    # -------------------------------------------------------------------------
    # Unified seed attribute name. All downstream Steps
    # must reference CFG.base_seed (not 'random_seed').
    base_seed: int = 2508

    # n_chains is initialized to a placeholder here.
    # The actual value is computed in __post_init__ based on
    # self.run_mode, because Python dataclass default values are
    # evaluated at class definition time (not at instantiation),
    # so the conditional expression 'n_chains: int = 2 if
    # run_mode == "debug" else 4' in the original code ALWAYS
    # evaluated to 4 regardless of run_mode.
    n_chains: int = 4  # Placeholder; overridden in __post_init__

    default_thin: int = 2

    # dockerHDDM-specific sampling flags
    enable_loglike: bool = False
    enable_ppc: bool = True
    ppc_samples: int = 500

    # Adaptive sampling limits
    max_adaptive_cycles: int = 5  # Overridden in __post_init__

    # -------------------------------------------------------------------------
    # 6. PPC ADEQUACY THRESHOLDS (absolute goodness-of-fit)
    #    Choice (rejection-rate) MAE is in probability units; RT-quantile
    #    MAE is in seconds. The RT-quantile tolerance (0.20) reflects that
    #    posterior-predictive sampling variability alone is ~0.05-0.15 s on
    #    response times of 1-3 s, i.e. ~10-15% relative error
    #    (Wiecki et al., 2013).
    # -------------------------------------------------------------------------
    ppc_choice_mae_max: float = 0.05
    ppc_choice_max_err: float = 0.10
    ppc_rt_quantile_mae_max: float = 0.20
    ppc_rt_quantile_max_err: float = 0.40

    # -------------------------------------------------------------------------
    # 7. MODEL ARCHITECTURES & DYNAMIC PROTOCOLS
    # -------------------------------------------------------------------------
    final_core_models: List[str] = field(
        default_factory=lambda: ['null', 'v', 'a', 'va']
    )
    # Exploratory set expanded to ensure each parameter family (z, t)
    # has at least one model where it varies independently alongside v,
    # enabling assessment of its marginal contribution beyond the va
    # baseline. Without vz and vt, z/t could only "ride along" with
    # va in vaz/vazt, conflating their individual contribution with
    # the v+a combination.
    final_exploratory_models: List[str] = field(
        default_factory=lambda: ['vz', 'vt', 'vat', 'vaz', 'vazt']
    )

    # -------------------------------------------------------------------------
    # 8. MEMORY SAFETY THRESHOLDS
    # -------------------------------------------------------------------------
    min_free_memory_gb: float = 3.0

    # Dynamic fields computed in __post_init__
    mcmc_protocols: Dict[str, Dict[str, Any]] = field(init=False)

    # -------------------------------------------------------------------------
    # DYNAMIC INITIALIZATION
    # -------------------------------------------------------------------------
    def _infer_model_tier(self, model_name: str) -> str:
        """
        Infers computational complexity tier based on the number
        of free DDM parameter families affected by the emotion
        regressor. Tiers determine MCMC sampling budgets.

        Classification logic:
          - null: always simple (intercept-only, no contrasts)
          - 1 family, no hard params (v, a): simple
          - 2 families with hard params (vz, vt): medium
            (z uses logit link, t uses exp link -> more complex
             posterior geometry requiring additional burn-in)
          - 2 families, no hard params (va): simple
          - 3+ families with hard params (vaz, vazt): complex
          - 3+ families, no hard params: medium
        """
        name = model_name.lower()
        if name == 'null':
            return "simple"
        free_families = sum([
            'v' in name, 'a' in name, 'z' in name, 't' in name
        ])
        has_hard_params = ('z' in name) or ('t' in name)

        if (free_families >= 4
                or (free_families >= 3 and has_hard_params)):
            return "complex"
        elif free_families >= 3:
            return "medium"
        elif has_hard_params:
            # Models like vz, vt, az, at: fewer families but
            # nonlinear link functions make posterior geometry
            # harder to explore. Upgrade from simple to medium.
            return "medium"
        else:
            return "simple"

    def __post_init__(self) -> None:
        """
        Computes run-mode-dependent hyperparameters and per-model
        MCMC sampling protocols. This is the ONLY place where
        self.run_mode is used to branch logic.
        """
        # Correctly set n_chains based on run_mode
        if self.run_mode == 'debug':
            self.n_chains = 2
            self.max_adaptive_cycles = 1
        else:
            self.n_chains = 4
            self.max_adaptive_cycles = 5

        # Build per-model MCMC protocols
        self.mcmc_protocols = {}
        all_models = self.final_core_models + self.final_exploratory_models

        # Tier-specific sampling constraints.
        # group_only_regressors=False increases parameter count
        # substantially, so budgets are set conservatively.
        tiers_config = {
            "simple": {
                "burn": 2000, "target_kept": 3000, "max_kept": 8000
            },
            "medium": {
                "burn": 3000, "target_kept": 4000, "max_kept": 10000
            },
            "complex": {
                "burn": 5000, "target_kept": 5000, "max_kept": 15000
            },
        }

        for model_name in all_models:
            tier = self._infer_model_tier(model_name)
            proto = tiers_config[tier]

            if self.run_mode == 'debug':
                self.mcmc_protocols[model_name] = {
                    "tier": tier,
                    "n_samples": 600,
                    "burn": 100,
                    "thin": 1
                }
            else:
                self.mcmc_protocols[model_name] = {
                    "tier": tier,
                    "n_samples": proto["burn"] + proto["target_kept"] * self.default_thin,
                    "burn": proto["burn"],
                    "thin": self.default_thin,
                    "max_samples": proto["burn"] + proto["max_kept"] * self.default_thin
                }

    # -------------------------------------------------------------------------
    # PARAMETER CLASSIFICATION
    # -------------------------------------------------------------------------
    @staticmethod
    def identify_focal_parameters(param_names: List[str]) -> List[str]:
        """
        Separates focal inferential parameters (group-level Intercepts
        and Treatment contrasts) from hierarchical nuisance parameters
        (subject-level deviations, transformed scales, variance terms).
        """
        nuisance_pattern = re.compile(
            r"(_subj|_trans|_std|_var|_log|\.[\d]+$)"
        )
        return [
            p for p in param_names
            if not nuisance_pattern.search(p)
        ]

    # -------------------------------------------------------------------------
    # COMPUTED PROPERTIES
    # -------------------------------------------------------------------------
    @property
    def final_all_models(self) -> List[str]:
        """Aggregation of core and exploratory model architectures."""
        return self.final_core_models + self.final_exploratory_models

    # -------------------------------------------------------------------------
    # DIRECTORY MANAGEMENT
    # -------------------------------------------------------------------------
    def initialize_directories(self) -> Dict[str, Path]:
        """Constructs and validates the publication output directory tree."""
        subdirs = [
            'manifests', 'audit', 'models', 'ppc', 'recovery',
            'figures_main', 'figures_supp', 'tables_main', 'tables_supp'
        ]
        root_out = self.base_dir / f"results_hddm_{self.run_mode}"
        paths = {name: root_out / name for name in subdirs}
        for p in paths.values():
            p.mkdir(parents=True, exist_ok=True)
        return paths

    # -------------------------------------------------------------------------
    # CRYPTOGRAPHIC FINGERPRINT -> PIPELINE LINEAGE FINGERPRINT
    # -------------------------------------------------------------------------
    def generate_pipeline_fingerprint(self) -> Dict[str, Any]:
        """
        Computes deterministic SHA-256 hash of structural hyperparameters,
        incorporates the empirical data hash, and generates a unified
        pipeline hash. Aesthetic parameters (colors, labels) are
        explicitly excluded from the hash computation.
        """
        critical_keys = [
            'run_mode', 'include_params',
            'group_only_regressors', 'p_outlier',
            'use_informative_priors', 'baseline_condition',
            'n_chains', 'base_seed',
            'rhat_focal', 'ess_bulk_focal', 'ess_tail_focal',
            'rhat_nuisance', 'ess_bulk_nuisance',
            'ess_tail_nuisance',
            'default_thin', 'enable_loglike', 'enable_ppc',
            'ppc_choice_mae_max', 'ppc_choice_max_err',
            'ppc_rt_quantile_mae_max', 'ppc_rt_quantile_max_err',
            'final_core_models', 'final_exploratory_models',
            'mcmc_protocols'
        ]

        cfg_dict = asdict(self)
        structural_params = {k: cfg_dict[k] for k in critical_keys}

        # 1. Configuration Hash
        json_str = json.dumps(structural_params, sort_keys=True)
        config_hash = hashlib.sha256(
            json_str.encode('utf-8')
        ).hexdigest()

        # 2. Data Hash (Acquired from Step 1 Artifact)
        data_fingerprint_path = self.base_dir / "data_fingerprint.json"
        data_hash = ""
        if data_fingerprint_path.exists():
            with open(data_fingerprint_path, 'r', encoding='utf-8') as f:
                data_hash = json.load(f).get('data_file_hash', '')
        else:
            print(
                "WARNING: 'data_fingerprint.json' not found. "
                "Data lineage will be empty."
            )

        # 3. Holistic Pipeline Hash
        pipeline_string = f"{config_hash}_{data_hash}"
        pipeline_hash = hashlib.sha256(
            pipeline_string.encode('utf-8')
        ).hexdigest()

        return {
            'timestamp_utc': datetime.utcnow().isoformat() + 'Z',
            'config_hash': config_hash,
            'data_hash': data_hash,
            'pipeline_hash': pipeline_hash,
            'structural_parameters': structural_params
        }


# =============================================================================
# ROBUST CONFIG SERIALIZATION VIA JSON
# =============================================================================
def _serialize_config_to_disk(
    cfg_obj: HDDMConfig, target_paths: Dict[str, Path]
) -> dict:
    """
    Serializes the configuration state to a Python module
    ('hddm_config.py') for cross-session recovery after kernel
    restarts.

    Rewritten to use JSON-based value persistence instead
    of line-by-line f.write(). The generated module loads a JSON file
    and reconstructs the config object at import time, eliminating
    syntax errors from special characters in paths.

    Writes load_active_lineage_state() (replacing the
    non-existent validate_pipeline_lineage), plus all other utility
    functions needed by downstream Steps.

    Writes check_memory_headroom() for Step 2b safety.

    Writes load_existing_manifest() and
    append_manifest_record() for Step 2b checkpoint resume.
    """
    # --- Phase 1: Persist config values as JSON ---
    cfg_values_path = 'hddm_config_values.json'
    cfg_dict = asdict(cfg_obj)

    # Convert Path objects to strings for JSON serialization
    serializable = {}
    for k, v in cfg_dict.items():
        if isinstance(v, Path):
            serializable[k] = str(v)
        else:
            serializable[k] = v

    with open(cfg_values_path, 'w', encoding='utf-8') as f:
        json.dump(serializable, f, indent=2, default=str)

    # --- Phase 2: Generate hddm_config.py module ---
    py_config_path = 'hddm_config.py'
    module_code = '''# -*- coding: utf-8 -*-
"""
Auto-generated SSOT configuration module.
DO NOT EDIT MANUALLY. Regenerate by running Step 2a.
"""
import os
import re
import gc
import json
import pandas as pd
from pathlib import Path


# =============================================================================
# CONFIGURATION LOADER
# =============================================================================
def _load_config_from_json():
    """
    Reconstructs configuration state from the JSON persistence file
    generated by Step 2a serialization.
    """
    config_json_path = Path(__file__).parent / "hddm_config_values.json"
    if not config_json_path.exists():
        raise FileNotFoundError(
            f"Configuration JSON not found at {config_json_path}. "
            f"Re-execute Step 2a."
        )
    with open(config_json_path, 'r', encoding='utf-8') as f:
        return json.load(f)


class _HDDMConfig:
    """Runtime configuration object reconstructed from JSON persistence."""

    def __init__(self):
        vals = _load_config_from_json()
        for k, v in vals.items():
            if k == 'base_dir':
                setattr(self, k, Path(v))
            else:
                setattr(self, k, v)

    @property
    def final_all_models(self):
        return self.final_core_models + self.final_exploratory_models

    def initialize_directories(self):
        subdirs = [
            'manifests', 'audit', 'models', 'ppc', 'recovery',
            'figures_main', 'figures_supp', 'tables_main',
            'tables_supp'
        ]
        root = self.base_dir / f"results_hddm_{self.run_mode}"
        paths = {n: root / n for n in subdirs}
        for p in paths.values():
            p.mkdir(parents=True, exist_ok=True)
        return paths

    @staticmethod
    def identify_focal_parameters(param_names):
        pat = re.compile(r"(_subj|_trans|_std|_var|_log|\\.\\d+$)")
        return [p for p in param_names if not pat.search(p)]


# Instantiate singleton configuration object
CFG = _HDDMConfig()


# =============================================================================
# LINEAGE VALIDATION UTILITIES
# =============================================================================
def load_active_lineage_state(paths, logger=None):
    """
    Loads the active pipeline lineage state from the configuration
    fingerprint. Returns dict with config_hash, data_hash,
    pipeline_hash.

    This is the canonical lineage validation entry point for ALL
    downstream Steps (2b, 3, 4, 5, 6).
    """
    fp = paths['manifests'] / 'config_fingerprint.json'
    if not fp.exists():
        raise FileNotFoundError(
            'Missing lineage fingerprint. Execute Step 2a.'
        )
    with open(fp, 'r', encoding='utf-8') as fh:
        lineage = json.load(fh)
    if logger:
        logger.info(
            f"Active Pipeline Hash: "
            f"{lineage.get('pipeline_hash', '')[:16]}..."
        )
    return lineage


def validate_artifact_lineage(artifact_df, active_lineage, logger=None):
    """
    Validates an artifact DataFrame's lineage hashes against
    the active pipeline configuration.
    """
    required_fields = ['config_hash', 'data_hash', 'pipeline_hash']
    missing = [
        col for col in required_fields
        if col not in artifact_df.columns
    ]
    if missing:
        raise KeyError(
            f'Strict Contract Failure: Artifact missing '
            f'lineage fields {missing}'
        )
    artifact_hash = str(artifact_df['pipeline_hash'].iloc[0])
    if artifact_hash != active_lineage['pipeline_hash']:
        raise RuntimeError(
            f'CRITICAL LINEAGE MISMATCH!\\n'
            f'Artifact Hash: {artifact_hash}\\n'
            f'Active Hash: {active_lineage["pipeline_hash"]}'
        )
    if logger:
        logger.info(
            'Artifact cryptographic lineage validated successfully.'
        )
    return artifact_hash


def identify_winning_model(paths):
    """
    Parses the Step 4 audit trail to extract the winning model name.
    Strict contract: exactly one model must have Is_Winner == True.
    """
    audit_path = paths['audit'] / 'final_model_selection_audit.csv'
    if not audit_path.exists():
        raise FileNotFoundError(
            'No audit manifest found. '
            'Ensure Step 4 completed successfully.'
        )
    df = pd.read_csv(audit_path)
    if 'Is_Winner' not in df.columns:
        raise KeyError(
            "Strict Contract Failure: "
            "'Is_Winner' column missing in audit file."
        )
    mask = (
        df['Is_Winner'].astype(str).str.strip().str.lower() == 'true'
    )
    winners = df[mask]
    if winners.empty:
        raise ValueError(
            'Strict Contract Failure: No explicit winner flagged.'
        )
    if len(winners) > 1:
        raise ValueError(
            'Strict Contract Failure: '
            'Multiple winners flagged ambiguously.'
        )
    return str(winners.iloc[0]['model_name']).lower()


# =============================================================================
# MODEL-LEVEL CHECKPOINT UTILITIES (for Step 2b)
# =============================================================================
def load_existing_manifest(manifest_path):
    """
    Loads previously completed model records from disk for
    checkpoint resume. Returns a tuple of:
      - set of completed model names (to skip)
      - list of existing manifest record dicts (to preserve)

    A model is considered 'completed' if it has a manifest entry
    without an 'error' field, AND its .nc file exists on disk.
    """
    manifest_path = Path(manifest_path)
    if not manifest_path.exists():
        return set(), []

    df = pd.read_csv(manifest_path, keep_default_na=False)
    if df.empty:
        return set(), []

    # Only count models that completed without error
    if 'error' in df.columns:
        completed_mask = df['error'].isna() | (df['error'] == '')
    else:
        completed_mask = pd.Series([True] * len(df))

    completed_names = set(df.loc[completed_mask, 'model_name'].tolist())

    return completed_names, df.to_dict('records')


def append_manifest_record(manifest_path, record):
    """
    Atomically appends a single model record to the manifest CSV.
    Uses write-to-temp-then-rename pattern to prevent data loss
    from partial writes during power failure.

    On POSIX systems, os.replace() is atomic (IEEE Std 1003.1).
    On Windows/Docker-on-Windows, this provides best-effort
    protection.
    """
    manifest_path = Path(manifest_path)
    tmp_path = manifest_path.with_suffix('.csv.tmp')

    if manifest_path.exists():
        df_existing = pd.read_csv(manifest_path, keep_default_na=False)
        df_new = pd.concat(
            [df_existing, pd.DataFrame([record])],
            ignore_index=True
        )
    else:
        df_new = pd.DataFrame([record])

    df_new.to_csv(tmp_path, index=False)

    # Atomic replacement
    if os.name == 'nt':
        # Windows: os.replace is atomic on NTFS
        os.replace(str(tmp_path), str(manifest_path))
    else:
        # POSIX: os.replace is guaranteed atomic
        os.replace(str(tmp_path), str(manifest_path))


# =============================================================================
# MEMORY SAFETY GUARD (for Step 2b)
# =============================================================================
def check_memory_headroom(min_free_gb=None, logger=None):
    """
    Verifies sufficient free memory before initiating model estimation.
    Prevents OOM-induced system hang in Docker containers configured
    with --oom-kill-disable.

    The default threshold (from CFG.min_free_memory_gb) accounts for:
      - Peak PyMC2 runtime overhead (~2GB for 4 parallel chains)
      - Log-likelihood matrix assembly (~1GB for typical datasets)
      - Safety margin for OS/kernel buffers
    """
    try:
        import psutil
    except ImportError:
        if logger:
            logger.warning(
                "psutil not available. Memory check skipped."
            )
        return True

    if min_free_gb is None:
        min_free_gb = getattr(CFG, 'min_free_memory_gb', 3.0)

    mem = psutil.virtual_memory()
    free_gb = mem.available / (1024 ** 3)

    if free_gb < min_free_gb:
        if logger:
            logger.warning(
                f"Memory headroom low: {free_gb:.1f}GB free "
                f"(minimum {min_free_gb}GB). Forcing gc..."
            )
        gc.collect()

        mem = psutil.virtual_memory()
        free_gb = mem.available / (1024 ** 3)

        if free_gb < min_free_gb:
            raise MemoryError(
                f"CRITICAL: Only {free_gb:.1f}GB available after gc. "
                f"Cannot safely initiate MCMC sampling. "
                f"Consider reducing n_chains or restarting kernel."
            )

    if logger:
        logger.info(f"Memory check passed: {free_gb:.1f}GB available.")
    return True
'''

    with open(py_config_path, 'w', encoding='utf-8') as f:
        f.write(module_code)

    # --- Phase 3: Persist cryptographic fingerprint ---
    lineage_state = cfg_obj.generate_pipeline_fingerprint()
    fp_path = target_paths['manifests'] / "config_fingerprint.json"
    with open(fp_path, 'w', encoding='utf-8') as f:
        json.dump(lineage_state, f, indent=2, default=str)

    return lineage_state


# =============================================================================
# EXECUTION & PERSISTENCE
# =============================================================================

# Instantiate configuration in active kernel memory
CFG = HDDMConfig()
PATHS = CFG.initialize_directories()

# Execute serialization
active_lineage = _serialize_config_to_disk(CFG, PATHS)

# Print confirmation
print("=" * 70)
print(f"  STEP 2a: Configuration SSOT & Pipeline Lineage Initialized")
print(f"  Run Mode:              {CFG.run_mode}")
print(f"  DDM Parameters:        v, a, t + {CFG.include_params}")
print(f"  Chains:                {CFG.n_chains}")
print(f"  Base Seed:             {CFG.base_seed}")
print(f"  group_only_regressors: {CFG.group_only_regressors}")
print(f"  Thin:                  {CFG.default_thin}")
print(f"  keep_regressor_trace:  {CFG.keep_regressor_trace}")
print(f"  LogLike (WAIC):        {CFG.enable_loglike}")
print(f"  Models:                {len(CFG.final_all_models)} architectures")
print("-" * 70)
print(f"  Config Hash:           {active_lineage['config_hash'][:16]}...")
print(f"  Data Hash:             {active_lineage['data_hash'][:16]}...")
print(f"  PIPELINE HASH:         {active_lineage['pipeline_hash'][:16]}...")
print("=" * 70)

# Display per-model protocols
for model_name in CFG.final_all_models:
    proto = CFG.mcmc_protocols[model_name]
    print(
        f"  {model_name:>6s} | tier={proto['tier']:>7s} | "
        f"samples={proto['n_samples']:>5d} | burn={proto['burn']:>4d} | thin={proto['thin']}"
    )

# Step 2b: Hierarchical Bayesian Model Specification and MCMC Estimation

In [ ]:
# -*- coding: utf-8 -*-
"""
=============================================================================
STEP 2b: Hierarchical Bayesian Model Specification & Adaptive MCMC
         Estimation (dockerHDDM ArviZ-Centric Workflow)
=============================================================================
Pipeline Position:
  Upstream:   Step 2a (imports CFG and utilities from hddm_config.py;
              reads 'hddm_data_unfair.csv' from Step 1)
  Downstream: Step 3 (reads .nc and .hddm files from models/ directory;
              reads model_manifest.csv from manifests/ directory)

Methodological Purpose:
  - Executes parallel-chain MCMC sampling via dockerHDDM for each model
    architecture defined in the configuration SSOT.
  - Generates ArviZ InferenceData objects (.nc) and serialized HDDM
    model objects (.hddm) for downstream consumption.
  - Implements a Dynamic Regressor Factory to construct Patsy formulas
    for treatment-coded condition effects.
  - Enforces continuous MCMC convergence evaluation using stratified
    diagnostics (focal vs. nuisance parameters) via adaptive sampling.
  - Implements model-level checkpoint resume: completed models are
    skipped on pipeline restart after interruption.
  - Implements memory safety guard before each model estimation.

Statistical Assumptions & Parameters:
  - Assumes input data contains positive reaction times (RT) in seconds.
  - Convergence criteria utilize Gelman-Rubin R-hat and Effective Sample
    Size (ESS) metrics (Vehtari et al., 2021; Gelman et al., 2020).
  - group_only_regressors=True: Treatment contrast betas are estimated
    at the group level. keep_regressor_trace=False: trial-level
    regressor traces discarded to prevent OOM during InferenceData
    conversion (Wiecki et al., 2013; Pan et al., 2025).

=============================================================================
"""

import os
import gc
import time
import json
import logging
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Tuple, Any

import numpy as np
import pandas as pd
import arviz as az
import hddm

# -----------------------------------------------------------------------------
# CONFIGURATION IMPORT
# Import load_active_lineage_state (replaces validate_pipeline_lineage)
# Import checkpoint utilities and memory guard from Step 2a serialized module
# -----------------------------------------------------------------------------
try:
    from hddm_config import (
        CFG,
        load_active_lineage_state,
        load_existing_manifest,
        append_manifest_record,
        check_memory_headroom
    )
except ImportError:
    raise ImportError(
        "CRITICAL ERROR: 'hddm_config.py' not found. "
        "Execution of Step 2a is mandatory to generate "
        "configuration SSOT."
    )

PATHS = CFG.initialize_directories()

# Establish deterministic behavior using the configured seed
np.random.seed(CFG.base_seed)


# =============================================================================
# LOGGING SETUP
# =============================================================================
def _setup_estimation_logger() -> logging.Logger:
    """
    Initializes dual-sink logging infrastructure (File + Stream)
    for MCMC estimation tracking. Timestamp-stamped log file prevents
    overwrites across multiple pipeline runs.
    """
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    logger = logging.getLogger(f'hddm_mcmc_{ts}')
    logger.handlers = []
    logger.setLevel(logging.INFO)

    fmt = logging.Formatter(
        '%(asctime)s - %(levelname)s - %(message)s',
        datefmt='%H:%M:%S'
    )
    fh = logging.FileHandler(
        PATHS['audit'] / f'mcmc_estimation_{ts}.log',
        encoding='utf-8'
    )
    ch = logging.StreamHandler()
    fh.setFormatter(fmt)
    ch.setFormatter(fmt)

    logger.addHandler(fh)
    logger.addHandler(ch)
    return logger


# =============================================================================
# DYNAMIC REGRESSOR FACTORY
# =============================================================================
def build_model_regressors(
    model_name: str, baseline: str
) -> List[str]:
    """
    Constructs Patsy formulas for hddm.HDDMRegressor.

    CRITICAL dockerHDDM COMPATIBILITY PATTERN:
    Every DDM parameter (v, a, t, z) MUST have an explicit regression
    formula --- either treatment-coded ("v ~ C(emotion, ...)") if the
    parameter varies by condition, or intercept-only ("v ~ 1") if it
    does not. This is because dockerHDDM's wiener_multi_like expects
    ALL parameters to be created as regression nodes (producing
    trial-indexed Pandas Series). Parameters without a formula are
    handled by HDDM's default node constructor, which produces scalar
    floats, causing AttributeError on .loc access.

    This pattern was validated on hcp4715/hddm:latest and matches
    the standard HDDMRegressor usage documented in Pan et al. (2022).

    The null model also uses this pattern with ALL parameters as
    intercept-only. This ensures ALL models share identical
    HDDMRegressor parameterization, making DIC/LOO/WAIC comparisons
    valid across the entire model space.

    Parameters:
      model_name: Architecture specification string.
      baseline:   Reference condition level for Treatment coding.

    Returns:
      List of 4 Patsy formula strings (one per DDM parameter).
    """
    name_lower = model_name.lower()
    regressors = []

    for param in ['v', 'a', 't', 'z']:
        if name_lower != 'null' and param in name_lower:
            # This parameter varies by emotion condition
            regressors.append(
                f"{param} ~ C(emotion, Treatment('{baseline}'))"
            )
        else:
            # This parameter is intercept-only (constant across conditions)
            regressors.append(f"{param} ~ 1")

    return regressors


# =============================================================================
# EMPIRICAL DATA VALIDATION
# =============================================================================
def validate_empirical_data(
    data_path: str, logger: logging.Logger
) -> pd.DataFrame:
    """
    Enforces HDDM structural requirements and datatype constraints
    on the empirical design matrix prior to estimation.

    Validates:
      - Mandatory columns: subj_idx, rt, response
      - RT values are numeric and positive
      - Response values are binary {0, 1}
      - Baseline emotion condition exists in the data
      - No NaN values in critical columns
    """
    if not os.path.exists(data_path):
        raise FileNotFoundError(
            f"Input data path unresolved: {data_path}"
        )

    df = pd.read_csv(data_path)
    logger.info(
        f"Empirical data loaded: {df.shape[0]} trials, "
        f"{df.shape[1]} columns"
    )

    # HDDM structural dependency validation
    required_cols = {'subj_idx', 'rt', 'response'}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(
            f"Mandatory HDDM columns missing: {missing}"
        )

    # Datatype normalization
    df['subj_idx'] = df['subj_idx'].astype(str)
    df['rt'] = pd.to_numeric(df['rt'], errors='coerce')
    df['response'] = pd.to_numeric(df['response'], errors='coerce')

    # Data integrity enforcement: exclude NaN entries
    n_before = len(df)
    df = df.dropna(subset=['rt', 'response'])
    n_dropped = n_before - len(df)
    if n_dropped > 0:
        logger.warning(
            f"Data exclusion: {n_dropped} rows dropped "
            f"due to NaN rt/response."
        )

    # Baseline condition verification
    if 'emotion' in df.columns:
        if CFG.baseline_condition not in df['emotion'].unique():
            raise ValueError(
                f"Baseline condition '{CFG.baseline_condition}' "
                f"absent. Available levels: "
                f"{df['emotion'].unique().tolist()}"
            )

    n_subjects = df['subj_idx'].nunique()
    n_conditions = (
        df['emotion'].nunique() if 'emotion' in df.columns else 0
    )
    logger.info(
        f"Validation complete: {n_subjects} subjects, "
        f"{n_conditions} conditions, {len(df)} trials retained."
    )
    logger.info(
        f"RT bounds: [{df['rt'].min():.3f}, "
        f"{df['rt'].max():.3f}] seconds."
    )

    return df


# =============================================================================
# STRATIFIED CONVERGENCE DIAGNOSTICS
# =============================================================================
def evaluate_stratified_convergence(
    infdata: az.InferenceData, logger: logging.Logger
) -> Tuple[bool, Dict[str, float], pd.DataFrame]:
    """
    Computes Gelman-Rubin (R-hat) and Effective Sample Size (ESS)
    statistics. Applies stratified thresholds based on parameter
    classification (focal vs. nuisance).

    Focal parameters (group-level intercepts and treatment contrasts)
    require stricter thresholds because they are the primary
    inferential targets. Nuisance parameters (subject-level
    deviations, variance terms) use relaxed thresholds because
    hierarchical shrinkage makes them inherently less variable.

    Reference: Vehtari et al. (2021), "Rank-normalization,
    folding, and localization", Bayesian Analysis.

    Parameters:
      infdata: ArviZ InferenceData with posterior group.
      logger:  Active logging instance.

    Returns:
      converged:  Boolean indicating all criteria satisfied.
      metrics:    Dictionary of computed diagnostic extrema.
      summary_df: Full ArviZ statistical summary DataFrame.
    """
    summary_df = az.summary(infdata, round_to=4, hdi_prob=0.95)

    # Parameter stratification
    focal_params = CFG.identify_focal_parameters(
        summary_df.index.tolist()
    )
    summary_df['param_class'] = [
        'focal' if p in focal_params else 'nuisance'
        for p in summary_df.index
    ]

    focal_df = summary_df[summary_df["param_class"] == "focal"]
    nuisance_df = summary_df[summary_df["param_class"] == "nuisance"]

    # Extremum metric extraction
    metrics = {
        "f_rhat": (
            focal_df["r_hat"].max()
            if not focal_df.empty else 1.0
        ),
        "f_bulk": (
            focal_df["ess_bulk"].min()
            if not focal_df.empty else float("inf")
        ),
        "f_tail": (
            focal_df["ess_tail"].min()
            if ("ess_tail" in focal_df.columns
                and not focal_df.empty)
            else float("inf")
        ),
        "n_rhat": (
            nuisance_df["r_hat"].max()
            if not nuisance_df.empty else 1.0
        ),
        "n_bulk": (
            nuisance_df["ess_bulk"].min()
            if not nuisance_df.empty else float("inf")
        ),
        "n_tail": (
            nuisance_df["ess_tail"].min()
            if ("ess_tail" in nuisance_df.columns
                and not nuisance_df.empty)
            else float("inf")
        ),
    }

    # Criteria evaluation
    focal_ok = (
        (metrics["f_rhat"] <= CFG.rhat_focal)
        and (metrics["f_bulk"] >= CFG.ess_bulk_focal)
        and (metrics["f_tail"] >= CFG.ess_tail_focal)
    )

    # Nuisance convergence: R-hat (chains reached the same stationary
    # distribution) plus a gentle ESS_bulk floor (ess_bulk_nuisance) so
    # subject-level parameters are sampled with adequate Monte-Carlo
    # precision. The floor is modest because subject-level parameters in
    # hierarchical DDMs have high autocorrelation (logit/exp scales for
    # z/t); ESS_tail is logged but not gated for nuisance parameters.
    nuisance_ok = (
        (metrics["n_rhat"] <= CFG.rhat_nuisance)
        and (metrics["n_bulk"] >= CFG.ess_bulk_nuisance)
    )

    converged = focal_ok and nuisance_ok

    logger.info(
        f"  FOCAL   | R-hat={metrics['f_rhat']:.3f} "
        f"(<={CFG.rhat_focal}) | "
        f"ESS_bulk={metrics['f_bulk']:.0f} "
        f"(>={CFG.ess_bulk_focal}) | "
        f"ESS_tail={metrics['f_tail']:.0f} "
        f"(>={CFG.ess_tail_focal}) | "
        f"{'PASS' if focal_ok else 'FAIL'}"
    )
    logger.info(
        f"  NUISANCE| R-hat={metrics['n_rhat']:.3f} "
        f"(<={CFG.rhat_nuisance}) | "
        f"ESS_bulk={metrics['n_bulk']:.0f} "
        f"(>={CFG.ess_bulk_nuisance}) | "
        f"ESS_tail={metrics['n_tail']:.0f} "
        f"(>={CFG.ess_tail_nuisance}) | "
        f"{'PASS' if nuisance_ok else 'FAIL'}"
    )

    return converged, metrics, summary_df


# =============================================================================
# CORE: SINGLE MODEL FITTING PIPELINE
# =============================================================================
def fit_single_model(
    model_name: str,
    data: pd.DataFrame,
    logger: logging.Logger,
    config_hash: str
) -> Dict[str, Any]:
    """
    Executes parameter estimation for a specified model architecture.

    Workflow:
      Phase 1: Construct model architecture (HDDMRegressor for ALL
               models including null).
      Phase 2: Execute initial MCMC sampling with dockerHDDM native
               parallel chains.
      Phase 3: Evaluate convergence; if insufficient, run adaptive
               sampling cycles with ESS-deficit-proportional extensions.
      Phase 4: Persist artifacts (.hddm, .nc) and export diagnostics.

    Parameters:
      model_name:  Architecture specification string.
      data:        Validated empirical matrix.
      logger:      Active logging instance.
      config_hash: Active SHA-256 fingerprint for lineage tagging.

    Returns:
      Manifest dictionary with convergence status, file paths,
      and diagnostic metrics.
    """
    logger.info(f"\n{'='*70}")
    logger.info(
        f"ESTIMATION SEQUENCE INITIATED: [{model_name.upper()}]"
    )
    logger.info(f"{'='*70}")

    proto = CFG.mcmc_protocols[model_name]
    logger.info(
        f"  Protocol: tier={proto['tier']}, "
        f"n_samples={proto['n_samples']}, burn={proto['burn']}, "
        f"chains={CFG.n_chains}"
    )

    # -----------------------------------------------------------------
    # MEMORY SAFETY CHECK
    # -----------------------------------------------------------------
    check_memory_headroom(logger=logger)

    # -----------------------------------------------------------------
    # PHASE 1: Architecture Construction
    # ALL models (including null) use HDDMRegressor with explicit
    # formulas for ALL 4 DDM parameters. This is the ONLY pattern
    # confirmed to work on dockerHDDM (hcp4715/hddm:latest).
    #
    # Every parameter gets either:
    #   - Treatment-coded formula (varies by condition)
    #   - Intercept-only "~ 1" formula (constant across conditions)
    #
    # include=['v','a','t','z'] ensures all 4 parameters are
    # created as regression nodes (producing trial-indexed Series),
    # preventing the scalar/.loc AttributeError in wiener_multi_like.
    # -----------------------------------------------------------------
    regressors = build_model_regressors(
        model_name, CFG.baseline_condition
    )

    # All 4 DDM parameters explicitly included
    full_include = ['v', 'a', 't', 'z']

    model = hddm.HDDMRegressor(
        data,
        regressors,
        include=full_include,
        is_group_model=True,
        group_only_regressors=CFG.group_only_regressors,
        keep_regressor_trace=CFG.keep_regressor_trace,
        informative=CFG.use_informative_priors,
        p_outlier=CFG.p_outlier
    )
    logger.info(f"  Architecture: hddm.HDDMRegressor")
    logger.info(f"  include={full_include}")
    for reg in regressors:
        logger.info(f"    -> {reg}")

    # -----------------------------------------------------------------
    # PHASE 2: Initial Posterior Sampling
    # Native parallel chains via dockerHDDM
    #
    # ppc=False here by design: posterior predictive checks are
    # executed in Step 3 via post_pred_gen() after convergence is
    # confirmed. Generating PPC during sampling (a) couples PPC
    # memory overhead with MCMC estimation, risking OOM; (b) PPC
    # from initial samples becomes stale after adaptive extensions
    # in Phase 3; (c) post_pred_gen(append_data=True) provides
    # condition-labeled DataFrames required for condition-level
    # adequacy evaluation, whereas InferenceData.posterior_predictive
    # stores only decontextualized numeric arrays.
    # -----------------------------------------------------------------
    save_prefix = str(PATHS['models'] / f"hddm_{model_name}")
    enable_loglike = CFG.enable_loglike

    t_start = time.time()
    logger.info(
        f"  Sampling: {proto['n_samples']} iterations x "
        f"{CFG.n_chains} chains (burn={proto['burn']})..."
    )

    try:
        infdata = model.sample(
            proto['n_samples'],
            burn=proto['burn'],
            thin=proto.get('thin', 1),
            chains=CFG.n_chains,
            return_infdata=True,
            loglike=enable_loglike,
            ppc=False,
            save_name=save_prefix
        )
    except MemoryError:
        # Fallback: bypass pointwise log-likelihood matrix.
        # This restricts downstream comparison to DIC only;
        # WAIC/LOO-CV become unavailable for this model.
        logger.warning(
            "  MemoryError with loglike=True. "
            "Fallback: sampling without log-likelihood."
        )
        enable_loglike = False
        gc.collect()

        infdata = model.sample(
            proto['n_samples'],
            burn=proto['burn'],
            thin=proto.get('thin', 1),
            chains=CFG.n_chains,
            return_infdata=True,
            loglike=False,
            ppc=False,
            save_name=save_prefix
        )

    t_elapsed = time.time() - t_start
    logger.info(
        f"  Initial sampling completed in {t_elapsed/60:.1f} minutes."
    )

    # -----------------------------------------------------------------
    # PHASE 3: Adaptive Convergence Loop
    # -----------------------------------------------------------------
    converged, metrics, summary_df = evaluate_stratified_convergence(
        infdata, logger
    )

    total_samples = proto['n_samples']
    max_samples = proto.get(
        'max_samples', proto['n_samples'] * 2
    )
    cycle = 0
    prev_metrics = None
    rhat_extension_count = 0

    while not converged and cycle < CFG.max_adaptive_cycles:
        cycle += 1

        # Compute extension size from ESS deficits. Focal ESS (the
        # primary inferential precision target) drives the extension,
        # with the nuisance ESS_bulk floor included so the sampler also
        # works toward adequate precision on subject-level parameters.
        # R-hat improvement cannot be predicted proportionally from
        # sample size (it depends on mixing, not volume), so R-hat
        # deficits instead use a fixed moderate extension.
        deficit_ratios = []

        # Focal ESS deficits (proportional extension)
        if metrics["f_bulk"] < CFG.ess_bulk_focal:
            deficit_ratios.append(
                CFG.ess_bulk_focal / max(metrics["f_bulk"], 1.0)
            )
        if metrics["f_tail"] < CFG.ess_tail_focal:
            deficit_ratios.append(
                CFG.ess_tail_focal / max(metrics["f_tail"], 1.0)
            )
        if metrics["n_bulk"] < CFG.ess_bulk_nuisance:
            deficit_ratios.append(
                CFG.ess_bulk_nuisance / max(metrics["n_bulk"], 1.0)
            )

        if deficit_ratios:
            # ESS-driven: scale extension by worst focal ESS deficit
            deficit_ratio = max(deficit_ratios)
            added_samples = max(
                1000,
                int(total_samples * (deficit_ratio - 1) * 1.25)
            )
        else:
            # R-hat-driven: focal ESS already sufficient, but R-hat
            # (focal or nuisance) has not yet converged. Append a
            # moderate fixed extension (30% of INITIAL n_samples)
            # to allow chains additional mixing time. This addresses
            # borderline R-hat cases (e.g., 1.054 -> 1.048) where
            # insufficient burn-in is the likely cause.
            # Uses initial n_samples as base (not current total) to
            # prevent compounding growth across cycles.
            # Maximum 2 R-hat-driven rounds; terminates early if
            # R-hat does not improve by >= 0.005 or worsens.
            deficit_ratio = 1.3
            added_samples = max(
                1000, int(proto['n_samples'] * 0.30)
            )
            rhat_extension_count += 1

        # R-hat improvement tracking.
        # For R-hat-driven extensions, verify that additional
        # samples actually improved R-hat. If R-hat did not
        # decrease by >= 0.005 (or worsened) after a round,
        # further extensions are unlikely to help --- the issue
        # is posterior geometry, not sample volume.
        if prev_metrics is not None and rhat_extension_count > 0:
            # Track the worst R-hat across focal and nuisance
            prev_worst_rhat = max(
                prev_metrics["f_rhat"], prev_metrics["n_rhat"]
            )
            curr_worst_rhat = max(
                metrics["f_rhat"], metrics["n_rhat"]
            )
            rhat_improvement = prev_worst_rhat - curr_worst_rhat

            logger.info(
                f"  R-hat change: {prev_worst_rhat:.4f} -> "
                f"{curr_worst_rhat:.4f} "
                f"(improvement={rhat_improvement:+.4f})"
            )

            if rhat_improvement < 0.005:
                logger.warning(
                    f"  R-hat improvement insufficient "
                    f"(<0.005) after R-hat extension round "
                    f"{rhat_extension_count}. "
                    f"Likely a posterior geometry or mixing "
                    f"problem. Terminating adaptive extension."
                )
                break

        if rhat_extension_count >= 2:
            logger.info(
                f"  R-hat extension limit reached (2 rounds). "
                f"Terminating adaptive extension."
            )
            break

        prev_metrics = metrics.copy()

        if total_samples + added_samples > max_samples:
            added_samples = max_samples - total_samples
            if added_samples <= 0:
                logger.warning(
                    f"  Sample ceiling ({max_samples}) reached. "
                    f"Terminating adaptive cycles."
                )
                break

        logger.info(
            f"\n  [Adaptive Cycle {cycle}/{CFG.max_adaptive_cycles}] "
            f"Appending {added_samples} samples "
            f"(Worst Deficit Ratio: {deficit_ratio:.2f})..."
        )

        model.sample(
            added_samples,
            burn=0,
            thin=proto.get('thin', 1),
            chains=CFG.n_chains,
            return_infdata=True,
            loglike=False,
            ppc=False,
            save_name=save_prefix
        )

        total_samples += added_samples

        # Regenerate InferenceData from model object
        # CRITICAL: Cannot rely on .nc file because initial save
        # may have failed (e.g., 'index' error or file lock).
        # model.to_infdata() rebuilds InferenceData directly from
        # the live PyMC2 trace, which always reflects appended samples.
        try:
            infdata = model.to_infdata(
                loglike=False, ppc=False
            )
            logger.info(
                f"  InferenceData refreshed from model object."
            )
        except Exception as e:
            logger.warning(
                f"  model.to_infdata() failed: {e}. "
                f"Diagnostics may reflect stale trace."
            )

        converged, metrics, summary_df = (
            evaluate_stratified_convergence(infdata, logger)
        )

    # -----------------------------------------------------------------
    # PHASE 4: Artifact Persistence & Diagnostics
    #
    # SIMPLIFIED PERSISTENCE (no .db relocation)
    #
    # dockerHDDM's save_name parameter generates per-chain .db
    # trace files (PyMC2 database) during model.sample(). These
    # are already on disk and require no manipulation.
    #
    # .db files and .nc files have distinct filenames and are
    # written by independent I/O backends (PyMC2 pickle vs
    # h5py/NetCDF4). No file-lock contention exists between them.
    # The previous relocation mechanism (move .db to temp dir,
    # save .nc, move .db back) introduced unnecessary failure modes
    # without addressing a real problem and has been removed.
    #
    # Artifact dependency map:
    #   .hddm  -> Step 3 (post_pred_gen), Step 4 (DIC)
    #   .nc    -> Step 3 (trace/rank plots), Step 4 (LOO/WAIC),
    #             Step 5 (posterior extraction), Step 6 (recovery)
    #   .db    -> Required by hddm.load() to reconnect PyMC2 trace
    #             database; must remain at original save_name path.
    # -----------------------------------------------------------------

    # 4a. Save serialized HDDM model object (.hddm)
    hddm_save_path = f"{save_prefix}.hddm"
    try:
        model.save(hddm_save_path)
        logger.info(
            f"  Model object saved: {Path(hddm_save_path).name}"
        )
    except Exception as e:
        logger.warning(f"  Failed to save .hddm: {e}")

    # 4b. Save ArviZ InferenceData (.nc)
    # The in-memory infdata from model.sample() may have inconsistent
    # dimensions after adaptive sampling cycles. If saving fails,
    # regenerate a clean InferenceData from the model using
    # model.to_infdata() as fallback.
    nc_save_path = f"{save_prefix}.nc"
    nc_saved = False

    # Primary: save the in-memory infdata
    if infdata is not None:
        try:
            infdata.to_netcdf(nc_save_path)
            logger.info(
                f"  InferenceData saved: {Path(nc_save_path).name}"
            )
            nc_saved = True
        except Exception as e:
            logger.warning(
                f"  In-memory infdata save failed: {e}. "
                f"Attempting regeneration via model.to_infdata()..."
            )

    # Fallback: regenerate from model object
    if not nc_saved:
        try:
            infdata_fresh = model.to_infdata(
                loglike=False, ppc=False
            )
            infdata_fresh.to_netcdf(nc_save_path)
            logger.info(
                f"  InferenceData regenerated and saved: "
                f"{Path(nc_save_path).name}"
            )
            # Update infdata reference for downstream use
            infdata = infdata_fresh
            nc_saved = True
        except Exception as e2:
            logger.warning(
                f"  Fallback .nc save also failed: {e2}. "
                f"Step 3 trace plots will be unavailable "
                f"for [{model_name}]. PPC via .hddm still works."
            )

    # 4c. Statistical summary export
    summary_path = PATHS['audit'] / f"summary_{model_name}.csv"
    summary_df.to_csv(summary_path)
    logger.info(
        f"  Statistical summary exported: {summary_path.name}"
    )

    try:
        dic_value = model.dic
    except Exception:
        dic_value = float('inf')
    logger.info(
        f"  Deviance Information Criterion (DIC) = {dic_value:.2f}"
    )

    manifest_record = {
        'config_hash': config_hash,
        'model_name': model_name,
        'tier': proto['tier'],
        'n_chains': CFG.n_chains,
        'total_samples': total_samples,
        'burn': proto['burn'],
        'converged': converged,
        'dic': dic_value,
        'loglike_available': enable_loglike,
        'ppc_available': False,
        'f_rhat_max': metrics.get('f_rhat', np.nan),
        'f_ess_bulk_min': metrics.get('f_bulk', np.nan),
        'f_ess_tail_min': metrics.get('f_tail', np.nan),
        'n_rhat_max': metrics.get('n_rhat', np.nan),
        'n_ess_bulk_min': metrics.get('n_bulk', np.nan),
        'elapsed_minutes': t_elapsed / 60,
        'nc_path': f"{save_prefix}.nc",
        'hddm_path': f"{save_prefix}.hddm"
    }

    logger.info(
        f"  FINAL STATUS: "
        f"{'CONVERGED' if converged else 'NOT CONVERGED'} "
        f"({total_samples} total samples)."
    )

    # Explicit memory deallocation
    del model
    gc.collect()

    return manifest_record


# =============================================================================
# MAIN EXECUTION THREAD
# =============================================================================
def run_estimation_pipeline():
    """
    Orchestrates the global MCMC estimation workflow sequentially
    across architectures defined in the configuration lineage.

    Implements model-level checkpoint resume:
      - On startup, reads existing manifest to identify completed models.
      - Completed models are skipped without re-estimation.
      - Each newly completed model is immediately persisted to the
        manifest via atomic file write.
      - After interruption and restart, only incomplete models are fitted.
    """
    logger = _setup_estimation_logger()

    # Lineage validation
    # Uses load_active_lineage_state (not validate_pipeline_lineage)
    active_lineage = load_active_lineage_state(PATHS, logger)
    config_hash = active_lineage['config_hash']

    logger.info("=" * 70)
    logger.info(
        f"ESTIMATION PIPELINE INITIALIZED "
        f"(Mode: {CFG.run_mode.upper()})"
    )
    logger.info(f"Lineage Hash: {config_hash[:24]}...")
    logger.info(
        f"DDM Dimensionality: v, a, t + {CFG.include_params}"
    )
    logger.info(f"Parallel Chains: {CFG.n_chains}")
    logger.info(
        f"group_only_regressors: {CFG.group_only_regressors}"
    )
    logger.info(
        f"ArviZ Config: loglike={CFG.enable_loglike}, "
        f"ppc=False (deferred to Step 3)"
    )
    logger.info(f"Target Architectures: {CFG.final_all_models}")
    logger.info("=" * 70)

    if CFG.run_mode == 'debug':
        logger.warning(
            "EXECUTION MODE: DEBUG. "
            "Results hold no scientific validity."
        )
        time.sleep(2)

    # Load and validate empirical data
    data_path = CFG.base_dir / "hddm_data_unfair.csv"
    data = validate_empirical_data(str(data_path), logger)

    # -----------------------------------------------------------------
    # CHECKPOINT RESUME LOGIC
    # -----------------------------------------------------------------
    manifest_path = PATHS['manifests'] / "model_manifest.csv"
    completed_models, existing_records = load_existing_manifest(
        manifest_path
    )

    if completed_models:
        logger.info(
            f"\n  RESUME MODE ACTIVATED: "
            f"{len(completed_models)} model(s) already completed: "
            f"{sorted(str(m) for m in completed_models if pd.notna(m))}"
        )
        logger.info(
            f"  These models will be skipped. Only remaining "
            f"models will be fitted."
        )

    # Track records for final summary display
    all_records = existing_records.copy()

    for model_name in CFG.final_all_models:
        # Checkpoint check: skip already-completed models
        if model_name in completed_models:
            logger.info(
                f"\n[{model_name.upper()}] "
                f"Already completed (checkpoint). Skipping."
            )
            continue

        try:
            record = fit_single_model(
                model_name, data, logger, config_hash
            )

            # Immediately persist to manifest
            append_manifest_record(manifest_path, record)
            logger.info(
                f"  [{model_name.upper()}] Manifest checkpoint saved."
            )

            all_records.append(record)

        except MemoryError as e:
            logger.error(
                f"MEMORY EXHAUSTION in [{model_name}]: {e}"
            )
            error_record = {
                'config_hash': config_hash,
                'model_name': model_name,
                'tier': CFG.mcmc_protocols[model_name]['tier'],
                'converged': False,
                'dic': float('inf'),
                'error': f"MemoryError: {str(e)}"
            }
            append_manifest_record(manifest_path, error_record)
            all_records.append(error_record)

            # Aggressive memory cleanup before attempting next model
            gc.collect()

        except Exception as e:
            logger.error(
                f"FATAL EXCEPTION in [{model_name}]: {e}",
                exc_info=True
            )
            error_record = {
                'config_hash': config_hash,
                'model_name': model_name,
                'tier': CFG.mcmc_protocols[model_name]['tier'],
                'converged': False,
                'dic': float('inf'),
                'error': str(e)
            }
            append_manifest_record(manifest_path, error_record)
            all_records.append(error_record)

    # -----------------------------------------------------------------
    # FINAL SUMMARY
    # -----------------------------------------------------------------
    df_manifest = pd.DataFrame(all_records)

    n_total = len(CFG.final_all_models)
    n_converged = (
        df_manifest['converged'].sum()
        if 'converged' in df_manifest.columns else 0
    )
    n_errors = (
        df_manifest['error'].notna().sum()
        if 'error' in df_manifest.columns else 0
    )

    logger.info(f"\n{'='*70}")
    logger.info(
        f"PIPELINE TERMINATED: "
        f"{n_converged}/{n_total} converged, "
        f"{n_errors} errors."
    )
    logger.info(f"{'='*70}")

    display_cols = [
        'model_name', 'tier', 'converged', 'dic',
        'f_rhat_max', 'f_ess_bulk_min', 'elapsed_minutes'
    ]
    available_cols = [
        c for c in display_cols if c in df_manifest.columns
    ]
    print("\n--- Estimation Summary ---")
    print(df_manifest[available_cols].to_string(index=False))


# =============================================================================
# PIPELINE EXECUTION ENTRY POINT
# =============================================================================
run_estimation_pipeline()

# Step 3: Posterior Trace Extraction and Predictive Simulation

In [ ]:
# -*- coding: utf-8 -*-
"""
=============================================================================
STEP 3: Posterior Predictive Checks (PPC) & Diagnostic Visualization
=============================================================================
Pipeline Position:
  Upstream:   Step 2b (reads .nc and .hddm files from models/ directory;
              reads model_manifest.csv from manifests/ directory)
  Downstream: Step 4 (reads ppc_metrics_{model}.json from ppc/ directory;
              reads observed/ppc_summary_long.csv from ppc/ directory;
              reads summary_{model}.csv from audit/ directory)

Methodological Purpose:
  - Consumes InferenceData artifacts (.nc) produced by Step 2b.
  - Executes rigorous Posterior Predictive Checks (PPC) at dual levels:
    1. Condition-level: Evaluates replication of empirical rejection
       rates per emotion condition via Mean Absolute Error (MAE).
    2. RT-quantile-level: Evaluates structural fidelity of simulated
       RT distributions (10th/50th/90th percentiles MAE).
  - Generates publication-grade diagnostic visualizations:
    * Trace plots for MCMC chain stationarity assessment.
    * Rank plots for mixing quality (Vehtari et al., 2021).
    * Global PPC density overlay (az.plot_ppc).
    * Condition-level PPC faceted RT distributions with per-iteration
      uncertainty overlays (Gabry et al., 2019).
  - Validates cryptographic lineage hashes prior to artifact
    consumption to prevent cross-contamination.

=============================================================================
"""

import os
import gc
import json
import logging
import warnings
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import arviz as az
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns
import hddm

# Suppress inconsequential dependency warnings for clean audit logs
warnings.filterwarnings('ignore', category=FutureWarning)

# -----------------------------------------------------------------------------
# CONFIGURATION IMPORT
# -----------------------------------------------------------------------------
try:
    from hddm_config import (
        CFG,
        load_active_lineage_state
    )
except ImportError:
    raise ImportError(
        "CRITICAL ERROR: 'hddm_config.py' unresolved. "
        "Execution of Step 2a is mandatory to generate "
        "configuration SSOT."
    )

PATHS = CFG.initialize_directories()

# -----------------------------------------------------------------------------
# PPC adequacy thresholds are taken from the configuration (Step 2a) and are
# applied without runtime modification, so the lineage fingerprint reflects
# the criteria actually used.
# -----------------------------------------------------------------------------

# Establish deterministic behavior using the configured seed
np.random.seed(CFG.base_seed)

# Posterior predictive sample count: 500 draws
# Wiecki, Sofer & Frank (2013) recommend S >= 500 for quantile-level PPC.
N_PPC_SAMPLES = 500

# -----------------------------------------------------------------------------
# PUBLICATION-GRADE VISUALIZATION AESTHETICS (APA / Nature Standards)
# -----------------------------------------------------------------------------
sns.set_theme(style="ticks", palette="colorblind")
OKABE_ITO = [
    '#E69F00', '#56B4E9', '#009E73', '#F0E442',
    '#0072B2', '#D55E00', '#CC79A7', '#000000'
]
plt.rcParams.update({
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.prop_cycle': plt.cycler(color=OKABE_ITO),
    'font.size': 11,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'legend.frameon': False,
    'figure.autolayout': True
})


# =============================================================================
# LOGGING SETUP
# =============================================================================
def _setup_ppc_logger() -> logging.Logger:
    """
    Initializes dual-sink logging infrastructure (File + Stream)
    for PPC auditing.
    """
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    logger = logging.getLogger(f'hddm_ppc_{ts}')
    logger.handlers = []
    logger.setLevel(logging.INFO)

    fmt = logging.Formatter(
        '%(asctime)s - %(levelname)s - %(message)s',
        datefmt='%H:%M:%S'
    )
    fh = logging.FileHandler(
        PATHS['audit'] / f'ppc_audit_{ts}.log',
        encoding='utf-8'
    )
    ch = logging.StreamHandler()
    fh.setFormatter(fmt)
    ch.setFormatter(fmt)

    logger.addHandler(fh)
    logger.addHandler(ch)
    return logger


# =============================================================================
# HELPER: FLATTEN 3-LEVEL MULTIINDEX FROM post_pred_gen
# =============================================================================
def flatten_ppc_dataframe(ppc_raw: pd.DataFrame,
                          model_data: pd.DataFrame,
                          logger=None) -> pd.DataFrame:
    """
    Converts the 3-level MultiIndex DataFrame returned by
    kabuki.analyze.post_pred_gen into a flat DataFrame with
    explicit 'node', 'sample', and 'trial_idx' columns, AND
    re-attaches condition labels from the original model.data.

    CRITICAL BUG FIX (v3.2.7):
      kabuki's _post_pred_generate uses:
        sampled_data.join(data.reset_index(), lsuffix='_sampled')
      This join aligns on the DataFrame INDEX. However, node.random()
      produces a 0-based index for each node, while model.data has
      a GLOBAL index (0..N-1 across all subjects). Only the first
      node (wfpt.0, starting at index 0) aligns correctly; all
      subsequent nodes produce NaN for all appended columns
      (emotion, subj_idx, rt, response, offer_amount).
      Result: 96.6% of rows lose their condition labels.

      FIX: Discard the broken appended columns from post_pred_gen.
      Instead, use Level 2 of the MultiIndex (trial_idx), which IS
      the original model.data index, to re-merge condition labels
      from model.data directly.

    Parameters
    ----------
    ppc_raw : pd.DataFrame
        Raw output from hddm.utils.post_pred_gen(append_data=True).
    model_data : pd.DataFrame
        Original observed data (model.data) with condition columns.
    logger : logging.Logger, optional
        Logger for diagnostic messages.

    Returns
    -------
    pd.DataFrame
        Flat DataFrame with columns: node, sample, trial_idx,
        rt_sampled, response_sampled, plus all columns from
        model_data (emotion, subj_idx, rt, response, etc.).
    """
    if not isinstance(ppc_raw.index, pd.MultiIndex):
        if logger:
            logger.warning(
                "  PPC DataFrame does not have MultiIndex. "
                "Returning as-is."
            )
        return ppc_raw

    n_levels = ppc_raw.index.nlevels
    if logger:
        logger.info(
            f"  PPC MultiIndex: {n_levels} levels, "
            f"names={ppc_raw.index.names}"
        )

    df = ppc_raw.reset_index()

    # Assign canonical column names based on level position
    # Level 0 = node (subject), Level 1 = sample (iteration),
    # Level 2 = trial index (original model.data index)
    level_cols = df.columns[:n_levels].tolist()

    rename_map = {}
    if n_levels >= 1:
        rename_map[level_cols[0]] = 'node'
    if n_levels >= 2:
        rename_map[level_cols[1]] = 'sample'
    if n_levels >= 3:
        rename_map[level_cols[2]] = 'trial_idx'

    df = df.rename(columns=rename_map)

    # ------------------------------------------------------------------
    # Discard broken appended columns and re-merge from
    # model.data using trial_idx as the join key.
    #
    # Keep only: node, sample, trial_idx, rt_sampled, response_sampled
    # (plus any other simulated columns that do NOT come from model.data)
    # ------------------------------------------------------------------
    model_data_cols = set(model_data.columns.tolist())
    # Columns that came from model.data via the broken join
    # (these are the ones with 96.6% NaN)
    broken_cols = [
        c for c in df.columns
        if c in model_data_cols and c not in [
            'node', 'sample', 'trial_idx'
        ]
    ]

    # Also detect the 'index' column produced by data.reset_index()
    if 'index' in df.columns:
        broken_cols.append('index')

    if broken_cols and logger:
        # Verify that these columns are indeed mostly NaN
        sample_col = broken_cols[0]
        nan_pct = df[sample_col].isna().mean() * 100
        logger.info(
            f"  Broken append_data columns detected: "
            f"{broken_cols} ({nan_pct:.1f}% NaN). "
            f"Re-merging from model.data via trial_idx."
        )

    # Drop broken columns
    df = df.drop(columns=broken_cols, errors='ignore')

    # Prepare model.data for merge: use its index as trial_idx
    obs_for_merge = model_data.copy()
    obs_for_merge['trial_idx'] = obs_for_merge.index

    # Merge observed data back using trial_idx
    df = df.merge(
        obs_for_merge,
        on='trial_idx',
        how='left'
    )

    # Verify merge success
    if logger:
        n_nodes = df['node'].nunique() if 'node' in df.columns else 0
        n_samples = (
            df['sample'].nunique() if 'sample' in df.columns else 0
        )
        n_emotion_valid = (
            df['emotion'].notna().sum() if 'emotion' in df.columns
            else 0
        )
        logger.info(
            f"  PPC flattened: {len(df)} rows, "
            f"{n_nodes} nodes, {n_samples} samples (iterations), "
            f"emotion coverage: "
            f"{n_emotion_valid}/{len(df)} "
            f"({n_emotion_valid/len(df)*100:.1f}%)"
        )

    return df


# =============================================================================
# CORE CLASS: PPC AUDIT & DIAGNOSTIC ENGINE
# =============================================================================
class PPCAuditEngine:
    """
    Orchestrates InferenceData consumption, multi-level PPC adequacy
    computation, diagnostic visualization, and data structuring for
    the subsequent analytical funnel in Step 4.
    """

    def __init__(self):
        self.logger = _setup_ppc_logger()

        self.active_lineage = load_active_lineage_state(
            PATHS, self.logger
        )
        self.active_hash = self.active_lineage['pipeline_hash']

        self.manifest = self._load_and_validate_manifest()

        self.observed_stats: List[Dict] = []
        self.ppc_stats: List[Dict] = []

        self.logger.info("=" * 70)
        self.logger.info(
            f"PPC AUDIT ENGINE INITIATED "
            f"(Mode: {CFG.run_mode.upper()})"
        )
        self.logger.info(
            f"Pipeline Hash: {self.active_hash[:24]}..."
        )
        self.logger.info(
            f"Target architecture count: {len(self.manifest)}"
        )
        self.logger.info(
            f"PPC samples per model: {N_PPC_SAMPLES}"
        )
        self.logger.info("=" * 70)

    def _load_and_validate_manifest(self) -> pd.DataFrame:
        """
        Retrieves the architecture manifest from Step 2b.
        Validates using config_hash for lineage consistency.
        """
        manifest_path = PATHS['manifests'] / "model_manifest.csv"
        if not manifest_path.exists():
            raise FileNotFoundError(
                "Architecture manifest unresolved. "
                "Step 2b execution required."
            )

        df = pd.read_csv(manifest_path)
        if df.empty:
            raise RuntimeError(
                "Manifest dataset empty. "
                "Step 2b process unverified."
            )

        if 'config_hash' in df.columns:
            artifact_hash = str(df['config_hash'].iloc[0])
            expected_hash = self.active_lineage['config_hash']
            if expected_hash != artifact_hash:
                raise RuntimeError(
                    f"\nCRITICAL LINEAGE MISMATCH!\n"
                    f"Active Config Hash: {expected_hash[:24]}...\n"
                    f"Manifest Config Hash: {artifact_hash[:24]}...\n"
                    f"Resolution: Re-execute Step 2b to synchronize."
                )

        return df

    # -----------------------------------------------------------------
    # ARVIZ-BASED DIAGNOSTIC VISUALIZATION
    # -----------------------------------------------------------------
    def _generate_trace_plots(
        self, infdata: az.InferenceData, model_name: str
    ):
        """
        Constructs MCMC trace visualizations (posterior density and
        sequential traces) restricted to focal parameters for visual
        convergence assessment.
        """
        all_vars = list(infdata.posterior.data_vars.keys())
        focal_vars = CFG.identify_focal_parameters(all_vars)

        if not focal_vars:
            self.logger.warning(
                f"  [{model_name}] Focal parameters absent. "
                f"Trace visualization bypassed."
            )
            return

        try:
            az.plot_trace(
                infdata,
                var_names=focal_vars,
                compact=True,
                figsize=(12, 2.5 * len(focal_vars))
            )
            out_path = (
                PATHS['figures_supp']
                / f"diagnostics_trace_{model_name}.pdf"
            )
            plt.savefig(out_path, dpi=300, bbox_inches='tight')
            plt.close()
            self.logger.info(
                f"  Trace visualization exported: {out_path.name}"
            )
        except Exception as e:
            self.logger.warning(
                f"  Trace visualization failed for "
                f"[{model_name}]: {e}"
            )

    def _generate_rank_plots(
        self, infdata: az.InferenceData, model_name: str
    ):
        """
        Constructs rank plots for focal parameters to detect
        non-stationarity and chain mixing discrepancies.
        Reference: Vehtari et al. (2021).
        """
        all_vars = list(infdata.posterior.data_vars.keys())
        focal_vars = CFG.identify_focal_parameters(all_vars)

        if not focal_vars:
            return

        try:
            az.plot_rank(
                infdata,
                var_names=focal_vars,
                kind='vlines',
                vlines_kwargs={'lw': 0},
                marker_vlines_kwargs={'lw': 2}
            )
            out_path = (
                PATHS['figures_supp']
                / f"diagnostics_rank_{model_name}.pdf"
            )
            plt.savefig(out_path, dpi=300, bbox_inches='tight')
            plt.close()
            self.logger.info(
                f"  Rank plot exported: {out_path.name}"
            )
        except Exception as e:
            self.logger.warning(
                f"  Rank plot failed for [{model_name}]: {e}"
            )

    def _generate_ppc_density_plots(
        self, infdata: az.InferenceData, model_name: str
    ):
        """
        Constructs posterior predictive density overlays via ArviZ.
        Requires 'posterior_predictive' group in InferenceData.
        """
        if not hasattr(infdata, 'posterior_predictive'):
            self.logger.warning(
                f"  [{model_name}] 'posterior_predictive' group "
                f"absent. Density visualization bypassed."
            )
            return

        try:
            az.plot_ppc(
                infdata,
                var_names=['rt'],
                num_pp_samples=100,
                flatten=[]
            )
            out_path = (
                PATHS['figures_supp']
                / f"ppc_density_global_{model_name}.pdf"
            )
            plt.savefig(out_path, dpi=300, bbox_inches='tight')
            plt.close()
            self.logger.info(
                f"  PPC density plot exported: {out_path.name}"
            )
        except Exception as e:
            self.logger.warning(
                f"  PPC density plot failed for "
                f"[{model_name}]: {e}"
            )

    # -----------------------------------------------------------------
    # CONDITION-LEVEL PPC FACETED VISUALIZATION
    # -----------------------------------------------------------------
    def _generate_conditionwise_ppc_plot(
        self,
        model_name: str,
        obs_df: pd.DataFrame,
        ppc_flat: pd.DataFrame
    ):
        """
        Per-emotion faceted RT distribution comparison between observed
        and posterior predictive simulated data. Each panel shows:
          - Black solid line: observed RT kernel density
          - Thin blue lines (alpha=0.05): individual posterior
            predictive iteration densities (up to 50 draws)
          - Dashed blue line: mean of simulated densities

        The ppc_flat DataFrame must be the FLATTENED output from
        flatten_ppc_dataframe, containing 'node', 'sample',
        'rt_sampled', and 'emotion' columns.
        """
        if obs_df.empty or ppc_flat.empty:
            return
        if 'emotion' not in obs_df.columns:
            return

        emotions = sorted(obs_df['emotion'].unique())
        n_emo = len(emotions)
        if n_emo == 0:
            return

        # Identify simulated RT column
        rt_col_sim = (
            'rt_sampled'
            if 'rt_sampled' in ppc_flat.columns
            else 'rt'
        )

        has_sample_col = 'sample' in ppc_flat.columns

        fig, axes = plt.subplots(
            1, n_emo, figsize=(4 * n_emo, 4), squeeze=False
        )
        axes = axes.flatten()

        max_overlay_draws = 50

        for idx, emo in enumerate(emotions):
            ax = axes[idx]
            emo_label = getattr(CFG, 'display_labels', {}).get(
                emo, emo
            )

            # Observed RT density
            obs_emo = obs_df[obs_df['emotion'] == emo]
            rt_upper = 6.0
            if not obs_emo.empty:
                obs_rt = np.abs(obs_emo['rt'].values)
                rt_upper = np.percentile(obs_rt, 99) * 1.2
                sns.kdeplot(
                    obs_rt, ax=ax, color='black',
                    linewidth=2.5, label='Observed',
                    clip=(0, rt_upper), zorder=3
                )

            # Simulated RT densities: per-iteration overlays
            sim_emo = ppc_flat[ppc_flat['emotion'] == emo]

            if not sim_emo.empty and has_sample_col:
                iterations = sorted(
                    sim_emo['sample'].unique()
                )
                draw_iters = iterations[:max_overlay_draws]

                for i, it in enumerate(draw_iters):
                    it_rt = np.abs(
                        sim_emo[
                            sim_emo['sample'] == it
                        ][rt_col_sim].values
                    )
                    if len(it_rt) < 3:
                        continue
                    it_rt = it_rt[it_rt <= rt_upper]
                    if len(it_rt) < 3:
                        continue
                    label = (
                        'Simulated draws' if i == 0 else None
                    )
                    sns.kdeplot(
                        it_rt, ax=ax, color='#56B4E9',
                        linewidth=0.5, alpha=0.08,
                        label=label, zorder=1,
                        clip=(0, rt_upper)
                    )

                # Mean simulated density (pooled reference)
                all_sim_rt = np.abs(sim_emo[rt_col_sim].values)
                all_sim_rt = all_sim_rt[all_sim_rt <= rt_upper]
                if len(all_sim_rt) >= 5:
                    sns.kdeplot(
                        all_sim_rt, ax=ax, color='#0072B2',
                        linewidth=2.0, linestyle='--',
                        label='Simulated mean', zorder=2,
                        clip=(0, rt_upper)
                    )
            elif not sim_emo.empty:
                # Fallback: single pooled density
                sim_rt = np.abs(sim_emo[rt_col_sim].values)
                sim_rt = sim_rt[sim_rt <= rt_upper]
                if len(sim_rt) >= 5:
                    sns.kdeplot(
                        sim_rt, ax=ax, color='#56B4E9',
                        linewidth=1.5, alpha=0.7,
                        label='Simulated', zorder=2
                    )

            ax.set_title(
                emo_label, fontsize=12, fontweight='bold'
            )
            ax.set_xlabel('RT (s)', fontsize=10)
            if idx == 0:
                ax.set_ylabel('Density', fontsize=10)
            else:
                ax.set_ylabel('')

            # Construct legend with visible handles.
            # 'Simulated draws' uses alpha=0.08 in the plot, which
            # renders invisible in the default legend swatch. Create
            # proxy handles with full opacity for readability.
            legend_handles = [
                Line2D([0], [0], color='black', linewidth=2.5,
                       label='Observed'),
                Line2D([0], [0], color='#56B4E9', linewidth=2.0,
                       alpha=0.45, label='Simulated draws'),
                Line2D([0], [0], color='#0072B2', linewidth=2.0,
                       linestyle='--', label='Simulated mean'),
            ]
            ax.legend(
                handles=legend_handles, fontsize=7,
                loc='upper right'
            )
            sns.despine(ax=ax, trim=True)

        plt.suptitle(
            f"Condition-Level PPC: {model_name.upper()}",
            fontsize=14, fontweight='bold', y=1.05
        )
        plt.tight_layout()

        out_path = (
            PATHS['figures_supp']
            / f"ppc_conditionwise_rt_{model_name}.pdf"
        )
        plt.savefig(out_path, dpi=300, bbox_inches='tight')
        plt.close(fig)
        self.logger.info(
            f"  Condition-level PPC plot exported: {out_path.name}"
        )

    # -----------------------------------------------------------------
    # QUANTITATIVE PPC ADEQUACY
    # Correct 3-level MultiIndex handling with per-iteration stats
    # -----------------------------------------------------------------
    def _compute_ppc_adequacy_from_hddm(
        self, model_name: str
    ) -> Dict[str, float]:
        """
        Quantifies posterior predictive adequacy using per-iteration
        statistics computed from the correct 'sample' level of the
        3-level MultiIndex returned by post_pred_gen.

        kabuki.analyze.post_pred_gen output structure:
          Level 0: 'node'   — subject identifier
          Level 1: 'sample' — posterior iteration (0..S-1)
          Level 2: trial index

        For each of S posterior samples, summary statistics (rejection
        rate, RT quantiles per condition) are computed across ALL
        subjects within that iteration. Statistics are then averaged
        across iterations to produce point estimates for comparison
        with observed empirical values.

        Parameters
        ----------
        model_name : str
            Architecture label (e.g., 'v', 'va', 'vat').

        Returns
        -------
        dict
            Keys: choice_mae, choice_max_err, rt_mae, rt_max_err,
                  pass (bool).
        """
        hddm_path = PATHS['models'] / f"hddm_{model_name}.hddm"
        if not hddm_path.exists():
            self.logger.warning(
                f"  [{model_name}] .hddm artifact unresolved. "
                f"Quantitative PPC bypassed."
            )
            return {
                'choice_mae': np.nan, 'rt_mae': np.nan,
                'pass': False
            }

        try:
            model = hddm.load(str(hddm_path))
        except Exception as e:
            self.logger.warning(
                f"  [{model_name}] Failed to load .hddm file: "
                f"{e}. Quantitative PPC bypassed."
            )
            return {
                'choice_mae': np.nan, 'rt_mae': np.nan,
                'pass': False
            }

        # --------------------------------------------------------------
        # Generate posterior predictive datasets (S=500)
        # --------------------------------------------------------------
        try:
            ppc_raw = hddm.utils.post_pred_gen(
                model, samples=N_PPC_SAMPLES, append_data=True
            )
        except Exception as e:
            self.logger.warning(
                f"  [{model_name}] post_pred_gen exception: {e}"
            )
            del model
            gc.collect()
            return {
                'choice_mae': np.nan, 'rt_mae': np.nan,
                'pass': False
            }

        # Flatten and re-merge condition labels
        ppc_flat = flatten_ppc_dataframe(
            ppc_raw, model.data, self.logger
        )

        # Identify simulated column names
        rt_col_sim = (
            'rt_sampled'
            if 'rt_sampled' in ppc_flat.columns
            else 'rt_sim'
        )
        resp_col_sim = (
            'response_sampled'
            if 'response_sampled' in ppc_flat.columns
            else 'response_sim'
        )

        if rt_col_sim not in ppc_flat.columns:
            self.logger.warning(
                f"  [{model_name}] Simulated RT column "
                f"'{rt_col_sim}' not found. "
                f"Columns: {ppc_flat.columns.tolist()}"
            )
            del model, ppc_raw
            gc.collect()
            return {
                'choice_mae': np.nan, 'rt_mae': np.nan,
                'pass': False
            }

        if 'emotion' not in ppc_flat.columns:
            self.logger.warning(
                f"  [{model_name}] 'emotion' column missing."
            )
            del model, ppc_raw
            gc.collect()
            return {
                'choice_mae': np.nan, 'rt_mae': np.nan,
                'pass': False
            }

        # Verify 'sample' column exists after flattening
        if 'sample' not in ppc_flat.columns:
            self.logger.warning(
                f"  [{model_name}] 'sample' column not found "
                f"after flattening. Cannot perform per-iteration "
                f"PPC. Falling back to pooled computation."
            )
            has_sample = False
        else:
            has_sample = True
            n_samples_actual = ppc_flat['sample'].nunique()
            self.logger.info(
                f"  [{model_name}] Posterior iterations "
                f"detected: {n_samples_actual}"
            )

        # --------------------------------------------------------------
        # Observed statistics from model.data
        # --------------------------------------------------------------
        obs_data = model.data.copy()
        obs_records = []
        emotions = sorted(obs_data['emotion'].unique())

        for emo in emotions:
            obs_emo = obs_data[obs_data['emotion'] == emo]
            obs_rej_rate = float(
                (obs_emo['response'] == 0).mean()
            )

            obs_records.append({
                'config_hash': self.active_lineage['config_hash'],
                'model_name': model_name,
                'emotion': emo,
                'stat_name': 'rejection_rate',
                'stat_value': obs_rej_rate,
                'source': 'observed'
            })

            for resp_val, resp_label in [
                (1, 'accept'), (0, 'reject')
            ]:
                obs_resp = obs_emo[
                    obs_emo['response'] == resp_val
                ]
                if len(obs_resp) < 5:
                    continue
                obs_rt = np.abs(obs_resp['rt'].values)
                for q_label, q_val in [
                    ('rt_q10', 0.10),
                    ('rt_q50', 0.50),
                    ('rt_q90', 0.90)
                ]:
                    obs_records.append({
                        'config_hash': (
                            self.active_lineage['config_hash']
                        ),
                        'model_name': model_name,
                        'emotion': emo,
                        'response_type': resp_label,
                        'stat_name': q_label,
                        'stat_value': float(
                            np.quantile(obs_rt, q_val)
                        ),
                        'source': 'observed'
                    })

        # --------------------------------------------------------------
        # Simulated statistics PER ITERATION
        # Group by 'sample' (Level 1 = posterior draw index)
        # Within each draw, pool all subjects for condition-level stats
        # --------------------------------------------------------------
        sim_records = []

        if has_sample:
            iterations = sorted(ppc_flat['sample'].unique())
            self.logger.info(
                f"  [{model_name}] Computing per-iteration PPC "
                f"statistics across {len(iterations)} posterior "
                f"draws."
            )

            # Accumulator: {(emo, stat_name, resp_type): [values]}
            sim_accumulator = {}

            for it in iterations:
                it_data = ppc_flat[ppc_flat['sample'] == it]

                for emo in emotions:
                    emo_data = it_data[
                        it_data['emotion'] == emo
                    ]
                    if emo_data.empty:
                        continue

                    # Per-iteration rejection rate
                    if resp_col_sim in emo_data.columns:
                        rej_rate = float(
                            (emo_data[resp_col_sim] == 0).mean()
                        )
                    else:
                        rej_rate = float(
                            (emo_data['response'] == 0).mean()
                        )

                    key_rej = (emo, 'rejection_rate', 'all')
                    sim_accumulator.setdefault(
                        key_rej, []
                    ).append(rej_rate)

                    # Per-iteration RT quantiles by response
                    for resp_val, resp_label in [
                        (1, 'accept'), (0, 'reject')
                    ]:
                        if resp_col_sim in emo_data.columns:
                            resp_data = emo_data[
                                emo_data[resp_col_sim] == resp_val
                            ]
                        else:
                            resp_data = emo_data[
                                emo_data['response'] == resp_val
                            ]

                        if len(resp_data) < 3:
                            continue

                        sim_rt = np.abs(
                            resp_data[rt_col_sim].values
                        )

                        for q_label, q_val in [
                            ('rt_q10', 0.10),
                            ('rt_q50', 0.50),
                            ('rt_q90', 0.90)
                        ]:
                            key_q = (emo, q_label, resp_label)
                            sim_accumulator.setdefault(
                                key_q, []
                            ).append(
                                float(np.quantile(sim_rt, q_val))
                            )

            # Average across iterations
            for (emo, stat_name, resp_type), values in (
                sim_accumulator.items()
            ):
                record = {
                    'config_hash': (
                        self.active_lineage['config_hash']
                    ),
                    'model_name': model_name,
                    'emotion': emo,
                    'stat_name': stat_name,
                    'stat_value': float(np.mean(values)),
                    'stat_std': float(np.std(values)),
                    'n_iterations': len(values),
                    'source': 'simulated'
                }
                if resp_type != 'all':
                    record['response_type'] = resp_type
                sim_records.append(record)

        else:
            # Fallback: pooled computation (no iteration structure)
            self.logger.warning(
                f"  [{model_name}] Using pooled PPC computation."
            )
            for emo in emotions:
                subset = ppc_flat[ppc_flat['emotion'] == emo]

                if resp_col_sim in subset.columns:
                    sim_rej_rate = float(
                        (subset[resp_col_sim] == 0).mean()
                    )
                else:
                    sim_rej_rate = float(
                        (subset['response'] == 0).mean()
                    )

                sim_records.append({
                    'config_hash': (
                        self.active_lineage['config_hash']
                    ),
                    'model_name': model_name,
                    'emotion': emo,
                    'stat_name': 'rejection_rate',
                    'stat_value': sim_rej_rate,
                    'source': 'simulated'
                })

                for resp_val, resp_label in [
                    (1, 'accept'), (0, 'reject')
                ]:
                    if resp_col_sim in subset.columns:
                        sim_sub = subset[
                            subset[resp_col_sim] == resp_val
                        ]
                    else:
                        sim_sub = subset[
                            subset['response'] == resp_val
                        ]
                    if len(sim_sub) < 5:
                        continue
                    sim_rt = np.abs(sim_sub[rt_col_sim].values)
                    for q_label, q_val in [
                        ('rt_q10', 0.10),
                        ('rt_q50', 0.50),
                        ('rt_q90', 0.90)
                    ]:
                        sim_records.append({
                            'config_hash': (
                                self.active_lineage['config_hash']
                            ),
                            'model_name': model_name,
                            'emotion': emo,
                            'response_type': resp_label,
                            'stat_name': q_label,
                            'stat_value': float(
                                np.quantile(sim_rt, q_val)
                            ),
                            'source': 'simulated'
                        })

        self.observed_stats.extend(obs_records)
        self.ppc_stats.extend(sim_records)

        # Generate condition-level PPC visualization
        self._generate_conditionwise_ppc_plot(
            model_name, obs_data, ppc_flat
        )

        # --------------------------------------------------------------
        # Compute MAE between observed and iteration-averaged simulated
        # --------------------------------------------------------------
        df_obs = pd.DataFrame(obs_records)
        df_sim = pd.DataFrame(sim_records)

        if df_obs.empty or df_sim.empty:
            del model, ppc_raw
            gc.collect()
            return {
                'choice_mae': np.nan, 'rt_mae': np.nan,
                'pass': False
            }

        # --------------------------------------------------------------
        # Separate merge for CHOICE vs RT statistics.
        # rejection_rate records lack 'response_type', while
        # RT quantile records have it. Merging both in one pass
        # with response_type as a key causes NaN-key mismatches
        # that silently drop rejection_rate rows.
        # --------------------------------------------------------------

        # --- Choice (rejection_rate) merge ---
        obs_choice = df_obs[
            df_obs['stat_name'] == 'rejection_rate'
        ][['emotion', 'stat_name', 'stat_value']].copy()
        sim_choice = df_sim[
            df_sim['stat_name'] == 'rejection_rate'
        ][['emotion', 'stat_name', 'stat_value']].copy()

        if not obs_choice.empty and not sim_choice.empty:
            merged_choice = pd.merge(
                obs_choice, sim_choice,
                on=['emotion', 'stat_name'],
                suffixes=('_obs', '_sim'),
                how='inner'
            )
            merged_choice['abs_error'] = np.abs(
                merged_choice['stat_value_obs']
                - merged_choice['stat_value_sim']
            )
            choice_errs = merged_choice['abs_error']
        else:
            choice_errs = pd.Series(dtype=float)

        # --- RT quantile merge ---
        obs_rt = df_obs[
            df_obs['stat_name'].str.startswith('rt_')
        ].copy()
        sim_rt = df_sim[
            df_sim['stat_name'].str.startswith('rt_')
        ].copy()

        rt_merge_keys = ['emotion', 'stat_name']
        if (
            'response_type' in obs_rt.columns
            and 'response_type' in sim_rt.columns
        ):
            rt_merge_keys.append('response_type')

        if not obs_rt.empty and not sim_rt.empty:
            merged_rt = pd.merge(
                obs_rt[rt_merge_keys + ['stat_value']],
                sim_rt[rt_merge_keys + ['stat_value']],
                on=rt_merge_keys,
                suffixes=('_obs', '_sim'),
                how='inner'
            )
            merged_rt['abs_error'] = np.abs(
                merged_rt['stat_value_obs']
                - merged_rt['stat_value_sim']
            )
            rt_errs = merged_rt['abs_error']
        else:
            rt_errs = pd.Series(dtype=float)

        choice_mae = (
            float(choice_errs.mean())
            if not choice_errs.empty else np.nan
        )
        choice_max = (
            float(choice_errs.max())
            if not choice_errs.empty else np.nan
        )
        rt_mae = (
            float(rt_errs.mean())
            if not rt_errs.empty else np.nan
        )
        rt_max = (
            float(rt_errs.max())
            if not rt_errs.empty else np.nan
        )

        # Adequacy classification using CFG thresholds
        is_adequate = (
            (not np.isnan(choice_mae))
            and (choice_mae <= CFG.ppc_choice_mae_max)
            and (choice_max <= CFG.ppc_choice_max_err)
            and (not np.isnan(rt_mae))
            and (rt_mae <= CFG.ppc_rt_quantile_mae_max)
            and (rt_max <= CFG.ppc_rt_quantile_max_err)
        )

        self.logger.info(
            f"  PPC Adequacy [v3.2]: "
            f"Choice MAE={choice_mae:.4f} "
            f"(Limit: {CFG.ppc_choice_mae_max}), "
            f"RT MAE={rt_mae:.4f} "
            f"(Limit: {CFG.ppc_rt_quantile_mae_max}) "
            f"-> {'PASS' if is_adequate else 'FAIL'}"
        )

        del model, ppc_raw
        gc.collect()

        return {
            'choice_mae': choice_mae,
            'choice_max_err': choice_max,
            'rt_mae': rt_mae,
            'rt_max_err': rt_max,
            'pass': is_adequate
        }

    # -----------------------------------------------------------------
    # POSTERIOR PARAMETER MANIFEST GENERATION
    # -----------------------------------------------------------------
    def _generate_posterior_manifest(
        self, infdata: az.InferenceData, model_name: str
    ):
        """
        Constructs a structural taxonomy of posterior parameters
        (identifying family, level, focal/nuisance status) for
        targeted downstream extraction in Steps 4-5.
        """
        all_vars = list(infdata.posterior.data_vars.keys())
        focal_params = CFG.identify_focal_parameters(all_vars)

        records = []
        for var in all_vars:
            family = var.split('_')[0] if '_' in var else var
            if family not in ['v', 'a', 't', 'z']:
                family = 'other'

            level = (
                'subject' if '_subj' in var
                else (
                    'sd'
                    if (var.endswith('_std')
                        or var.endswith('_var'))
                    else 'group'
                )
            )
            is_focal = var in focal_params

            records.append({
                'variable_name': var,
                'family': family,
                'level': level,
                'focal': is_focal
            })

        df_manifest = pd.DataFrame(records)
        out_path = (
            PATHS['audit']
            / f"posterior_manifest_{model_name}.csv"
        )
        df_manifest.to_csv(out_path, index=False)

    # -----------------------------------------------------------------
    # SINGLE MODEL PROCESSING
    # -----------------------------------------------------------------
    def process_model(self, model_name: str):
        """
        Executes the comprehensive audit sequence for a singular
        target architecture: load InferenceData, generate diagnostics,
        compute PPC adequacy, export all artifacts.
        """
        self.logger.info(f"\n{'─'*50}")
        self.logger.info(
            f"Target Architecture: [{model_name.upper()}]"
        )
        self.logger.info(f"{'─'*50}")

        # Verify target architecture in manifest
        model_row = self.manifest[
            self.manifest['model_name'] == model_name
        ]
        if model_row.empty:
            self.logger.warning(
                f"  [{model_name}] Unregistered in manifest. "
                f"Bypassed."
            )
            return

        nc_path = PATHS['models'] / f"hddm_{model_name}.nc"
        if not nc_path.exists():
            self.logger.warning(
                f"  [{model_name}] NetCDF unresolved at "
                f"{nc_path.name}. Bypassed."
            )
            return

        infdata = az.from_netcdf(str(nc_path))
        self.logger.info(
            f"  InferenceData imported. Groups: "
            f"{list(infdata.groups())}"
        )

        # Posterior parameter manifest
        self._generate_posterior_manifest(infdata, model_name)

        # ArviZ statistical summary with focal/nuisance classification
        summary_df = az.summary(
            infdata, round_to=4, hdi_prob=0.95
        )
        focal_params = CFG.identify_focal_parameters(
            summary_df.index.tolist()
        )
        summary_df['param_class'] = [
            'focal' if p in focal_params else 'nuisance'
            for p in summary_df.index
        ]
        summary_path = (
            PATHS['audit'] / f"summary_{model_name}.csv"
        )
        summary_df.to_csv(summary_path)
        self.logger.info(
            f"  ArviZ summary exported: {summary_path.name}"
        )

        # Diagnostic visualizations
        self._generate_trace_plots(infdata, model_name)
        self._generate_rank_plots(infdata, model_name)
        self._generate_ppc_density_plots(infdata, model_name)

        # Quantitative PPC adequacy (v3.2: correct MultiIndex)
        ppc_metrics = self._compute_ppc_adequacy_from_hddm(
            model_name
        )

        # Persist PPC metrics as JSON for Step 4 consumption
        ppc_record_path = (
            PATHS['ppc'] / f"ppc_metrics_{model_name}.json"
        )
        with open(ppc_record_path, 'w', encoding='utf-8') as f:
            json.dump(
                {
                    'config_hash': (
                        self.active_lineage['config_hash']
                    ),
                    'pipeline_hash': self.active_hash,
                    'model_name': model_name,
                    'ppc_samples': N_PPC_SAMPLES,
                    'enforced_thresholds': {
                        'ppc_choice_mae_max': CFG.ppc_choice_mae_max,
                        'ppc_choice_max_err': CFG.ppc_choice_max_err,
                        'ppc_rt_quantile_mae_max': CFG.ppc_rt_quantile_mae_max,
                        'ppc_rt_quantile_max_err': CFG.ppc_rt_quantile_max_err,
                    },
                    **ppc_metrics
                },
                f, indent=2
            )

        del infdata
        gc.collect()

    # -----------------------------------------------------------------
    # MASTER PIPELINE
    # -----------------------------------------------------------------
    def run(self):
        """
        Orchestrates the sequential multi-architecture PPC audit
        pipeline across all model architectures defined in the
        configuration SSOT.
        """
        for model_name in CFG.final_all_models:
            self.process_model(model_name)

        # Export aggregated statistics for Step 4 fallback path
        if self.observed_stats:
            df_obs = pd.DataFrame(self.observed_stats)
            df_obs.to_csv(
                PATHS['ppc'] / "observed_summary_long.csv",
                index=False
            )

        if self.ppc_stats:
            df_sim = pd.DataFrame(self.ppc_stats)
            df_sim.to_csv(
                PATHS['ppc'] / "ppc_summary_long.csv",
                index=False
            )

        self.logger.info(f"\n{'='*70}")
        self.logger.info(
            f"STEP 3 PIPELINE TERMINATED [v3.2]: "
            f"Audit executed across "
            f"{len(CFG.final_all_models)} architectures."
        )
        self.logger.info(
            f"PPC samples per model: {N_PPC_SAMPLES}"
        )
        self.logger.info(
            f"Totals: Observed records={len(self.observed_stats)}"
            f" | Simulated records={len(self.ppc_stats)}"
        )
        self.logger.info(f"{'='*70}")


# =============================================================================
# PIPELINE EXECUTION ENTRY POINT
# =============================================================================
engine = PPCAuditEngine()
engine.run()

# Step 4: Convergence Diagnostics and Model Selection

In [ ]:
# -*- coding: utf-8 -*-
"""
=============================================================================
STEP 4: Four-Level Diagnostic Funnel & Optimal Model Selection
=============================================================================
Pipeline Position:
  Upstream:   Step 2b (reads .nc and .hddm files from models/ directory;
              reads model_manifest.csv from manifests/ directory)
              Step 3 (reads ppc_metrics_{model}.json from ppc/ directory;
              reads observed/ppc_summary_long.csv from ppc/ directory)
  Downstream: Step 5 (reads final_model_selection_audit.csv from audit/;
              reads .nc file of winning model from models/)
              Step 6 (same as Step 5)

Methodological Purpose:
  - Implements a hierarchical evaluation framework (Four-Level Funnel):
    Level 1 (Technical): Validates artifact integrity (.nc, .hddm).
    Level 2 (Convergence): Evaluates stratified MCMC stationarity via
                           ArviZ InferenceData (Vehtari et al., 2021).
    Level 3 (PPC Adequacy): Enforces absolute goodness-of-fit via
                            empirical MAE thresholds from Step 3.
    Level 4 (Relative): Ranks surviving architectures via DIC as the
                        primary operational criterion. PSIS-LOO-CV and
                        WAIC are computed only when pointwise
                        log-likelihood is available (enable_loglike=True
                        in Step 2a), which is disabled by default to
                        prevent memory exhaustion (Pan et al., 2025,
                        report 20-30 GB peak usage with loglike=True).
  - Selects optimal model exclusively from architectures passing L1-L3.
  - Constructs publication-grade diagnostics for the winning model.
  - Exports structured cryptographic audit trails.

=============================================================================
"""

import gc
import json
import logging
import warnings
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Any, Optional

import numpy as np
import pandas as pd
import arviz as az
import matplotlib.pyplot as plt
import seaborn as sns
import hddm

# Suppress inconsequential dependency warnings
warnings.filterwarnings('ignore', category=FutureWarning)

# -----------------------------------------------------------------------------
# CONFIGURATION IMPORT
# Uses load_active_lineage_state (not validate_pipeline_lineage)
# -----------------------------------------------------------------------------
try:
    from hddm_config import (
        CFG,
        load_active_lineage_state
    )
except ImportError:
    raise ImportError(
        "CRITICAL ERROR: 'hddm_config.py' unresolved. "
        "Execution of Step 2a is mandatory to generate "
        "configuration SSOT."
    )

PATHS = CFG.initialize_directories()

# Establish deterministic behavior using the configured seed
np.random.seed(CFG.base_seed)

# -----------------------------------------------------------------------------
# PUBLICATION-GRADE VISUALIZATION AESTHETICS
# -----------------------------------------------------------------------------
sns.set_theme(style="ticks", palette="colorblind")
OKABE_ITO = [
    '#E69F00', '#56B4E9', '#009E73', '#F0E442',
    '#0072B2', '#D55E00', '#CC79A7', '#000000'
]
plt.rcParams.update({
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.prop_cycle': plt.cycler(color=OKABE_ITO),
    'font.size': 11,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'legend.frameon': False,
    'figure.autolayout': True
})


# =============================================================================
# LOGGING SETUP
# =============================================================================
def _setup_funnel_logger() -> logging.Logger:
    """
    Initializes dual-sink logging for the diagnostic funnel audit.
    """
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    logger = logging.getLogger(f'hddm_funnel_{ts}')
    logger.handlers = []
    logger.setLevel(logging.INFO)

    fmt = logging.Formatter(
        '%(asctime)s - %(levelname)s - %(message)s',
        datefmt='%H:%M:%S'
    )
    fh = logging.FileHandler(
        PATHS['audit'] / f'model_selection_funnel_{ts}.log',
        encoding='utf-8'
    )
    ch = logging.StreamHandler()
    fh.setFormatter(fmt)
    ch.setFormatter(fmt)

    logger.addHandler(fh)
    logger.addHandler(ch)
    return logger


# =============================================================================
# CORE CLASS: FOUR-LEVEL DIAGNOSTIC FUNNEL
# =============================================================================
class DiagnosticFunnelEngine:
    """
    Executes a sequential, hierarchical model evaluation framework.
    Filters model architectures through progressive stringency,
    computing relative information criteria (PSIS-LOO/WAIC/DIC)
    exclusively for empirically adequate models.
    """

    def __init__(self):
        self.logger = _setup_funnel_logger()

        # Use load_active_lineage_state
        self.active_lineage = load_active_lineage_state(
            PATHS, self.logger
        )
        self.active_hash = self.active_lineage['config_hash']

        self.manifest = self._load_manifest()
        self.comparison_records: List[Dict] = []

        self.logger.info("=" * 70)
        self.logger.info(
            f"DIAGNOSTIC FUNNEL ENGINE INITIATED "
            f"(Mode: {CFG.run_mode.upper()})"
        )
        self.logger.info(
            f"Pipeline Hash: "
            f"{self.active_lineage['pipeline_hash'][:24]}..."
        )
        self.logger.info(
            f"Stratified Convergence: "
            f"Focal R-hat<={CFG.rhat_focal}, "
            f"ESS>={CFG.ess_bulk_focal} | "
            f"Nuisance R-hat<={CFG.rhat_nuisance}, "
            f"ESS>={CFG.ess_bulk_nuisance}"
        )
        self.logger.info(
            f"PPC Adequacy: Choice MAE<={CFG.ppc_choice_mae_max}, "
            f"RT MAE<={CFG.ppc_rt_quantile_mae_max}"
        )
        self.logger.info("=" * 70)

    def _load_manifest(self) -> pd.DataFrame:
        """Retrieves Step 2b architecture manifest."""
        manifest_path = PATHS['manifests'] / "model_manifest.csv"
        if manifest_path.exists():
            return pd.read_csv(manifest_path)
        self.logger.warning(
            "Architecture manifest unresolved. "
            "Initiating dynamic discovery."
        )
        return pd.DataFrame()

    # -----------------------------------------------------------------
    # LEVEL 1: TECHNICAL ARTIFACT VERIFICATION
    # -----------------------------------------------------------------
    def evaluate_level1_technical(
        self, model_name: str
    ) -> bool:
        """
        Validates the structural presence of compiled ArviZ (.nc)
        and HDDM (.hddm) artifacts on disk.
        """
        nc_file = PATHS['models'] / f"hddm_{model_name}.nc"
        hddm_file = PATHS['models'] / f"hddm_{model_name}.hddm"

        nc_ok = nc_file.exists()
        hddm_ok = hddm_file.exists()

        if not nc_ok:
            self.logger.warning(
                f"  L1: NetCDF unresolved -> {nc_file.name}"
            )
        if not hddm_ok:
            self.logger.warning(
                f"  L1: HDDM artifact unresolved -> "
                f"{hddm_file.name}"
            )

        return nc_ok and hddm_ok

    # -----------------------------------------------------------------
    # LEVEL 2: STRATIFIED CONVERGENCE DIAGNOSTICS
    # -----------------------------------------------------------------
    def evaluate_level2_convergence(
        self, infdata: az.InferenceData, model_name: str
    ) -> Dict[str, Any]:
        """
        Quantifies MCMC stationarity and mixing efficiency via
        stratified constraints. Focal group-level parameters require
        stricter thresholds than nuisance subject-level deviations.
        """
        summary_df = az.summary(
            infdata, round_to=4, hdi_prob=0.95
        )

        required_cols = {'r_hat', 'ess_bulk'}
        missing = required_cols - set(summary_df.columns)
        if missing:
            return {
                'pass': False,
                'reason': (
                    f"ArviZ schema violation: missing {missing}"
                ),
                'f_rhat_max': np.nan,
                'f_ess_bulk_min': np.nan
            }

        focal_params = CFG.identify_focal_parameters(
            summary_df.index.tolist()
        )
        summary_df['param_class'] = [
            'focal' if p in focal_params else 'nuisance'
            for p in summary_df.index
        ]

        focal_df = summary_df[
            summary_df['param_class'] == 'focal'
        ]
        nuisance_df = summary_df[
            summary_df['param_class'] == 'nuisance'
        ]

        f_rhat = (
            focal_df['r_hat'].max()
            if not focal_df.empty else 1.0
        )
        f_bulk = (
            focal_df['ess_bulk'].min()
            if not focal_df.empty else float('inf')
        )
        f_tail = (
            focal_df['ess_tail'].min()
            if ('ess_tail' in focal_df.columns
                and not focal_df.empty)
            else float('inf')
        )

        n_rhat = (
            nuisance_df['r_hat'].max()
            if not nuisance_df.empty else 1.0
        )
        n_bulk = (
            nuisance_df['ess_bulk'].min()
            if not nuisance_df.empty else float('inf')
        )
        n_tail = (
            nuisance_df['ess_tail'].min()
            if ('ess_tail' in nuisance_df.columns
                and not nuisance_df.empty)
            else float('inf')
        )

        focal_ok = (
            f_rhat <= CFG.rhat_focal
            and f_bulk >= CFG.ess_bulk_focal
            and f_tail >= CFG.ess_tail_focal
        )
        # Model SELECTION gates nuisance parameters on R-hat only: a
        # model is never excluded from selection for nuisance Monte-Carlo
        # precision. The nuisance ESS_bulk floor is a sampling target in
        # Step 2b, not a selection criterion; focal parameters carry the
        # ESS requirement here (Vehtari et al., 2021).
        nuisance_ok = (
            n_rhat <= CFG.rhat_nuisance
        )

        converged = focal_ok and nuisance_ok

        reason = (
            "Criteria Satisfied" if converged
            else (
                f"Focal: R-hat={f_rhat:.3f},"
                f"ESS_b={f_bulk:.0f},"
                f"ESS_t={f_tail:.0f} | "
                f"Nuisance: R-hat={n_rhat:.3f},"
                f"ESS_b={n_bulk:.0f}"
            )
        )

        self.logger.info(
            f"  L2 FOCAL:    R-hat={f_rhat:.3f}, "
            f"ESS_bulk={f_bulk:.0f}, ESS_tail={f_tail:.0f} "
            f"-> {'PASS' if focal_ok else 'FAIL'}"
        )
        self.logger.info(
            f"  L2 NUISANCE: R-hat={n_rhat:.3f}, "
            f"ESS_bulk={n_bulk:.0f}, ESS_tail={n_tail:.0f} "
            f"-> {'PASS' if nuisance_ok else 'FAIL'}"
        )

        if not focal_df.empty:
            focal_df.to_csv(
                PATHS['audit']
                / f"focal_parameter_audit_{model_name}.csv"
            )

        return {
            'pass': converged,
            'reason': reason,
            'f_rhat_max': f_rhat,
            'f_ess_bulk_min': f_bulk,
            'f_ess_tail_min': f_tail,
            'n_rhat_max': n_rhat,
            'n_ess_bulk_min': n_bulk,
            'n_ess_tail_min': n_tail
        }

    # -----------------------------------------------------------------
    # LEVEL 3: PPC ADEQUACY
    # -----------------------------------------------------------------
    def evaluate_level3_ppc(
            self, model_name: str
        ) -> Dict[str, Any]:
            """
            Validates predictive fidelity constraints. Integrates JSON
            metrics from Step 3 or falls back to long-format CSV.

            If posterior-predictive adequacy cannot be evaluated for a
            model (no metrics available), Level 3 is treated as not
            satisfied.
            """
            ppc_path = (
                PATHS['ppc'] / f"ppc_metrics_{model_name}.json"
            )

            if not ppc_path.exists():
                self.logger.warning(
                    f"  L3: PPC JSON unresolved for [{model_name}]. "
                    f"Executing long-format fallback..."
                )
                fallback = self._evaluate_ppc_from_long_format(model_name)
                # A model whose predictive adequacy cannot be evaluated
                # does not pass Level 3.
                if np.isnan(fallback.get('choice_mae', np.nan)):
                    self.logger.warning(
                        f"  L3: PPC data unavailable for [{model_name}]. "
                        f"Marking Level 3 as not satisfied."
                    )
                    return {
                        'pass': False,
                        'choice_mae': np.nan,
                        'rt_mae': np.nan,
                        'ppc_unavailable': True
                    }
                return fallback

            with open(ppc_path, 'r', encoding='utf-8') as f:
                ppc_data = json.load(f)

            is_adequate = ppc_data.get('pass', False)
            choice_mae = ppc_data.get('choice_mae', np.nan)
            rt_mae = ppc_data.get('rt_mae', np.nan)

            return {
                'pass': is_adequate,
                'choice_mae': choice_mae,
                'rt_mae': rt_mae
            }
    def _evaluate_ppc_from_long_format(
        self, model_name: str
    ) -> Dict[str, Any]:
        """
        Fallback adequacy evaluation using aggregated CSV formats
        when target-specific JSON manifests are unavailable.
        """
        obs_path = PATHS['ppc'] / "observed_summary_long.csv"
        sim_path = PATHS['ppc'] / "ppc_summary_long.csv"

        if not obs_path.exists() or not sim_path.exists():
            return {
                'pass': False,
                'choice_mae': np.nan,
                'rt_mae': np.nan
            }

        df_obs = pd.read_csv(obs_path)
        df_sim = pd.read_csv(sim_path)

        df_sim_model = df_sim[
            df_sim['model_name'] == model_name
        ]
        if df_sim_model.empty:
            return {
                'pass': False,
                'choice_mae': np.nan,
                'rt_mae': np.nan
            }

        merge_keys = ['emotion', 'stat_name']
        if 'response_type' in df_sim_model.columns:
            merge_keys.append('response_type')

        sim_agg = (
            df_sim_model
            .groupby(merge_keys)['stat_value']
            .mean()
            .reset_index()
            .rename(columns={'stat_value': 'sim_value'})
        )

        obs_subset = df_obs
        if 'source' in df_obs.columns:
            obs_subset = df_obs[df_obs['source'] == 'observed']

        merged = pd.merge(
            obs_subset,
            sim_agg,
            on=merge_keys,
            how='inner'
        )

        if ('stat_value' not in merged.columns
                or 'sim_value' not in merged.columns):
            return {
                'pass': False,
                'choice_mae': np.nan,
                'rt_mae': np.nan
            }

        merged['abs_error'] = np.abs(
            merged['stat_value'] - merged['sim_value']
        )

        choice_errs = merged[
            merged['stat_name'] == 'rejection_rate'
        ]['abs_error']
        rt_errs = merged[
            merged['stat_name'].str.startswith('rt_')
        ]['abs_error']

        choice_mae = (
            float(choice_errs.mean())
            if not choice_errs.empty else np.nan
        )
        rt_mae = (
            float(rt_errs.mean())
            if not rt_errs.empty else np.nan
        )

        is_adequate = (
            (not np.isnan(choice_mae))
            and (choice_mae <= CFG.ppc_choice_mae_max)
            and (not np.isnan(rt_mae))
            and (rt_mae <= CFG.ppc_rt_quantile_mae_max)
        )

        return {
            'pass': is_adequate,
            'choice_mae': choice_mae,
            'rt_mae': rt_mae
        }

    # -----------------------------------------------------------------
    # LEVEL 4: RELATIVE MODEL COMPARISON
    # Pareto-k diagnostic for PSIS-LOO-CV
    # DIC fallback with methodological warning
    # -----------------------------------------------------------------
    def compute_model_comparison_metrics(
        self,
        model_name: str,
        infdata: az.InferenceData
    ) -> Dict[str, Any]:
        """
        Computes predictive performance indicators: DIC (primary
        operational criterion), PSIS-LOO-CV and WAIC (available
        only when enable_loglike=True in Step 2a).

        PSIS-LOO-CV includes Pareto-k diagnostic.
        When >10% of observations have k > 0.7, the PSIS
        approximation is unreliable and a warning is issued.
        Reference: Vehtari, Gelman & Gabry (2017), Statistics
        and Computing, Section 3.4.

        DIC fallback includes methodological caveat
        that DIC is not uniquely defined for hierarchical models.
        Reference: Gelman, Hwang & Vehtari (2014).
        """
        metrics: Dict[str, Any] = {
            'dic': float('inf'),
            'loo': np.nan,
            'waic': np.nan,
            'loo_reliable': False,
            'pareto_k_pct_bad': np.nan
        }

        # DIC from HDDM model object
        hddm_path = PATHS['models'] / f"hddm_{model_name}.hddm"
        if hddm_path.exists():
            try:
                model = hddm.load(str(hddm_path))
                metrics['dic'] = float(model.dic)
                del model
                gc.collect()
            except Exception as e:
                self.logger.warning(
                    f"  DIC derivation exception: {e}"
                )

        # PSIS-LOO-CV and WAIC from ArviZ InferenceData
        if hasattr(infdata, 'log_likelihood'):
            # --- PSIS-LOO-CV with Pareto-k diagnostic ---
            try:
                loo_result = az.loo(infdata)
                metrics['loo'] = float(loo_result.elpd_loo)

                # Pareto-k diagnostic
                pareto_k = loo_result.pareto_k
                if pareto_k is not None:
                    k_values = np.array(pareto_k).flatten()
                    n_bad = np.sum(k_values > 0.7)
                    pct_bad = float(n_bad / len(k_values) * 100)
                    metrics['pareto_k_pct_bad'] = pct_bad

                    if pct_bad > 10.0:
                        metrics['loo_reliable'] = False
                        self.logger.warning(
                            f"  PSIS-LOO WARNING: "
                            f"{pct_bad:.1f}% of observations "
                            f"have Pareto k > 0.7. "
                            f"PSIS approximation is unreliable. "
                            f"Preferring WAIC/DIC for this model. "
                            f"(Vehtari et al., 2017, Sec 3.4)"
                        )
                    else:
                        metrics['loo_reliable'] = True
                        self.logger.info(
                            f"  PSIS-LOO: elpd={metrics['loo']:.2f}"
                            f", Pareto-k OK "
                            f"({pct_bad:.1f}% > 0.7)"
                        )
                else:
                    # pareto_k not available; assume reliable
                    metrics['loo_reliable'] = True
                    self.logger.info(
                        f"  PSIS-LOO: elpd={metrics['loo']:.2f} "
                        f"(Pareto-k not available)"
                    )

            except Exception as e:
                self.logger.warning(
                    f"  PSIS-LOO derivation exception: {e}"
                )

            # --- WAIC ---
            try:
                waic_result = az.waic(infdata)
                metrics['waic'] = float(waic_result.elpd_waic)
                self.logger.info(
                    f"  WAIC: elpd={metrics['waic']:.2f}"
                )
            except Exception as e:
                self.logger.warning(
                    f"  WAIC derivation exception: {e}"
                )
        else:
            # DIC-only mode (expected when enable_loglike=False)
            # DIC is the default comparison criterion for HDDM
            # (Pan et al., 2025, Table 5; Spiegelhalter et al., 2002).
            # PSIS-LOO-CV and WAIC require pointwise log-likelihood
            # which is disabled by default to avoid 20-30 GB memory peaks.
            self.logger.info(
                f"  [{model_name}] DIC-only mode "
                f"(enable_loglike=False). "
                f"DIC={metrics.get('dic', 'pending'):.2f}"
            )

        return metrics

    # -----------------------------------------------------------------
    # WINNING MODEL DIAGNOSTICS
    # -----------------------------------------------------------------
    def generate_winner_diagnostics(
        self,
        model_name: str,
        infdata: az.InferenceData
    ):
        """
        Constructs comprehensive visualization suite for the
        selected optimal architecture: trace, rank, and pair plots.
        """
        self.logger.info(
            f"\nGenerating diagnostics for optimal architecture: "
            f"[{model_name.upper()}]"
        )

        all_vars = list(infdata.posterior.data_vars.keys())
        focal_vars = CFG.identify_focal_parameters(all_vars)

        if not focal_vars:
            self.logger.warning(
                "Focal parameters absent. "
                "Diagnostics bypassed."
            )
            return

        # 1. Trace plots
        try:
            az.plot_trace(
                infdata,
                var_names=focal_vars,
                compact=True,
                figsize=(12, 2.5 * len(focal_vars))
            )
            plt.savefig(
                PATHS['figures_supp']
                / f"winner_trace_{model_name}.pdf",
                dpi=300, bbox_inches='tight'
            )
            plt.close()
            self.logger.info("  Winner trace plot exported.")
        except Exception as e:
            self.logger.warning(
                f"  Trace plot failed: {e}"
            )

        # 2. Rank plots (Vehtari et al., 2021)
        try:
            az.plot_rank(
                infdata,
                var_names=focal_vars,
                kind='vlines',
                vlines_kwargs={'lw': 0},
                marker_vlines_kwargs={'lw': 2}
            )
            plt.savefig(
                PATHS['figures_supp']
                / f"winner_rank_{model_name}.pdf",
                dpi=300, bbox_inches='tight'
            )
            plt.close()
            self.logger.info("  Winner rank plot exported.")
        except Exception as e:
            self.logger.warning(
                f"  Rank plot failed: {e}"
            )

        # 3. Posterior pair plots (KDE)
        try:
            pair_vars = (
                focal_vars[:8]
                if len(focal_vars) > 8
                else focal_vars
            )
            az.plot_pair(
                infdata,
                var_names=pair_vars,
                kind='kde',
                marginals=True,
                figsize=(12, 12)
            )
            plt.savefig(
                PATHS['figures_supp']
                / f"winner_pair_{model_name}.pdf",
                dpi=300, bbox_inches='tight'
            )
            plt.close()
            self.logger.info("  Winner pair plot exported.")
        except Exception as e:
            self.logger.warning(
                f"  Pair plot failed: {e}"
            )

    # -----------------------------------------------------------------
    # FULL FUNNEL EXECUTION
    # -----------------------------------------------------------------
    def execute_funnel(self):
        """
        Iterates all architectures through the sequential
        four-level evaluation funnel. Populates
        self.comparison_records for downstream model selection.
        """
        for model_name in CFG.final_all_models:
            self.logger.info(f"\n{'─'*50}")
            self.logger.info(
                f"Evaluating: [{model_name.upper()}]"
            )
            self.logger.info(f"{'─'*50}")

            tier = CFG.mcmc_protocols.get(
                model_name, {}
            ).get('tier', 'unknown')

            record = {
                'config_hash': self.active_hash,
                'model_name': model_name,
                'tier': tier,
                'L1_technical': False,
                'L2_converged': False,
                'L3_ppc_adequate': False,
                'dic': float('inf'),
                'loo_elpd': np.nan,
                'loo_reliable': False,
                'pareto_k_pct_bad': np.nan,
                'waic_elpd': np.nan,
                'f_rhat_max': np.nan,
                'f_ess_bulk_min': np.nan,
                'total_kept_samples': np.nan,
                'focal_mcse_mean': np.nan,
                'choice_mae': np.nan,
                'rt_mae': np.nan,
                'overall_pass': False,
                'rejection_reason': 'Pending Evaluation'
            }

            # LEVEL 1
            if not self.evaluate_level1_technical(model_name):
                record['rejection_reason'] = (
                    'L1: Artifact Verification Failed'
                )
                self.comparison_records.append(record)
                self.logger.info(
                    f"  TERMINATED at Level 1 "
                    f"(Artifact Verification)"
                )
                continue

            record['L1_technical'] = True

            nc_path = (
                PATHS['models'] / f"hddm_{model_name}.nc"
            )
            infdata = az.from_netcdf(str(nc_path))

            # LEVEL 2
            l2_result = self.evaluate_level2_convergence(
                infdata, model_name
            )
            record['L2_converged'] = l2_result['pass']
            record['f_rhat_max'] = l2_result.get(
                'f_rhat_max', np.nan
            )
            record['f_ess_bulk_min'] = l2_result.get(
                'f_ess_bulk_min', np.nan
            )

            # Extract total kept samples and focal MCSE for
            # cross-model Monte Carlo quality comparison.
            # These are reported regardless of L2 pass/fail to
            # enable reviewers to assess whether adaptive sampling
            # introduced asymmetric MC noise across models.
            try:
                post = infdata.posterior
                n_chains = post.dims.get('chain', 0)
                n_draws = post.dims.get('draw', 0)
                record['total_kept_samples'] = n_chains * n_draws

                # Focal MCSE: mean of MCSE(mean) across focal params
                summary_tmp = az.summary(
                    infdata, round_to=6, hdi_prob=0.95
                )
                focal_tmp = CFG.identify_focal_parameters(
                    summary_tmp.index.tolist()
                )
                if ('mcse_mean' in summary_tmp.columns
                        and focal_tmp):
                    focal_mcse = summary_tmp.loc[
                        summary_tmp.index.isin(focal_tmp),
                        'mcse_mean'
                    ]
                    record['focal_mcse_mean'] = float(
                        focal_mcse.mean()
                    )
                    self.logger.info(
                        f"  MC quality: "
                        f"{record['total_kept_samples']} kept, "
                        f"focal MCSE(mean)="
                        f"{record['focal_mcse_mean']:.5f}"
                    )
            except Exception as e:
                self.logger.warning(
                    f"  MC quality extraction failed: {e}"
                )

            if not l2_result['pass']:
                record['rejection_reason'] = (
                    f"L2: {l2_result['reason']}"
                )
                self.comparison_records.append(record)
                self.logger.info(
                    f"  TERMINATED at Level 2: "
                    f"{l2_result['reason']}"
                )
                del infdata
                gc.collect()
                continue

            self.logger.info(
                f"  Level 2 Constraints Satisfied"
            )

            # LEVEL 3
            l3_result = self.evaluate_level3_ppc(model_name)
            record['L3_ppc_adequate'] = l3_result['pass']
            record['choice_mae'] = l3_result.get(
                'choice_mae', np.nan
            )
            record['rt_mae'] = l3_result.get('rt_mae', np.nan)

            if not l3_result['pass']:
                record['rejection_reason'] = (
                    f"L3: Predictive Inadequacy "
                    f"(Choice MAE="
                    f"{l3_result['choice_mae']:.4f}, "
                    f"RT MAE={l3_result['rt_mae']:.4f})"
                )
                self.comparison_records.append(record)
                self.logger.info(
                    f"  TERMINATED at Level 3: "
                    f"{record['rejection_reason']}"
                )
                del infdata
                gc.collect()
                continue

            self.logger.info(
                f"  Level 3 Constraints Satisfied "
                f"(Choice MAE="
                f"{l3_result['choice_mae']:.4f})"
            )

            # LEVEL 4
            l4_metrics = self.compute_model_comparison_metrics(
                model_name, infdata
            )
            record['dic'] = l4_metrics['dic']
            record['loo_elpd'] = l4_metrics.get('loo', np.nan)
            record['loo_reliable'] = l4_metrics.get(
                'loo_reliable', False
            )
            record['pareto_k_pct_bad'] = l4_metrics.get(
                'pareto_k_pct_bad', np.nan
            )
            record['waic_elpd'] = l4_metrics.get(
                'waic', np.nan
            )
            record['overall_pass'] = True
            record['rejection_reason'] = (
                'Satisfied Diagnostic Funnel'
            )

            self.comparison_records.append(record)

            self.logger.info(
                f"  Level 4: DIC={l4_metrics['dic']:.2f}, "
                f"LOO elpd="
                f"{l4_metrics.get('loo', 'N/A')}, "
                f"WAIC elpd="
                f"{l4_metrics.get('waic', 'N/A')}"
            )
            self.logger.info(f"  STATUS: FUNNEL COMPLETED")

            del infdata
            gc.collect()

    # -----------------------------------------------------------------
    # OPTIMAL MODEL SELECTION & EXPORT
    # Pareto-k aware selection priority
    # -----------------------------------------------------------------
    def select_optimal_model(self) -> Optional[str]:
        """
        Determines the optimal architecture from the set of
        architectures that satisfied Levels 1-3.

        Selection priority:
          1. DIC minimization (primary operational criterion;
             Spiegelhalter et al., 2002; Pan et al., 2025).
          2. PSIS-LOO-CV ELPD (if log-likelihood available and
             Pareto-k < 0.7 for >90% of observations).
          3. WAIC ELPD (if LOO unreliable but log-likelihood
             available).
        """
        df = pd.DataFrame(self.comparison_records)

        if df.empty:
            raise RuntimeError(
                "Funnel execution produced no records. "
                "Ensure execute_funnel() was called first."
            )

        eligible = df[df['overall_pass'] == True].copy()

        if eligible.empty:
            self.logger.error(
                "\nCRITICAL: Zero architectures satisfied "
                "the diagnostic funnel."
            )
            df.to_csv(
                PATHS['tables_main']
                / "model_comparison_summary.csv",
                index=False
            )
            raise RuntimeError(
                "Funnel collapsed. All architectures rejected. "
                "Consult log metrics for adjustments."
            )

        # --- Selection priority logic ---
        # Check LOO reliability via Pareto-k
        has_reliable_loo = (
            'loo_elpd' in eligible.columns
            and 'loo_reliable' in eligible.columns
            and eligible['loo_reliable'].any()
        )

        has_waic = (
            'waic_elpd' in eligible.columns
            and not eligible['waic_elpd'].isna().all()
        )

        if has_reliable_loo:
            # Use only models with reliable LOO
            reliable_mask = eligible['loo_reliable'] == True
            if reliable_mask.any():
                reliable_subset = eligible[reliable_mask]
                winner_idx = reliable_subset['loo_elpd'].idxmax()
                selection_method = "PSIS-LOO-CV (Pareto-k verified)"
            else:
                # All LOO unreliable, fall through to WAIC
                has_reliable_loo = False

        if not has_reliable_loo and has_waic:
            winner_idx = eligible['waic_elpd'].idxmax()
            selection_method = "WAIC (LOO unreliable or unavailable)"
            self.logger.info(
                "  Selection via WAIC (LOO unavailable or "
                "Pareto-k > 0.7 for >10% of observations)."
            )

        if not has_reliable_loo and not has_waic:
            # DIC as primary criterion (default when loglike=False).
            # DIC is the standard comparison metric in the HDDM
            # ecosystem (Pan et al., 2025, Table 5). While Gelman
            # et al. (2014) note theoretical limitations for
            # hierarchical models, DIC remains well-validated for
            # DDM model selection in practice.
            winner_idx = eligible['dic'].idxmin()
            selection_method = (
                "DIC (primary; Spiegelhalter et al., 2002)"
            )
            self.logger.info(
                f"  Selection via DIC (primary criterion). "
                f"DIC={eligible.at[winner_idx, 'dic']:.2f}"
            )

        winning_model = df.at[winner_idx, 'model_name']
        win_reason = (
            f"OPTIMAL ({selection_method} among "
            f"{len(eligible)} candidates)"
        )

        df['Is_Winner'] = False
        df.at[winner_idx, 'Is_Winner'] = True
        df.at[winner_idx, 'rejection_reason'] = win_reason

        df = df.sort_values(
            by=['overall_pass', 'dic'],
            ascending=[False, True]
        )

        # Export comparison table
        df.to_csv(
            PATHS['tables_main']
            / "model_comparison_summary.csv",
            index=False
        )

        # Export audit record for Steps 5-6
        audit_record = df[df['Is_Winner'] == True].copy()
        audit_record.to_csv(
            PATHS['audit']
            / "final_model_selection_audit.csv",
            index=False
        )

        # Log winner
        self.logger.info(f"\n{'='*70}")
        self.logger.info(
            f"OPTIMAL ARCHITECTURE: [{winning_model.upper()}]"
        )
        self.logger.info(f"  Selection Method: {selection_method}")

        # Report key metric for the winner
        winner_row = df.loc[winner_idx]
        if has_reliable_loo:
            self.logger.info(
                f"  LOO-CV elpd={winner_row['loo_elpd']:.2f}, "
                f"Pareto-k bad="
                f"{winner_row.get('pareto_k_pct_bad', 0):.1f}%"
            )
        elif has_waic:
            self.logger.info(
                f"  WAIC elpd={winner_row['waic_elpd']:.2f}"
            )
        else:
            self.logger.info(
                f"  DIC={winner_row['dic']:.2f}"
            )
        self.logger.info(f"{'='*70}")

        # Consistency check between IC methods
        if (not eligible['loo_elpd'].isna().all()
                and not eligible['dic'].isna().all()):
            dic_best = eligible.loc[
                eligible['dic'].idxmin(), 'model_name'
            ]
            consistent = (dic_best == winning_model)
            self.logger.info(
                f"  DIC Optimum: [{dic_best}] "
                f"({'CONSISTENT' if consistent else 'INCONSISTENT'}"
                f" with primary IC)"
            )

        # Display summary table
        display_cols = [
            'model_name', 'tier',
            'L1_technical', 'L2_converged',
            'L3_ppc_adequate', 'overall_pass',
            'total_kept_samples', 'focal_mcse_mean',
            'loo_elpd', 'loo_reliable',
            'waic_elpd', 'dic',
            'f_rhat_max', 'rejection_reason'
        ]
        available = [c for c in display_cols if c in df.columns]
        print("\n--- Diagnostic Funnel Resolution ---")
        print(df[available].to_string(index=False))

        # Generate winner diagnostics
        nc_path = (
            PATHS['models'] / f"hddm_{winning_model}.nc"
        )
        if nc_path.exists():
            infdata = az.from_netcdf(str(nc_path))
            self.generate_winner_diagnostics(
                winning_model, infdata
            )

            # ArviZ compare with Pareto-k info
            if hasattr(infdata, 'log_likelihood'):
                self._try_arviz_compare(eligible)

            del infdata
            gc.collect()

        return winning_model

    # -----------------------------------------------------------------
    # MODEL SHORTLIST FOR MULTI-MODEL REPORTING
    # -----------------------------------------------------------------
    def select_shortlist(self, delta_dic_threshold=15.0):
        """
        Identifies a shortlist of candidate models within a DIC
        threshold of the optimal model, enabling multi-model
        reporting in Step 5 when close competitors exist.

        The shortlist is exported as JSON (machine-readable, consumed
        by Step 5) and CSV (human-readable audit trail). The optimal
        model (Is_Winner=True) is always first in the list.

        Selection logic:
          1. Start with all models that passed the diagnostic funnel.
          2. Sort by DIC ascending.
          3. Include all models within delta_dic_threshold of the
             DIC-optimal model.

        Parameters
        ----------
        delta_dic_threshold : float
            Maximum DIC difference from the optimal model for
            inclusion. Default 15.0, following Burnham & Anderson
            (2002): ΔDIC < 10 = substantial support, ΔDIC < 15 =
            worth reporting.

        Returns
        -------
        list of str
            Shortlisted model names, winner first.
        """
        df = pd.DataFrame(self.comparison_records)
        eligible = df[df['overall_pass'] == True].copy()

        if eligible.empty:
            self.logger.warning(
                "No eligible models for shortlist."
            )
            return []

        eligible = eligible.sort_values(
            'dic', ascending=True
        )
        best_dic = eligible['dic'].iloc[0]

        shortlisted = eligible[
            eligible['dic'] <= best_dic + delta_dic_threshold
        ]['model_name'].tolist()

        # Build shortlist metadata
        shortlist_records = []
        for model_name in shortlisted:
            row = eligible[
                eligible['model_name'] == model_name
            ].iloc[0]
            shortlist_records.append({
                'model_name': model_name,
                'dic': float(row['dic']),
                'delta_dic': float(row['dic'] - best_dic),
                'is_winner': model_name == shortlisted[0],
                'choice_mae': float(
                    row.get('choice_mae', np.nan)
                ),
                'rt_mae': float(
                    row.get('rt_mae', np.nan)
                ),
            })

        # Export JSON for Step 5 consumption
        shortlist_path = (
            PATHS['audit'] / "model_shortlist.json"
        )
        with open(shortlist_path, 'w', encoding='utf-8') as f:
            json.dump({
                'delta_dic_threshold': delta_dic_threshold,
                'shortlist': shortlist_records
            }, f, indent=2)

        # Export CSV for human readability
        pd.DataFrame(shortlist_records).to_csv(
            PATHS['audit'] / "model_shortlist.csv",
            index=False
        )

        self.logger.info(
            f"\n  Model Shortlist "
            f"(ΔDIC < {delta_dic_threshold}):"
        )
        for rec in shortlist_records:
            marker = (
                " <- OPTIMAL" if rec['is_winner'] else ""
            )
            self.logger.info(
                f"    {rec['model_name'].upper()}: "
                f"DIC={rec['dic']:.2f} "
                f"(ΔDIC={rec['delta_dic']:+.2f})"
                f"{marker}"
            )

        self.logger.info(
            f"  Shortlist exported: {shortlist_path.name} "
            f"({len(shortlisted)} models)"
        )

        return shortlisted

    # -----------------------------------------------------------------
    # ARVIZ MODEL COMPARISON TABLE
    # -----------------------------------------------------------------
    def _try_arviz_compare(self, eligible_df: pd.DataFrame):
        """
        Derives formal ArviZ model comparison tables for viable
        models containing valid log_likelihood structures.
        Includes Pareto-k warning information.
        """
        compare_dict = {}
        for _, row in eligible_df.iterrows():
            model_name = row['model_name']
            nc_path = (
                PATHS['models'] / f"hddm_{model_name}.nc"
            )
            if nc_path.exists():
                idata = az.from_netcdf(str(nc_path))
                if hasattr(idata, 'log_likelihood'):
                    compare_dict[model_name] = idata

        if len(compare_dict) < 2:
            for idata in compare_dict.values():
                del idata
            gc.collect()
            return

        try:
            loo_compare = az.compare(compare_dict, ic='loo')
            loo_compare.to_csv(
                PATHS['tables_main']
                / "arviz_loo_comparison.csv"
            )
            self.logger.info(
                "  ArviZ LOO-CV comparison table exported."
            )

            # Check if warning column exists
            if 'warning' in loo_compare.columns:
                warned = loo_compare[
                    loo_compare['warning'] == True
                ]
                if not warned.empty:
                    self.logger.warning(
                        f"  Pareto-k warnings in LOO compare "
                        f"for: {warned.index.tolist()}"
                    )

        except Exception as e:
            self.logger.warning(
                f"  ArviZ LOO comparison exception: {e}"
            )

        try:
            waic_compare = az.compare(compare_dict, ic='waic')
            waic_compare.to_csv(
                PATHS['tables_main']
                / "arviz_waic_comparison.csv"
            )
            self.logger.info(
                "  ArviZ WAIC comparison table exported."
            )
        except Exception as e:
            self.logger.warning(
                f"  ArviZ WAIC comparison exception: {e}"
            )
        finally:
            for idata in compare_dict.values():
                del idata
            gc.collect()


# =============================================================================
# PIPELINE EXECUTION ENTRY POINT
# Added select_shortlist() for multi-model Step 5 support.
# =============================================================================
funnel = DiagnosticFunnelEngine()
funnel.execute_funnel()
winning_model = funnel.select_optimal_model()
shortlist = funnel.select_shortlist(delta_dic_threshold=15.0)

# Step 5: Statistical Inference and Publication-Ready Visualization

In [ ]:
# -*- coding: utf-8 -*-
"""
=============================================================================
STEP 5: Statistical Inference and Publication-Ready Visualization
=============================================================================
Pipeline Position:
  Upstream:   Step 4 (reads final_model_selection_audit.csv from audit/)
              Step 2b (reads .nc file of winning model from models/)
  Downstream: None (terminal visualization step)

Methodological Purpose:
  - Loads the winning model's ArviZ InferenceData (.nc) and generates
    publication-grade Bayesian posterior visualizations.
  - Calibrates ROPE half-widths empirically from the posterior as a
    fixed fraction of the mean within-contrast SD, with a robustness check
    across ROPE widths.
  - Produces per-varying-parameter Ridge plots with HDI, ROPE, and
    posterior probability of direction (Pd) annotations.
  - Produces per-fixed-parameter single-density plots with HDI.
  - Generates condition-level RT distribution mirror plots showing
    observed Accept/Reject RT distributions per emotion condition.
  - Exports comprehensive inference summary CSV with both contrast
    and absolute parameter statistics.
  - Exports ROPE calibration audit CSV documenting the empirical
    basis for each ROPE range.

Outputs per varying parameter:
  - posterior_ridge_{p}_{model}.png/pdf
Outputs per fixed parameter:
  - posterior_fixed_{p}_{model}.png/pdf
Outputs (once):
  - rt_mirror_{model}.png/pdf
  - inference_summary_{model}.csv
  - rope_calibration_{model}.csv

=============================================================================
"""

import gc
import warnings
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import arviz as az
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter
from scipy.stats import gaussian_kde

warnings.filterwarnings('ignore', category=FutureWarning)

# =============================================================================
# VISUAL CONSTANTS
# =============================================================================

# Emotion condition styling (Okabe-Ito inspired, colorblind-friendly)
EMOTION_STYLE = {
    'dis': {'color': '#E64B35', 'label': 'Disgust',     'order': 0},
    'dom': {'color': '#F39B7F', 'label': 'Dominance',   'order': 1},
    'neu': {'color': '#8491B4', 'label': 'Neutral',     'order': 2},
    'aff': {'color': '#00A087', 'label': 'Affiliative', 'order': 3},
    'rew': {'color': '#3C5488', 'label': 'Reward',      'order': 4},
}

# Mirror plot palette
ACC_COLOR = "#004D40"   # Deep teal (Accept / upper boundary)
REJ_COLOR = "#4A148C"   # Deep purple (Reject / lower boundary)
ACC_ALPHA = 0.52
REJ_ALPHA = 0.42

# ROPE fallback defaults (used only if calibration fails for a parameter)
ROPE_FALLBACK = {
    'v': (-0.05, 0.05), 'a': (-0.01, 0.01),
    'z': (-0.005, 0.005), 't': (-0.015, 0.015)
}

# Minimum ROPE half-width per parameter family.
# Prevents degenerate ROPEs when posterior SD is very small.
ROPE_FLOOR = {
    'v': 0.02, 'a': 0.005, 'z': 0.002, 't': 0.005
}

def calibrate_rope(idata, pvars, baseline='neu',
                   scale_factor=0.1):
    """
    Calibrates ROPE half-widths on an effect-size scale (Kruschke, 2018):
    a practically negligible effect is taken as scale_factor (default 0.1)
    times the BETWEEN-SUBJECT SD of the baseline parameter -- roughly half
    of Cohen's "small" effect (d = 0.2) expressed relative to the
    population dispersion. Robustness of each decision to the ROPE width
    is assessed by rope_sensitivity_analysis().

    Why between-subject SD rather than posterior SD:
      A ROPE set as a fraction of the *posterior* SD of a contrast (a
      standard-error-like quantity that shrinks as the sample grows) makes
      the negligibility threshold a function of measurement precision: more
      data -> narrower ROPE -> effects declared non-negligible more easily,
      the paradox Kruschke (2018) warns against. Anchoring to the
      between-subject SD ties the ROPE to a substantive, sample-size-stable
      scale. The half-width is floored at ROPE_FLOOR[p] purely as a
      degeneracy guard; no cosmetic rounding is applied, so the reported
      ROPE is exactly scale_factor x SD and does not silently shift any
      HDI-vs-ROPE decision.

    Method:
      For each DDM parameter family p the between-subject SD is the
      posterior mean of the hierarchical group-level SD node
      '{p}_Intercept_std'. If that node is unavailable, the function falls
      back to the previous precision-referenced scale (mean within-contrast
      posterior SD for varying parameters, intercept posterior SD for fixed
      parameters) and flags the fallback in the audit trail.

    Parameters
    ----------
    idata : az.InferenceData
        Posterior samples from the winning model.
    pvars : list
        Variable names in the posterior.
    baseline : str
        Baseline condition code (default 'neu').
    scale_factor : float
        Fraction of the between-subject SD used as the ROPE half-width
        (default 0.1; half of Cohen's "small" effect).

    Returns
    -------
    rope_dict : dict
        {param: (lo, hi)} ROPE ranges per parameter family.
    calibration_records : list of dict
        Per-parameter calibration audit trail.
    """
    rope_dict = {}
    records = []

    def _between_subject_sd(param):
        """Posterior mean of the hierarchical group-level SD node."""
        node = f"{param}_Intercept_std"
        if node in pvars:
            return float(np.mean(idata.posterior[node].values))
        return None

    for p in ['v', 'a', 'z', 't']:
        # Identify contrast variables for this parameter
        contrast_vars = [
            v for v in pvars
            if f"{p}_C(emotion" in v and '[T.' in v
        ]

        iname = f"{p}_Intercept"
        has_intercept = iname in pvars

        sd_between = _between_subject_sd(p)

        if contrast_vars or has_intercept:
            if sd_between is not None:
                sd_anchor = sd_between
                sd_source = f"{p}_Intercept_std (between-subject SD)"
                method = (
                    f'Effect-size referenced '
                    f'(+/-{scale_factor} x between-subject SD)'
                )
            else:
                # Fallback: precision-referenced scale
                if contrast_vars:
                    all_samples = [
                        idata.posterior[cv].values.reshape(-1)
                        for cv in contrast_vars
                    ]
                    sd_anchor = float(
                        np.mean([np.std(s) for s in all_samples])
                    )
                    sd_source = 'mean within-contrast posterior SD'
                else:
                    sd_anchor = float(np.std(
                        idata.posterior[iname].values.reshape(-1)
                    ))
                    sd_source = 'intercept posterior SD'
                method = (
                    f'Fallback precision-referenced '
                    f'(+/-{scale_factor} x {sd_source})'
                )

            raw_half = scale_factor * sd_anchor
            floored = max(raw_half, ROPE_FLOOR.get(p, 0.001))
            final_half = round(floored, 4)

            rope_dict[p] = (-final_half, final_half)

            records.append({
                'parameter': p,
                'type': 'varying' if contrast_vars else 'fixed',
                'n_conditions': len(contrast_vars),
                'sd_between_subject': (
                    round(sd_between, 5)
                    if sd_between is not None else np.nan
                ),
                'sd_anchor': round(sd_anchor, 5),
                'sd_source': sd_source,
                'raw_half_width': round(raw_half, 5),
                'floor_applied': floored > raw_half,
                'final_half_width': final_half,
                'rope_lo': -final_half,
                'rope_hi': final_half,
                'method': method,
            })

        else:
            # Parameter not in model: use fallback constant
            fallback = ROPE_FALLBACK.get(p, (-0.05, 0.05))
            rope_dict[p] = fallback
            records.append({
                'parameter': p,
                'type': 'absent',
                'final_half_width': abs(fallback[1]),
                'rope_lo': fallback[0],
                'rope_hi': fallback[1],
                'method': 'Fallback (parameter not in model)',
            })

    return rope_dict, records

# Publication-standard parameter display names
PARAM_LABELS = {
    'v': 'Drift Rate (v)', 'a': 'Decision Threshold (a)',
    'z': 'Starting Point Bias (z)', 't': 'Non-decision Time (t)'
}


# =============================================================================
# STYLE INITIALIZATION
# =============================================================================
def _setup_style():
    """
    Configures matplotlib rcParams for APA / Nature Human Behaviour
    publication standards: Type 42 fonts for vector embedding,
    despined axes, and appropriate font sizing hierarchy.
    """
    plt.rcParams.update({
        'font.family': 'sans-serif',
        'font.sans-serif': ['DejaVu Sans', 'Helvetica', 'Arial'],
        'font.size': 11, 'axes.labelsize': 13,
        'axes.titlesize': 14, 'axes.titleweight': 'bold',
        'axes.spines.top': False, 'axes.spines.right': False,
        'axes.linewidth': 0.8,
        'xtick.labelsize': 11, 'ytick.labelsize': 11,
        'legend.fontsize': 10, 'legend.frameon': False,
        'figure.dpi': 150, 'savefig.dpi': 300,
        'savefig.bbox': 'tight', 'savefig.pad_inches': 0.3,
        'pdf.fonttype': 42, 'ps.fonttype': 42,
    })


# =============================================================================
# HELPER FUNCTIONS
# =============================================================================
def _detect_conditions(pvars):
    """Extracts non-baseline condition codes from Treatment-coded
    variable names."""
    return sorted({
        v.split('[T.')[1].rstrip(']')
        for v in pvars if '[T.' in v
    })


def _detect_varying(pvars):
    """Identifies DDM parameter families that have condition-varying
    regressors."""
    return [
        p for p in ['v', 'a', 't', 'z']
        if any(f"{p}_C(emotion" in v for v in pvars)
    ]


def _lbl(e, es):
    """Returns display label for an emotion condition code."""
    return es.get(e, {}).get('label', e)


def _clr(e, es):
    """Returns display color for an emotion condition code."""
    return es.get(e, {}).get('color', '#888')


# =============================================================================
# CORE CLASS
# =============================================================================
class HDDMPosteriorVisualizer:
    """
    Publication-grade posterior visualization engine for HDDM models.

    Consumes ArviZ InferenceData (.nc) for posterior extraction.
    Produces ridge plots, fixed-parameter density plots, observed
    RT mirror plots, and inference summary tables.
    """

    def __init__(self, nc_path, model_name, baseline='neu',
                 output_dir='figures', rope_overrides=None,
                 emotion_style=None, data_path=None):
        """
        Parameters:
            nc_path:        Path to ArviZ InferenceData (.nc file)
            model_name:     Model architecture name (e.g., 'va', 'vaz')
            baseline:       Baseline emotion condition code
            output_dir:     Directory for figure output
            rope_overrides: Custom ROPE ranges per parameter family
                            (overrides calibrated values if provided)
            emotion_style:  Custom emotion color/label/order dict
            data_path:      Path to hddm_data_unfair.csv for RT plots
        """
        _setup_style()
        self.nc_path = Path(nc_path)
        self.model_name = model_name.lower()
        self.baseline = baseline
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.es = emotion_style or EMOTION_STYLE
        self.data_path = Path(data_path) if data_path else None

        # Load InferenceData for posterior extraction
        self.idata = az.from_netcdf(str(self.nc_path))
        self.pvars = list(self.idata.posterior.data_vars)
        self.varying = _detect_varying(self.pvars)
        self.conditions = sorted(
            [self.baseline] + [
                c for c in _detect_conditions(self.pvars)
                if c != self.baseline
            ],
            key=lambda x: self.es.get(x, {}).get('order', 99))

        # Calibrate ROPE from posterior data
        self.rope, self.rope_calibration = calibrate_rope(
            self.idata, self.pvars, self.baseline
        )
        # Apply manual overrides if provided
        if rope_overrides:
            self.rope.update(rope_overrides)

        print(f"Model: {self.model_name.upper()}")
        print(f"Varying: {self.varying}")
        print(f"Conditions: {self.conditions}")
        print(f"ROPE (calibrated):")
        for p in ['v', 'a', 'z', 't']:
            if p in self.rope:
                lo, hi = self.rope[p]
                print(f"  {p}: [{lo:+.3f}, {hi:+.3f}]")

    # -----------------------------------------------------------------
    # Sample extraction
    # -----------------------------------------------------------------
    def _flat(self, n):
        """Flattens posterior samples across chains and draws into
        1D array."""
        return self.idata.posterior[n].values.reshape(-1)

    def _contrasts(self, p):
        """Extracts Treatment contrast posteriors
        (condition - baseline)."""
        if p not in self.varying:
            return {}
        res = {}
        for e in self.conditions:
            if e == self.baseline:
                continue
            cn = (
                f"{p}_C(emotion, Treatment('{self.baseline}'))"
                f"[T.{e}]"
            )
            if cn in self.pvars:
                res[e] = self._flat(cn)
        return res

    def _absolute(self, p):
        """
        Reconstructs absolute parameter values per condition.
        For the baseline: absolute = Intercept.
        For non-baseline: absolute = Intercept + Treatment contrast.
        """
        iname = f"{p}_Intercept"
        if iname not in self.pvars:
            return {}
        ic = self._flat(iname)
        res = {self.baseline: ic}
        if p in self.varying:
            for e in self.conditions:
                if e == self.baseline:
                    continue
                cn = (
                    f"{p}_C(emotion, Treatment('{self.baseline}'))"
                    f"[T.{e}]"
                )
                if cn in self.pvars:
                    res[e] = ic + self._flat(cn)
        return res

    @staticmethod
    def _hdi(s, prob=0.95):
        """
        Computes the Highest Density Interval (HDI) for a 1D sample
        array. Uses the narrowest-interval method (Kruschke, 2015).
        """
        s = np.sort(s)
        n = len(s)
        w = int(np.ceil(prob * n))
        if w >= n:
            return (float(s[0]), float(s[-1]))
        widths = s[w:] - s[:n - w]
        idx = int(np.argmin(widths))
        return (float(s[idx]), float(s[idx + w]))

    # =================================================================
    # RIDGE DENSITY PLOT (main results figure for varying parameters)
    # =================================================================
    def plot_ridge(self, param, figsize=(10, None), bw=0.15,
                   row_height=1.8):
        """
        Ridge plot: per-condition posterior density + HDI + ROPE +
        stats. Each row represents one non-baseline condition's
        Treatment contrast posterior (condition - baseline).
        """
        cd = self._contrasts(param)
        if not cd:
            return None

        sorted_e = sorted(
            cd.keys(),
            key=lambda x: self.es.get(x, {}).get('order', 99))
        n = len(sorted_e)
        bl = _lbl(self.baseline, self.es)
        rope = self.rope.get(param, (-0.1, 0.1))

        if figsize[1] is None:
            fig_h = max(4.0, n * row_height + 2.0)
            figsize = (figsize[0], fig_h)

        fig, ax = plt.subplots(figsize=figsize)

        all_v = np.concatenate(list(cd.values()))
        data_pad = (all_v.max() - all_v.min()) * 0.12
        xmin = all_v.min() - data_pad
        xmax = all_v.max() + data_pad
        ann_margin = (xmax - xmin) * 0.42
        xg = np.linspace(xmin, xmax, 500)

        # Zero reference only (no ROPE band; effects are reported by the
        # 95% HDI and probability of direction, not a ROPE decision rule)
        ax.axvline(0, color='#424242', ls='-', lw=0.8, alpha=0.5,
                   zorder=1)

        # Compute densities with shared global normalization
        dens = {}
        gmax = 0
        for e in sorted_e:
            kde = gaussian_kde(cd[e], bw_method=bw)
            d = kde(xg)
            dens[e] = d
            gmax = max(gmax, d.max())

        # Draw each condition row
        for i, e in enumerate(sorted_e):
            y_base = i * row_height
            c = _clr(e, self.es)
            s = cd[e]
            d = (dens[e] / gmax * (row_height * 0.82)
                 if gmax > 0 else dens[e])

            h95 = self._hdi(s, 0.95)
            h89 = self._hdi(s, 0.89)
            med = float(np.median(s))
            ex0 = h95[0] > 0 or h95[1] < 0

            # Density fill and outline
            ax.fill_between(xg, y_base, y_base + d,
                            alpha=0.25, color=c, lw=0, zorder=2)
            ax.plot(xg, y_base + d, color=c, lw=1.5, alpha=0.85,
                    zorder=3)

            # Row baseline
            ax.axhline(y_base, color='#EEEEEE', lw=0.4, zorder=0)

            # 95% HDI thin bar with tick marks
            ax.plot(h95, [y_base, y_base], color=c, lw=2.0,
                    solid_capstyle='butt', zorder=5)
            tick_h = row_height * 0.06
            for xval in h95:
                ax.plot([xval, xval],
                        [y_base - tick_h, y_base + tick_h],
                        color=c, lw=1.2, zorder=5)

            # 89% HDI thick bar
            ax.plot(h89, [y_base, y_base], color=c, lw=5.5,
                    solid_capstyle='round', alpha=0.60, zorder=6)

            # Median dot
            ax.plot(med, y_base, 'o', color=c, ms=6,
                    markeredgecolor='white', markeredgewidth=1.0,
                    zorder=7)

            # Right annotation: posterior median and 95% HDI. Whether the
            # HDI excludes zero is shown by its position relative to the
            # zero line and by bold styling of the interval (no text flag).
            fw = 'bold' if ex0 else 'normal'

            ax.text(xmax + ann_margin * 0.03,
                    y_base + row_height * 0.30,
                    f"{med:+.3f}", fontsize=10.5, color=c,
                    fontweight='bold', va='center', ha='left',
                    clip_on=False)
            ax.text(xmax + ann_margin * 0.03,
                    y_base + row_height * 0.08,
                    f"[{h95[0]:+.3f}, {h95[1]:+.3f}]",
                    fontsize=8.5, color=c, fontweight=fw,
                    va='center', ha='left', clip_on=False)

        # Y-axis labels
        ax.set_yticks([i * row_height + row_height * 0.30
                       for i in range(n)])
        ax.set_yticklabels([_lbl(e, self.es) for e in sorted_e],
                           fontsize=11.5, fontweight='bold')
        for tl, e in zip(ax.get_yticklabels(), sorted_e):
            tl.set_color(_clr(e, self.es))

        ax.set_xlim(xmin, xmax + ann_margin)
        ax.set_ylim(-0.4, n * row_height + 0.3)
        ax.set_xlabel(
            f'Δ{PARAM_LABELS.get(param, param)} (vs {bl})')
        ax.set_title(
            f'{PARAM_LABELS.get(param, param)}: '
            f'Posterior Contrasts vs {bl}',
            pad=15)
        ax.spines['left'].set_visible(False)
        ax.tick_params(left=False)
        ax.grid(axis='x', alpha=0.10, ls='--')

        # Legend
        legend_elements = [
            Line2D([0], [0], color='#424242', lw=0.8, alpha=0.5,
                   label='Zero (no effect)'),
            Line2D([0], [0], color='#666', lw=5.5, alpha=0.60,
                   solid_capstyle='round',
                   label='89% HDI'),
            Line2D([0], [0], color='#666', lw=2.0,
                   label='95% HDI'),
            Line2D([0], [0], color='#666', marker='o', lw=0,
                   ms=6, markeredgecolor='white',
                   markeredgewidth=1.0,
                   label='Posterior median'),
        ]
        ax.legend(
            handles=legend_elements,
            loc='upper right',
            bbox_to_anchor=(1.0, 1.0),
            fontsize=8.5, frameon=True,
            fancybox=True, framealpha=0.92,
            edgecolor='#DDD',
            handlelength=2.0, handleheight=1.2,
        )

        fig.tight_layout()
        return fig

    # =================================================================
    # FIXED PARAMETER DENSITY
    # =================================================================
    def plot_fixed(self, param, figsize=(7, 4.5), bw=0.15):
        """
        Single-density plot for parameters that do not vary by
        condition. Displays 95% HDI shaded region, median line,
        and reference lines (0.5 for z, 0 for v).
        """
        iname = f"{param}_Intercept"
        if iname not in self.pvars:
            return None

        s = self._flat(iname)
        lo, hi = self._hdi(s, 0.95)
        med = float(np.median(s))

        fig, ax = plt.subplots(figsize=figsize)
        kde = gaussian_kde(s, bw_method=bw)
        pad = (s.max() - s.min()) * 0.25
        xg = np.linspace(s.min() - pad, s.max() + pad, 500)
        d = kde(xg)
        c = _clr(self.baseline, self.es)

        ax.fill_between(xg, d, alpha=0.25, color=c)
        ax.plot(xg, d, color=c, lw=2.0)
        mask = (xg >= lo) & (xg <= hi)
        ax.fill_between(xg, d, where=mask, alpha=0.35, color=c,
                         label='95% HDI')
        ax.axvline(med, color=c, ls='--', lw=1.5, alpha=0.7)

        bar_y = -0.04 * d.max()
        ax.plot([lo, hi], [bar_y, bar_y], color=c, lw=4.0,
                solid_capstyle='round', zorder=5)
        ax.plot(med, bar_y, 'o', color=c, ms=7,
                markeredgecolor='white', markeredgewidth=1.2,
                zorder=6)

        ax.text(0.97, 0.95,
                f'Median = {med:.3f}\n'
                f'95% HDI = [{lo:.3f}, {hi:.3f}]',
                transform=ax.transAxes, fontsize=9,
                va='top', ha='right',
                bbox=dict(boxstyle='round,pad=0.4', fc='white',
                          ec='#ccc', alpha=0.9))

        if param == 'z':
            ax.axvline(0.5, color='red', ls='--', lw=1.0,
                       alpha=0.5, label='Unbiased (0.5)')
        if param == 'v':
            ax.axvline(0, color='red', ls='--', lw=1.0,
                       alpha=0.4, label='Zero')

        ax.set_xlabel(PARAM_LABELS.get(param, param))
        ax.set_ylabel('Density')
        ax.set_title(f'{PARAM_LABELS.get(param, param)} '
                     f'(Fixed Across Conditions)')
        ax.set_ylim(bottom=bar_y * 2.5)
        ax.legend(loc='upper left', frameon=False, fontsize=9)
        fig.tight_layout()
        return fig

    # =================================================================
    # ABSOLUTE PARAMETER POSTERIOR DENSITY PLOT
    # Follows Wiecki, Sofer & Frank (2013, Figure 5):
    # overlapping posterior densities per condition.
    # =================================================================
    def plot_absolute(self, params=None, figsize=None):
        """
        Overlapping posterior density plot showing reconstructed
        absolute parameter values per condition.

        Follows the canonical HDDM visualization paradigm from
        Wiecki, Sofer & Frank (2013, Frontiers in Human Neuroscience,
        Figure 5): multiple conditions' posterior densities overlaid
        on the same axis, each in a distinct color. This shows:
          - Full posterior distribution shape per condition
          - Degree of overlap (visual separability)
          - Median and 95% HDI per condition in legend
          - Reference lines (zero for v, 0.5 for z)

        Layout: one subplot per varying parameter, vertically stacked.

        Parameters
        ----------
        params : list of str, optional
            Which parameter families to plot. Default: all varying.
        figsize : tuple, optional
            Figure size. Auto-scaled if None.

        Returns
        -------
        matplotlib Figure or None if no varying parameters found.
        """
        if params is None:
            params = [
                p for p in self.varying
                if self._absolute(p)
            ]

        if not params:
            return None

        n_panels = len(params)
        if figsize is None:
            figsize = (10, 3.5 * n_panels)

        fig, axes = plt.subplots(
            n_panels, 1, figsize=figsize, squeeze=False
        )

        for panel_idx, param in enumerate(params):
            ax = axes[panel_idx, 0]
            ab = self._absolute(param)
            if not ab:
                ax.axis('off')
                continue

            # Sort conditions by display order
            sorted_conds = sorted(
                ab.keys(),
                key=lambda x: self.es.get(
                    x, {}
                ).get('order', 99)
            )

            # Plot each condition's posterior density
            for cond in sorted_conds:
                samples = ab[cond]
                c = _clr(cond, self.es)
                lbl = _lbl(cond, self.es)
                med = float(np.median(samples))
                lo, hi = self._hdi(samples, 0.95)

                if cond == self.baseline:
                    lbl += ' (baseline)'

                # Posterior density curve
                kde = gaussian_kde(samples, bw_method=0.15)
                pad = (samples.max() - samples.min()) * 0.3
                xg = np.linspace(
                    samples.min() - pad,
                    samples.max() + pad, 500
                )
                d = kde(xg)

                ax.fill_between(
                    xg, d, alpha=0.15, color=c, lw=0
                )
                ax.plot(
                    xg, d, color=c, lw=2.0, alpha=0.85,
                    label=(
                        f'{lbl}  ({med:.2f} '
                        f'[{lo:.2f}, {hi:.2f}])'
                    )
                )

                # Median tick mark on x-axis
                ax.plot(
                    med, 0, '|', color=c, ms=12, mew=2.0,
                    zorder=5
                )

            # Reference lines
            if param == 'v':
                ax.axvline(
                    0, color='#888', ls='--', lw=0.8,
                    alpha=0.4, zorder=1
                )
            if param == 'z':
                ax.axvline(
                    0.5, color='#888', ls='--', lw=0.8,
                    alpha=0.4, zorder=1
                )

            ax.set_xlabel(
                PARAM_LABELS.get(param, param), fontsize=12
            )
            ax.set_ylabel('Posterior Density', fontsize=11)
            ax.set_title(
                PARAM_LABELS.get(param, param),
                fontsize=13, fontweight='bold', pad=10
            )

            # Remove y-axis values (density is arbitrary)
            ax.set_yticks([])
            ax.spines['left'].set_visible(False)

            # Legend with median [HDI]
            ax.legend(
                fontsize=9, loc='upper right',
                frameon=True, fancybox=True,
                framealpha=0.9, edgecolor='#DDD'
            )

        if n_panels > 1:
            fig.suptitle(
                'DDM Parameter Posteriors by Condition',
                fontsize=14, fontweight='bold', y=1.02
            )

        fig.tight_layout()
        return fig

    # =================================================================
    # OBSERVED-ONLY RT MIRROR PLOT
    # =================================================================
    def plot_rt_mirror(self, figsize=(14, 9)):
        """
        Per-emotion RT distribution using mirror-style layout.
        Shows observed data only; PPC diagnostics are handled
        separately in Step 3.

        Structure per panel:
          - Upper half (positive y): Accept (response=1) RT histogram
          - Lower half (negative y): Reject (response=0) RT histogram

        Returns:
          matplotlib Figure or None if data_path is missing.
        """
        if self.data_path is None or not self.data_path.exists():
            print("  RT mirror plot skipped: data_path not "
                  "provided or not found.")
            return None

        df = pd.read_csv(self.data_path)
        emotions = sorted(
            df['emotion'].unique(),
            key=lambda x: self.es.get(x, {}).get('order', 99))

        n_emo = len(emotions)
        ncols = 3
        nrows = int(np.ceil((n_emo + 1) / ncols))

        fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
        axes_flat = axes.flatten()

        bins = np.linspace(0, 2.5, 40)

        # Overall panel (all emotions pooled)
        self._draw_mirror_panel(
            df, axes_flat[0], 'Overall', bins
        )

        # Per-emotion panels
        for idx, emo in enumerate(emotions):
            emo_df = df[df['emotion'] == emo]
            lbl = _lbl(emo, self.es)
            clr = _clr(emo, self.es)
            self._draw_mirror_panel(
                emo_df, axes_flat[idx + 1], lbl, bins,
                title_color=clr
            )

        # Hide unused axes
        for j in range(n_emo + 1, len(axes_flat)):
            axes_flat[j].axis('off')

        # Figure legend
        handles = [
            mpatches.Patch(facecolor=ACC_COLOR, alpha=ACC_ALPHA,
                           label='Accept'),
            mpatches.Patch(facecolor=REJ_COLOR, alpha=REJ_ALPHA,
                           label='Reject'),
        ]
        fig.legend(
            handles=handles, loc='lower center',
            ncol=len(handles), frameon=True, fancybox=True,
            bbox_to_anchor=(0.5, -0.02), fontsize=10
        )

        fig.suptitle(
            'RT Distributions by Emotion (Unfair Offers)',
            fontsize=15, fontweight='bold', y=1.01
        )
        fig.tight_layout(rect=[0.02, 0.02, 0.98, 0.97])
        return fig

    @staticmethod
    def _draw_mirror_panel(data, ax, title, bins,
                           title_color='#333'):
        """
        Mirror histogram panel: Accept (upper) / Reject (lower).
        Observed data only, no PPC overlay.

        Parameters:
          data:          DataFrame with 'rt' and 'response' columns
          ax:            matplotlib Axes target
          title:         Panel title string
          bins:          Histogram bin edges
          title_color:   Title text color
        """
        accepted = data[data['response'] == 1]['rt']
        rejected = data[data['response'] == 0]['rt']

        # Use absolute RT values for histogram
        # (HDDM stimulus coding: negative RT = lower boundary)
        acc_rt = np.abs(accepted.values)
        rej_rt = np.abs(rejected.values)

        hist_acc, _ = np.histogram(acc_rt, bins=bins)
        hist_rej, _ = np.histogram(rej_rt, bins=bins)

        bin_centers = (bins[:-1] + bins[1:]) / 2
        width = bins[1] - bins[0]

        # Accept histogram (upper half, positive frequency)
        ax.barh(bin_centers, hist_acc, height=width * 0.8,
                alpha=ACC_ALPHA, facecolor=ACC_COLOR, zorder=2)

        # Reject histogram (lower half, negative frequency)
        ax.barh(bin_centers, -hist_rej, height=width * 0.8,
                alpha=REJ_ALPHA, facecolor=REJ_COLOR, zorder=2)

        # Title with descriptive statistics
        if len(data) > 0:
            acc_rate = data['response'].mean()
            n_total = len(data)
            title_full = (
                f'{title}\n'
                f'(N={n_total}, Accept: {acc_rate:.1%}, '
                f'Reject: {1-acc_rate:.1%})'
            )
        else:
            title_full = title

        ax.set_title(title_full, fontsize=10, fontweight='bold',
                     color=title_color, pad=5)
        ax.set_ylabel('RT (s)')
        ax.set_xlabel('Frequency')
        ax.axvline(0, color='black', lw=0.5)
        ax.grid(True, alpha=0.2, axis='y')

        # Absolute value formatter for frequency axis
        ax.xaxis.set_major_formatter(
            FuncFormatter(lambda x, _: f'{int(abs(x))}'))

    # =================================================================
    # INFERENCE SUMMARY TABLE (with absolute values)
    # =================================================================
    def export_inference_table(self):
        """
        Export comprehensive inference table including both contrast
        statistics and absolute parameter values per condition.

        For each varying parameter and each non-baseline condition:
          - Contrast statistics: median, mean, 95% HDI, Pd,
            ROPE overlap percentage, HDI-excludes-zero flag
          - Absolute statistics: median and 95% HDI of the
            reconstructed condition-level parameter value

        Baseline absolute values are appended with contrast = 0.

        Returns:
          DataFrame with all inference statistics.
        """
        rows = []
        for p in self.varying:
            cd = self._contrasts(p)
            ab = self._absolute(p)
            rope = self.rope.get(p, (-0.1, 0.1))

            for e, s in cd.items():
                lo, hi = self._hdi(s, 0.95)
                med = float(np.median(s))
                mn = float(np.mean(s))
                pd_v = (
                    float((s > 0).mean()) if mn > 0
                    else float((s < 0).mean())
                )
                rp = float(
                    ((s >= rope[0]) & (s <= rope[1])).mean() * 100
                )

                abs_med = (
                    float(np.median(ab[e])) if e in ab
                    else np.nan
                )
                abs_hdi = (
                    self._hdi(ab[e], 0.95) if e in ab
                    else (np.nan, np.nan)
                )

                rows.append({
                    'Parameter': PARAM_LABELS.get(p, p),
                    'Condition': _lbl(e, self.es),
                    'Contrast_Median': round(med, 4),
                    'Contrast_Mean': round(mn, 4),
                    'Contrast_HDI95_lo': round(lo, 4),
                    'Contrast_HDI95_hi': round(hi, 4),
                    'Pd': round(pd_v, 4),
                    'ROPE_overlap_pct': round(rp, 2),
                    'HDI_excludes_zero': lo > 0 or hi < 0,
                    'Absolute_Median': round(abs_med, 4),
                    'Absolute_HDI95_lo': round(abs_hdi[0], 4),
                    'Absolute_HDI95_hi': round(abs_hdi[1], 4),
                })

        # Append baseline absolute values
        for p in self.varying:
            iname = f"{p}_Intercept"
            if iname in self.pvars:
                s = self._flat(iname)
                med = float(np.median(s))
                lo, hi = self._hdi(s, 0.95)
                rows.append({
                    'Parameter': PARAM_LABELS.get(p, p),
                    'Condition': (
                        f'{_lbl(self.baseline, self.es)} '
                        f'(baseline)'
                    ),
                    'Contrast_Median': 0.0,
                    'Contrast_Mean': 0.0,
                    'Contrast_HDI95_lo': 0.0,
                    'Contrast_HDI95_hi': 0.0,
                    'Pd': np.nan,
                    'ROPE_overlap_pct': np.nan,
                    'HDI_excludes_zero': False,
                    'Absolute_Median': round(med, 4),
                    'Absolute_HDI95_lo': round(lo, 4),
                    'Absolute_HDI95_hi': round(hi, 4),
                })

        df = pd.DataFrame(rows)
        path = (
            self.output_dir
            / f'inference_summary_{self.model_name}.csv'
        )
        df.to_csv(path, index=False)
        print(f"Inference table saved: {path}")
        return df

    # =================================================================
    # ROPE SENSITIVITY ANALYSIS
    # =================================================================
    def rope_sensitivity_analysis(self, scales=(0.5, 1.0, 1.5)):
        """
        Tests robustness of inferential conclusions across ROPE
        widths. For each varying parameter and each condition
        contrast, evaluates three decision categories at multiple
        ROPE scales:

          - 'Significant': 95% HDI fully excludes ROPE
          - 'Negligible':  95% HDI fully inside ROPE
          - 'Undecided':   95% HDI partially overlaps ROPE

        A robust conclusion shows identical decisions across all
        tested scales.

        Parameters
        ----------
        scales : tuple of float
            Multiplicative factors applied to the calibrated ROPE
            half-width. Default (0.5, 1.0, 1.5) tests 50%, 100%,
            and 150% of the calibrated value.

        Returns
        -------
        pd.DataFrame
            Sensitivity table with one row per parameter × condition
            × scale, including decision category and flip flag.
        """
        rows = []

        for p in self.varying:
            cd = self._contrasts(p)
            base_rope = self.rope.get(p, (-0.05, 0.05))
            base_half = abs(base_rope[1])

            for e, s in cd.items():
                h95 = self._hdi(s, 0.95)

                for scale in scales:
                    half = base_half * scale
                    rope_lo, rope_hi = -half, half

                    # Decision logic
                    hdi_above_rope = h95[0] > rope_hi
                    hdi_below_rope = h95[1] < rope_lo
                    hdi_inside_rope = (
                        h95[0] >= rope_lo and h95[1] <= rope_hi
                    )

                    if hdi_above_rope or hdi_below_rope:
                        decision = 'Significant'
                    elif hdi_inside_rope:
                        decision = 'Negligible'
                    else:
                        decision = 'Undecided'

                    rows.append({
                        'Parameter': PARAM_LABELS.get(p, p),
                        'Condition': _lbl(e, self.es),
                        'ROPE_scale': scale,
                        'ROPE_half': round(half, 4),
                        'ROPE_range': f'[{-half:+.4f}, {half:+.4f}]',
                        'HDI95_lo': round(h95[0], 4),
                        'HDI95_hi': round(h95[1], 4),
                        'Decision': decision,
                    })

        df = pd.DataFrame(rows)

        # Flag flips: a decision is stable if it does not change across the
        # tested scales. Assign index-aligned via groupby-transform (a prior
        # positional append attached the flag to the wrong rows because the
        # groupby-sorted order differs from the DataFrame row order).
        df['Stable'] = (
            df.groupby(['Parameter', 'Condition'])['Decision']
              .transform(lambda s: s.nunique() == 1)
        )

        # Export
        path = (
            self.output_dir
            / f'rope_sensitivity_{self.model_name}.csv'
        )
        df.to_csv(path, index=False)

        # Console summary
        n_total = len(
            df[df['ROPE_scale'] == 1.0]
        )
        n_stable = df.groupby(
            ['Parameter', 'Condition']
        )['Stable'].first().sum()
        n_flip = n_total - n_stable

        print(f"  ROPE sensitivity: {n_stable}/{n_total} "
              f"contrasts stable across ×0.5–×1.5 "
              f"({n_flip} flipped)")

        if n_flip > 0:
            flipped = df[~df['Stable']].drop_duplicates(
                subset=['Parameter', 'Condition']
            )
            for _, row in flipped.iterrows():
                print(f"    FLIP: {row['Parameter']} / "
                      f"{row['Condition']}")

        return df

    # =================================================================
    # POSTERIOR PARAMETER CORRELATION ANALYSIS
    # =================================================================
    def posterior_correlation_analysis(self, figsize=(8, 7)):
        """
        Computes and visualizes pairwise Pearson correlations between
        all varying-parameter Treatment contrasts to detect parameter
        trade-offs (e.g., v-z compensation).

        Strong negative correlations between parameters for the same
        condition suggest statistical compensation: the model trades
        off one parameter's effect against another to achieve similar
        fit. This is important for interpreting parameters whose
        effects seem counterintuitive.

        Outputs:
          - posterior_correlations_{model}.csv  (full correlation matrix)
          - posterior_correlations_{model}.png/pdf  (heatmap)

        Returns
        -------
        pd.DataFrame
            Correlation matrix.
        """
        # Collect all contrast posteriors
        labels = []
        samples = []

        for p in self.varying:
            cd = self._contrasts(p)
            for e, s in cd.items():
                label = f"{p}_{_lbl(e, self.es)}"
                labels.append(label)
                samples.append(s)

        if len(labels) < 2:
            print("  Posterior correlations: fewer than 2 "
                  "contrast parameters. Skipped.")
            return None

        # Build sample matrix (n_samples × n_params)
        # Truncate to shortest chain length for alignment
        min_len = min(len(s) for s in samples)
        matrix = np.column_stack(
            [s[:min_len] for s in samples]
        )
        corr_df = pd.DataFrame(
            np.corrcoef(matrix, rowvar=False),
            index=labels, columns=labels
        )

        # Export CSV
        csv_path = (
            self.output_dir
            / f'posterior_correlations_{self.model_name}.csv'
        )
        corr_df.to_csv(csv_path)

        # Heatmap visualization
        import seaborn as sns

        fig, ax = plt.subplots(figsize=figsize)

        mask = np.triu(np.ones_like(corr_df, dtype=bool), k=1)
        sns.heatmap(
            corr_df, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5,
            cbar_kws={'label': 'Pearson r', 'shrink': 0.8},
            ax=ax
        )

        ax.set_title(
            'Posterior Contrast Correlations\n'
            '(detecting parameter trade-offs)',
            fontsize=13, fontweight='bold', pad=15
        )
        plt.xticks(rotation=45, ha='right', fontsize=9)
        plt.yticks(rotation=0, fontsize=9)
        fig.tight_layout()

        for ext in ['png', 'pdf']:
            fig.savefig(
                self.output_dir
                / f'posterior_correlations_'
                  f'{self.model_name}.{ext}'
            )
        plt.close(fig)

        # Console summary: flag strong correlations (|r| > 0.3)
        print(f"  Posterior correlations exported.")
        n_params = len(labels)
        strong_pairs = []
        for i in range(n_params):
            for j in range(i + 1, n_params):
                r = corr_df.iloc[i, j]
                if abs(r) > 0.3:
                    strong_pairs.append(
                        (labels[i], labels[j], r)
                    )

        if strong_pairs:
            print(f"  Strong correlations (|r|>0.3):")
            for l1, l2, r in sorted(
                strong_pairs, key=lambda x: abs(x[2]),
                reverse=True
            ):
                direction = (
                    'compensation' if r < 0 else 'co-variation'
                )
                print(f"    {l1} <-> {l2}: r={r:+.3f} "
                      f"({direction})")
        else:
            print(f"  No strong correlations (|r|>0.3) detected.")

        return corr_df

    # =================================================================
    # RENDER ALL
    # =================================================================
    def render_all(self):
        """Generate all publication figures and inference table."""

        # Export ROPE calibration audit trail
        if self.rope_calibration:
            rope_df = pd.DataFrame(self.rope_calibration)
            rope_path = (
                self.output_dir
                / f'rope_calibration_{self.model_name}.csv'
            )
            rope_df.to_csv(rope_path, index=False)
            print(f"  ROPE calibration exported: {rope_path.name}")

        # Per-parameter plots
        for p in ['v', 'a', 't', 'z']:
            if f"{p}_Intercept" not in self.pvars:
                print(f"  [{p}] Not found in posterior. Skipped.")
                continue

            if p in self.varying:
                # Contrast ridge plot
                fig = self.plot_ridge(p)
                if fig:
                    for ext in ['png', 'pdf']:
                        fig.savefig(
                            self.output_dir
                            / f'posterior_ridge_{p}_'
                              f'{self.model_name}.{ext}')
                    plt.close(fig)
                    print(f"  [{p}] Ridge plot exported.")
            else:
                fig = self.plot_fixed(p)
                if fig:
                    for ext in ['png', 'pdf']:
                        fig.savefig(
                            self.output_dir
                            / f'posterior_fixed_{p}_'
                              f'{self.model_name}.{ext}')
                    plt.close(fig)
                    print(f"  [{p}] Fixed density exported.")

        # Combined absolute parameter value panel plot
        fig_abs = self.plot_absolute()
        if fig_abs:
            for ext in ['png', 'pdf']:
                fig_abs.savefig(
                    self.output_dir
                    / f'posterior_absolute_{self.model_name}.{ext}')
            plt.close(fig_abs)
            print(f"  Absolute parameter plot exported.")

        # Observed-only RT mirror plot
        fig = self.plot_rt_mirror()
        if fig:
            for ext in ['png', 'pdf']:
                fig.savefig(
                    self.output_dir
                    / f'rt_mirror_{self.model_name}.{ext}')
            plt.close(fig)
            print("  RT mirror plot exported.")

        # Inference summary table
        self.export_inference_table()

        # ROPE sensitivity analysis
        self.rope_sensitivity_analysis()

        # Posterior parameter correlation analysis
        self.posterior_correlation_analysis()

        print(f"\nAll outputs saved to: {self.output_dir}")


# =============================================================================
# MULTI-MODEL VISUALIZATION ENTRY POINT
# Supports: (1) Auto shortlist from Step 4, (2) Manual override.
# =============================================================================
def run_visualization_pipeline(
    model_override=None,
    shortlist_path=None
):
    """
    Multi-model visualization entry point.

    Supports three modes of model selection (in priority order):
      1. model_override: Explicit list of model names.
         Example: model_override=['va', 'vaz']
      2. shortlist_path: Path to model_shortlist.json from Step 4.
      3. Auto-detection: Reads model_shortlist.json from standard
         audit directory. Falls back to single winner from
         final_model_selection_audit.csv if shortlist absent.

    Each model gets its own output subdirectory under figures_main:
      figures_main/{model_name}/

    Parameters
    ----------
    model_override : list of str, optional
        Explicit model names to process. Overrides all auto-detection.
    shortlist_path : str or Path, optional
        Custom path to model_shortlist.json.
    """
    import json as _json

    try:
        from hddm_config import CFG, identify_winning_model
        PATHS = CFG.initialize_directories()
    except ImportError as e:
        raise RuntimeError(
            "Cannot import hddm_config. "
            "Ensure Step 2a has completed."
        ) from e

    data_path = CFG.base_dir / 'hddm_data_unfair.csv'

    # -----------------------------------------------------------------
    # Resolve model list
    # -----------------------------------------------------------------
    if model_override is not None:
        # Mode 1: Explicit override
        models_to_render = list(model_override)
        print(f"Mode: Manual override")
        print(f"Models: {[m.upper() for m in models_to_render]}")

    else:
        # Mode 2/3: Read from shortlist or fallback to winner
        if shortlist_path is not None:
            sl_path = Path(shortlist_path)
        else:
            sl_path = PATHS['audit'] / "model_shortlist.json"

        if sl_path.exists():
            with open(sl_path, 'r', encoding='utf-8') as f:
                sl_data = _json.load(f)
            models_to_render = [
                rec['model_name']
                for rec in sl_data['shortlist']
            ]
            threshold = sl_data.get(
                'delta_dic_threshold', 'N/A'
            )
            print(
                f"Mode: Shortlist from Step 4 "
                f"(ΔDIC < {threshold})"
            )
            print(
                f"Models: "
                f"{[m.upper() for m in models_to_render]}"
            )
        else:
            # Fallback: single winner
            try:
                winner = identify_winning_model(PATHS)
                models_to_render = [winner]
                print(f"Mode: Single winner fallback")
                print(f"Models: [{winner.upper()}]")
            except Exception as e:
                raise RuntimeError(
                    f"Cannot resolve models: {e}. "
                    f"Provide model_override explicitly."
                ) from e

    # -----------------------------------------------------------------
    # Render each model
    # -----------------------------------------------------------------
    for i, model_name in enumerate(models_to_render):
        nc_path = PATHS['models'] / f'hddm_{model_name}.nc'
        if not nc_path.exists():
            print(
                f"\n[{model_name.upper()}] .nc file not found. "
                f"Skipped."
            )
            continue

        # Each model gets its own output subdirectory
        out_dir = PATHS['figures_main'] / model_name
        out_dir.mkdir(parents=True, exist_ok=True)

        print(f"\n{'='*60}")
        print(
            f"Rendering [{model_name.upper()}] "
            f"({i+1}/{len(models_to_render)})"
        )
        print(f"{'='*60}")

        engine = HDDMPosteriorVisualizer(
            nc_path=str(nc_path),
            model_name=model_name,
            output_dir=str(out_dir),
            data_path=str(data_path),
        )
        engine.render_all()

    print(f"\n{'='*60}")
    print(
        f"STEP 5 COMPLETE: {len(models_to_render)} model(s) "
        f"rendered."
    )
    print(f"{'='*60}")


# =============================================================================
# PIPELINE EXECUTION ENTRY POINT
# =============================================================================
# Default: auto-detect from Step 4 shortlist
# Override: run_visualization_pipeline(model_override=['va', 'vaz'])
run_visualization_pipeline()

# Step 6: Data Informed Group-Level Parameter Recovery

In [ ]:
# -*- coding: utf-8 -*-
"""
=============================================================================
STEP 6: Multi-Iteration Dual-Track Parameter Recovery Validation
=============================================================================
Pipeline Position:
  Upstream:   Step 4 (reads final_model_selection_audit.csv to identify
              winning model; reads .nc file from models/)
              Step 1 (reads hddm_data_unfair.csv for design skeleton)
  Downstream: None (terminal validation step)

Methodological Purpose:
  - Validates the structural identifiability of the specified model
    via two complementary forward-simulation-and-refitting paradigms:
    1. Posterior-Anchored: Extracts pseudo-truth from the empirical
       joint posterior to evaluate self-consistency.
    2. Prior-Predictive: Samples pseudo-truth from ecologically valid
       informative priors to stress-test identifiability.
  - Executes N_ITERATIONS (default 20) independent recovery cycles
    per mode to obtain distributional estimates of Bias, RMSE,
    Coverage Rate, and Correlation (Wilson & Collins, 2019, eLife).
  - Constructs synthetic datasets preserving the empirical trial
    skeleton WITH subject-level parameter variability drawn from
    the hierarchical prior structure (Lerche & Voss, 2016).
  - Validates refit convergence before computing recovery metrics.
  - Implements memory guard, incremental result persistence, and
    checkpoint resume to survive interruptions.

=============================================================================
"""

import os
import gc
import time
import logging
import traceback
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import arviz as az
import hddm
from hddm.generate import gen_rand_data

from scipy.special import expit, logit
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import seaborn as sns

# -----------------------------------------------------------------------------
# CONFIGURATION IMPORT
# -----------------------------------------------------------------------------
try:
    from hddm_config import (
        CFG,
        load_active_lineage_state,
        identify_winning_model,
        check_memory_headroom
    )
except ImportError:
    raise ImportError(
        "CRITICAL ERROR: 'hddm_config.py' unresolved. "
        "Execution of Step 2a is mandatory."
    )

PATHS = CFG.initialize_directories()

# Deterministic seed from configuration
np.random.seed(CFG.base_seed)

# Publication-grade aesthetics
sns.set_theme(style="ticks", palette="colorblind")
OKABE_ITO = [
    '#E69F00', '#56B4E9', '#009E73', '#F0E442',
    '#0072B2', '#D55E00', '#CC79A7', '#000000'
]
plt.rcParams.update({
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.prop_cycle': plt.cycler(color=OKABE_ITO),
    'font.size': 11,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'legend.frameon': False,
    'figure.autolayout': True
})

# =============================================================================
# RECOVERY CONFIGURATION CONSTANTS
# =============================================================================
# 20 iterations per mode provides sufficient distributional stability
# for Bias, RMSE, and Coverage while keeping total runtime feasible
# for HDDMRegressor recovery refits (2-5 hours each).
N_ITERATIONS_FINAL = 20
N_ITERATIONS_DEBUG = 3

# Consecutive genuine exceptions (MemoryError, numerical failures)
# before terminating a recovery mode.
MAX_CONSECUTIVE_FAILURES = 5

# Convergence threshold for recovery refits. Relaxed relative to
# Step 2b main fitting (1.01 focal / 1.05 nuisance) because
# recovery refits use lighter MCMC budgets.
RECOVERY_RHAT_THRESHOLD = 1.10

# Adaptive sampling constants for recovery refits.
# Recovery-specific: lighter ceilings than Step 2b main fitting
# to bound total compute per iteration.
RECOVERY_MAX_ADAPTIVE_CYCLES = 3
RECOVERY_MAX_RHAT_EXTENSIONS = 2
RECOVERY_RHAT_IMPROVEMENT_MIN = 0.005

# Recovery-specific ESS thresholds. Relaxed relative to Step 2b
# (1000 bulk / 500 tail) because recovery only needs sufficient
# precision for bias/coverage estimation, not full posterior
# characterization.
RECOVERY_ESS_BULK_FOCAL = 400.0
RECOVERY_ESS_TAIL_FOCAL = 200.0


# =============================================================================
# LOGGING SETUP
# =============================================================================
def _setup_recovery_logger() -> logging.Logger:
    """Initializes dual-sink logging for recovery module."""
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    logger = logging.getLogger(f'hddm_recovery_{ts}')
    logger.handlers = []
    logger.setLevel(logging.INFO)

    fmt = logging.Formatter(
        '%(asctime)s - %(levelname)s - %(message)s',
        datefmt='%H:%M:%S'
    )
    fh = logging.FileHandler(
        PATHS['audit'] / f'dual_recovery_{ts}.log',
        encoding='utf-8'
    )
    ch = logging.StreamHandler()
    fh.setFormatter(fmt)
    ch.setFormatter(fmt)

    logger.addHandler(fh)
    logger.addHandler(ch)
    return logger


# =============================================================================
# INPUT DATA VALIDATION (ported from Step 2b)
# =============================================================================
def validate_empirical_data(
    data_path: str, logger: logging.Logger
) -> pd.DataFrame:
    """
    Enforces HDDM structural requirements and datatype constraints
    on the empirical design matrix used as recovery skeleton.

    Validates:
      - Mandatory columns: subj_idx, rt, response
      - RT values are numeric and positive
      - Response values are binary {0, 1}
      - Baseline emotion condition exists in the data
      - No NaN values in critical columns
    """
    if not os.path.exists(data_path):
        raise FileNotFoundError(
            f"Input data path unresolved: {data_path}"
        )

    df = pd.read_csv(data_path)
    logger.info(
        f"Empirical data loaded: {df.shape[0]} trials, "
        f"{df.shape[1]} columns"
    )

    # HDDM structural dependency validation
    required_cols = {'subj_idx', 'rt', 'response'}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(
            f"Mandatory HDDM columns missing: {missing}"
        )

    # Datatype normalization
    df['subj_idx'] = df['subj_idx'].astype(str)
    df['rt'] = pd.to_numeric(df['rt'], errors='coerce')
    df['response'] = pd.to_numeric(df['response'], errors='coerce')

    # Data integrity enforcement: exclude NaN entries
    n_before = len(df)
    df = df.dropna(subset=['rt', 'response'])
    n_dropped = n_before - len(df)
    if n_dropped > 0:
        logger.warning(
            f"Data exclusion: {n_dropped} rows dropped "
            f"due to NaN rt/response."
        )

    # Baseline condition verification
    if 'emotion' in df.columns:
        if CFG.baseline_condition not in df['emotion'].unique():
            raise ValueError(
                f"Baseline condition '{CFG.baseline_condition}' "
                f"absent. Available levels: "
                f"{df['emotion'].unique().tolist()}"
            )

    n_subjects = df['subj_idx'].nunique()
    n_conditions = (
        df['emotion'].nunique() if 'emotion' in df.columns else 0
    )
    logger.info(
        f"Validation complete: {n_subjects} subjects, "
        f"{n_conditions} conditions, {len(df)} trials retained."
    )
    logger.info(
        f"RT bounds: [{df['rt'].min():.3f}, "
        f"{df['rt'].max():.3f}] seconds."
    )

    return df


# =============================================================================
# STRATIFIED CONVERGENCE DIAGNOSTICS (adapted from Step 2b)
# =============================================================================
def _evaluate_recovery_convergence(
    infdata: az.InferenceData, logger: logging.Logger
) -> Tuple[bool, Dict[str, float], pd.DataFrame]:
    """
    Computes Gelman-Rubin (R-hat) and Effective Sample Size (ESS)
    statistics with stratified thresholds for recovery refits.

    Focal parameters (group-level intercepts and treatment contrasts)
    require both R-hat and ESS evaluation. Nuisance parameters
    (subject-level deviations, variance terms) require R-hat only,
    following Vehtari et al. (2021) recommendations for hierarchical
    models with link-function-induced autocorrelation.

    Returns:
      converged:  Boolean indicating all criteria satisfied.
      metrics:    Dictionary of computed diagnostic extrema.
      summary_df: Full ArviZ statistical summary DataFrame.
    """
    summary_df = az.summary(infdata, round_to=4, hdi_prob=0.94)

    # Parameter stratification using CFG's canonical classifier
    focal_params = CFG.identify_focal_parameters(
        summary_df.index.tolist()
    )
    summary_df['param_class'] = [
        'focal' if p in focal_params else 'nuisance'
        for p in summary_df.index
    ]

    focal_df = summary_df[summary_df["param_class"] == "focal"]
    nuisance_df = summary_df[
        summary_df["param_class"] == "nuisance"
    ]

    # Extremum metric extraction
    metrics = {
        "f_rhat": (
            focal_df["r_hat"].max()
            if not focal_df.empty else 1.0
        ),
        "f_bulk": (
            focal_df["ess_bulk"].min()
            if not focal_df.empty else float("inf")
        ),
        "f_tail": (
            focal_df["ess_tail"].min()
            if ("ess_tail" in focal_df.columns
                and not focal_df.empty)
            else float("inf")
        ),
        "n_rhat": (
            nuisance_df["r_hat"].max()
            if not nuisance_df.empty else 1.0
        ),
        "n_bulk": (
            nuisance_df["ess_bulk"].min()
            if not nuisance_df.empty else float("inf")
        ),
        "n_tail": (
            nuisance_df["ess_tail"].min()
            if ("ess_tail" in nuisance_df.columns
                and not nuisance_df.empty)
            else float("inf")
        ),
    }

    # Criteria evaluation with recovery-specific thresholds
    focal_ok = (
        (metrics["f_rhat"] <= RECOVERY_RHAT_THRESHOLD)
        and (metrics["f_bulk"] >= RECOVERY_ESS_BULK_FOCAL)
        and (metrics["f_tail"] >= RECOVERY_ESS_TAIL_FOCAL)
    )

    # Nuisance convergence: R-hat only.
    nuisance_ok = (
        metrics["n_rhat"] <= RECOVERY_RHAT_THRESHOLD
    )

    converged = focal_ok and nuisance_ok

    logger.info(
        f"    FOCAL   | R-hat={metrics['f_rhat']:.3f} "
        f"(<={RECOVERY_RHAT_THRESHOLD}) | "
        f"ESS_bulk={metrics['f_bulk']:.0f} "
        f"(>={RECOVERY_ESS_BULK_FOCAL:.0f}) | "
        f"ESS_tail={metrics['f_tail']:.0f} "
        f"(>={RECOVERY_ESS_TAIL_FOCAL:.0f}) | "
        f"{'PASS' if focal_ok else 'FAIL'}"
    )
    logger.info(
        f"    NUISANCE| R-hat={metrics['n_rhat']:.3f} "
        f"(<={RECOVERY_RHAT_THRESHOLD}) | "
        f"ESS_bulk={metrics['n_bulk']:.0f} | "
        f"ESS_tail={metrics['n_tail']:.0f} | "
        f"{'PASS' if nuisance_ok else 'FAIL'}"
    )

    return converged, metrics, summary_df


# =============================================================================
# DYNAMIC REGRESSOR FACTORY
# =============================================================================
def _build_regressors(
    model_name: str, baseline: str
) -> List[str]:
    """
    Constructs Patsy formulas matching Step 2b specifications.
    ALL 4 DDM parameters get explicit formulas (treatment-coded
    or intercept-only) to ensure dockerHDDM compatibility.
    """
    name_lower = model_name.lower()
    regressors = []

    for param in ['v', 'a', 't', 'z']:
        if name_lower != 'null' and param in name_lower:
            regressors.append(
                f"{param} ~ C(emotion, "
                f"Treatment('{baseline}'))"
            )
        else:
            regressors.append(f"{param} ~ 1")

    return regressors


# =============================================================================
# FORWARD LINK FUNCTIONS — PHYSICAL → LP CONVERSION
# =============================================================================
def _physical_to_lp(
    param_family: str, value: float, is_intercept: bool
) -> float:
    """
    Maps a posterior-summary parameter value onto the scale that
    HDDMRegressor uses for forward simulation.

    Called with formula strings and the default link, HDDMRegressor uses
    the identity link for v, a and t, and the logit link for z. Intercepts
    are therefore returned unchanged for v/a/t and mapped with the logit
    for z. Treatment contrasts are additive deviations on the same scale
    and are returned unchanged for all families.

    Parameters:
      param_family:  One of 'v', 'a', 't', 'z'.
      value:         The raw value from az.summary() or a posterior draw.
      is_intercept:  True for Intercept parameters, False for Treatment
                     contrast coefficients.
    """
    # Treatment contrasts are already in LP scale — pass through
    if not is_intercept:
        return float(value)

    # Intercepts require physical → LP conversion
    if param_family == 'v':
        return float(value)
    elif param_family == 'a':
        # identity link: boundary separation is on the physical scale
        return float(value)
    elif param_family == 't':
        # identity link: non-decision time is on the physical scale
        return float(value)
    elif param_family == 'z':
        return float(logit(np.clip(value, 1e-6, 1.0 - 1e-6)))
    return float(value)


# =============================================================================
# CORE CLASS: SINGLE-MODEL PARAMETER RECOVERY ENGINE
# =============================================================================
class ParameterRecoveryEngine:
    """
    Orchestrates N-iteration Posterior-Anchored and Prior-Predictive
    identifiability evaluations for a single specified HDDM model.

    Simplified interface: accepts a single target_model
    string directly, without multi-model orchestration overhead.
    """

    def __init__(
        self,
        target_model: str,
        empirical_data_path: str = 'hddm_data_unfair.csv',
        n_iterations: Optional[int] = None
    ):
        self.logger = _setup_recovery_logger()

        self.active_lineage = load_active_lineage_state(
            PATHS, self.logger
        )
        self.active_hash = (
            self.active_lineage['pipeline_hash']
        )

        # Validate target model against canonical architecture list
        valid_models = [
            m.lower() for m in CFG.final_all_models
        ]
        target_lower = target_model.strip().lower()
        if target_lower not in valid_models:
            raise ValueError(
                f"Target model '{target_model}' is not a valid "
                f"architecture. Valid options: {valid_models}"
            )
        self.target_model = target_lower

        # Log the Step 4 audit winner for provenance record
        audit_winner = identify_winning_model(PATHS)
        self.logger.info(
            f"Step 4 audit winner: [{audit_winner.upper()}]"
        )
        self.logger.info(
            f"Recovery target (user-specified): "
            f"[{self.target_model.upper()}]"
        )

        self.empirical_data_path = empirical_data_path

        # Recovery MCMC hyperparameters (lighter than main fitting)
        if CFG.run_mode == 'debug':
            self.recovery_chains = 2
            self.recovery_samples = 500
            self.recovery_burn = 100
            self.n_iterations = N_ITERATIONS_DEBUG
        else:
            self.recovery_chains = CFG.n_chains
            self.recovery_samples = 3000
            self.recovery_burn = 1000
            self.n_iterations = N_ITERATIONS_FINAL

        # User override for iteration count (enables easy extension
        # without modifying global constants). Checkpoint resume
        # automatically skips already-completed iterations.
        if n_iterations is not None:
            self.n_iterations = n_iterations

        # Validate empirical data before use as skeleton
        self.empirical_df = validate_empirical_data(
            empirical_data_path, self.logger
        )
        self.n_subjects = self.empirical_df['subj_idx'].nunique()

        self.logger.info("=" * 70)
        self.logger.info(
            f"PARAMETER RECOVERY INITIATED "
            f"(Mode: {CFG.run_mode.upper()})"
        )
        self.logger.info(
            f"Pipeline Hash: {self.active_hash[:24]}..."
        )
        self.logger.info(
            f"Target Model: [{self.target_model.upper()}]"
        )
        self.logger.info(
            f"Iterations: {self.n_iterations} per mode"
        )
        self.logger.info(
            f"Recovery MCMC: {self.recovery_samples} samples, "
            f"{self.recovery_burn} burn, "
            f"{self.recovery_chains} chains"
        )
        self.logger.info(
            f"Design: {self.n_subjects} subjects"
        )
        self.logger.info(
            f"Adaptive: max {RECOVERY_MAX_ADAPTIVE_CYCLES} "
            f"cycles, ESS_bulk>={RECOVERY_ESS_BULK_FOCAL:.0f}, "
            f"ESS_tail>={RECOVERY_ESS_TAIL_FOCAL:.0f}"
        )
        self.logger.info("=" * 70)

    # -----------------------------------------------------------------
    # INVERSE LINK FUNCTIONS (LP → PHYSICAL)
    # -----------------------------------------------------------------
    @staticmethod
    def _apply_link_and_clip(
        param_family: str, lp_val: float
    ) -> float:
        """
        Applies the HDDMRegressor inverse links and enforces bounds.
        Identity for v, a and t; logit for z.
        v: identity, clipped to [-8, 8]
        a: identity, clipped to [0.2, 4.0]
        t: identity, clipped to [0.05, 2.0]
        z: expit, clipped to [0.05, 0.95]
        """
        if param_family == 'v':
            return float(np.clip(lp_val, -8.0, 8.0))
        elif param_family == 'a':
            return float(np.clip(lp_val, 0.2, 4.0))
        elif param_family == 't':
            return float(np.clip(lp_val, 0.05, 2.0))
        elif param_family == 'z':
            return float(np.clip(expit(lp_val), 0.05, 0.95))
        return lp_val

    # -----------------------------------------------------------------
    # GROUND TRUTH ESTABLISHMENT
    # With scale-aligned physical → LP conversion
    # -----------------------------------------------------------------
    def establish_ground_truth(
        self, mode: str, iteration: int
    ) -> Tuple[Dict[str, float], Dict[str, Dict[str, float]]]:
        """
        Synthesizes pseudo-truth parameter matrices.

        Returns:
          group_truth: Dict of group-level LP-scale parameters.
          subject_truths: Dict mapping subj_idx -> dict of
                         physical-scale parameters per condition.
                         Includes subject-level variability drawn
                         from hierarchical prior structure.

        Posterior mode: az.summary() of the fitted .nc returns parameter
        means on the scale HDDMRegressor uses (identity for v/a/t, logit
        for z). Intercepts are mapped to that scale with _physical_to_lp;
        treatment contrasts are additive deviations and are kept unchanged.
        The same mapping is applied when sampling an individual posterior
        draw (iteration > 0).
        """
        # Unique seed per mode x iteration
        iter_seed = (
            CFG.base_seed + iteration * 100
            + (0 if mode == 'posterior' else 50)
        )
        np.random.seed(iter_seed)

        group_truth = {}
        model_lower = self.target_model.lower()

        if mode == 'posterior':
            nc_path = (
                PATHS['models']
                / f"hddm_{self.target_model}.nc"
            )
            if not nc_path.exists():
                raise FileNotFoundError(
                    f"InferenceData unresolved: {nc_path.name}"
                )

            infdata = az.from_netcdf(str(nc_path))
            summary = az.summary(infdata, round_to=4)

            # Extract focal fixed-effect parameters
            focal_indices = [
                idx for idx in summary.index
                if ('Intercept' in idx or 'Treatment' in idx
                    or 'C(emotion' in idx)
                and not any(
                    s in idx for s in
                    ['_subj', '_std', '_var', '_log', '_trans']
                )
            ]

            # =========================================================
            # Map posterior-summary values onto the generative scale.
            # HDDMRegressor uses the identity link for v/a/t and the logit
            # link for z:
            #   Intercepts: v/a/t unchanged (physical = linear predictor);
            #               z mapped with the logit.
            #   Contrasts:  additive deviations on the same scale; unchanged.
            # =========================================================
            if iteration == 0:
                # Iteration 0: use posterior mean
                for param_name in focal_indices:
                    raw_val = float(summary.loc[param_name, 'mean'])
                    family = param_name.split('_')[0]
                    is_intercept = 'Intercept' in param_name
                    group_truth[param_name] = _physical_to_lp(
                        family, raw_val, is_intercept
                    )
            else:
                # Iteration > 0: sample a single posterior draw
                # Same Intercept/Contrast distinction applies
                post = infdata.posterior
                draw_idx = np.random.randint(
                    0, post.dims['draw']
                )
                chain_idx = np.random.randint(
                    0, post.dims['chain']
                )

                for param_name in focal_indices:
                    if param_name in post.data_vars:
                        raw_val = float(
                            post[param_name]
                            .values[chain_idx, draw_idx]
                        )
                    else:
                        # Fallback to summary mean if variable
                        # not found in posterior group
                        raw_val = float(
                            summary.loc[param_name, 'mean']
                        )

                    family = param_name.split('_')[0]
                    is_intercept = 'Intercept' in param_name
                    group_truth[param_name] = _physical_to_lp(
                        family, raw_val, is_intercept
                    )

            # Log ground truth for audit trail
            self.logger.info(
                f"    Ground truth ({mode}, iter {iteration}):"
            )
            for k, v in group_truth.items():
                family = k.split('_')[0]
                phys_check = self._apply_link_and_clip(family, v)
                self.logger.info(
                    f"      {k}: LP={v:.4f} → phys={phys_check:.4f}"
                )

            del infdata
            gc.collect()

        elif mode == 'prior':
            # Prior-predictive mode: sample from ecologically
            # valid informative priors. These are generated
            # directly in LP scale — no conversion needed.
            for param_family in ['v', 'a', 't', 'z']:
                is_varying = (
                    param_family in model_lower
                    and model_lower != 'null'
                )

                # Intercept synthesis (in LP scale)
                if param_family == 'v':
                    # v uses identity link: LP = physical
                    phys_val = stats.norm.rvs(
                        loc=1.0, scale=1.5
                    )
                    lp_int = phys_val
                elif param_family == 'a':
                    # identity link: parameter on the physical scale
                    phys_val = stats.gamma.rvs(
                        a=9.0, scale=0.166
                    )
                    phys_val = np.clip(phys_val, 0.5, 3.5)
                    lp_int = phys_val
                elif param_family == 't':
                    # identity link: parameter on the physical scale
                    phys_val = stats.truncnorm.rvs(
                        a=(0.1 - 0.3) / 0.1,
                        b=(0.5 - 0.3) / 0.1,
                        loc=0.3, scale=0.1
                    )
                    phys_val = np.clip(phys_val, 0.05, 0.8)
                    lp_int = phys_val
                elif param_family == 'z':
                    # z uses logit link: LP = logit(physical)
                    phys_val = stats.beta.rvs(a=5, b=5)
                    phys_val = np.clip(phys_val, 0.1, 0.9)
                    lp_int = logit(phys_val)
                else:
                    lp_int = 0.0

                group_truth[
                    f"{param_family}_Intercept"
                ] = float(lp_int)

                # Treatment contrasts (always in LP scale;
                # contrasts are additive deviations on LP scale)
                if is_varying:
                    for emo in CFG.emotion_order:
                        if emo == CFG.baseline_condition:
                            continue

                        effect_key = (
                            f"{param_family}_C(emotion, "
                            f"Treatment('"
                            f"{CFG.baseline_condition}'))"
                            f"[T.{emo}]"
                        )

                        if param_family == 'v':
                            eff = stats.norm.rvs(
                                loc=0, scale=0.5
                            )
                        elif param_family in ['a', 'z']:
                            eff = stats.norm.rvs(
                                loc=0, scale=0.1
                            )
                        elif param_family == 't':
                            eff = stats.norm.rvs(
                                loc=0, scale=0.05
                            )
                        else:
                            eff = 0.0

                        group_truth[effect_key] = float(eff)

        # Generate subject-level parameters with hierarchical
        # variability. Subject-level SD (LP scale) per family.
        subject_truths = {}
        subjects = self.empirical_df['subj_idx'].unique()
        # Subject-level SD for the hierarchical deviations, on each
        # parameter's generative scale (drift for v; physical for a, t;
        # logit for z). In posterior mode these are taken from the fitted
        # group-level SDs (_Intercept_std) so synthetic data reproduce the
        # empirical between-subject spread; in prior mode ecologically
        # plausible values matched to the fitted scale are used.
        subj_sd_default = {'v': 1.4, 'a': 0.45, 't': 0.19, 'z': 0.25}
        if mode == 'posterior':
            subj_sd = {}
            for _p in ['v', 'a', 't', 'z']:
                _k = f"{_p}_Intercept_std"
                subj_sd[_p] = (float(summary.loc[_k, 'mean'])
                               if _k in summary.index
                               else subj_sd_default[_p])
        else:
            subj_sd = dict(subj_sd_default)

        for subj in subjects:
            # Draw ONE subject-level deviation per parameter, shared across
            # all emotion conditions, realising a per-subject random
            # intercept (the hierarchical structure assumed by
            # group_only_regressors=True). subject_truths[subj][baseline]
            # therefore defines a well-posed per-subject intercept truth
            # for the subject-level recovery diagnostic.
            subj_dev = {
                p: np.random.normal(0, subj_sd[p])
                for p in ['v', 'a', 't', 'z']
            }
            subj_params = {}
            for emo in CFG.emotion_order:
                cell_params = {}
                for p in ['v', 'a', 't', 'z']:
                    # Group-level LP for this condition
                    lp_val = group_truth.get(
                        f"{p}_Intercept", 0.0
                    )
                    if emo != CFG.baseline_condition:
                        effect_key = (
                            f"{p}_C(emotion, "
                            f"Treatment('"
                            f"{CFG.baseline_condition}'))"
                            f"[T.{emo}]"
                        )
                        if effect_key not in group_truth:
                            effect_key = (
                                f"{p}_C(emotion)[T.{emo}]"
                            )
                        lp_val += group_truth.get(
                            effect_key, 0.0
                        )

                    # Add the subject's shared random-intercept deviation
                    # (LP scale), constant across emotion conditions
                    subj_lp = lp_val + subj_dev[p]

                    # Inverse link: LP → physical (with clipping)
                    cell_params[p] = self._apply_link_and_clip(
                        p, subj_lp
                    )

                subj_params[emo] = cell_params
            subject_truths[subj] = subj_params

        return group_truth, subject_truths

    # -----------------------------------------------------------------
    # SYNTHETIC DATA GENERATION
    # -----------------------------------------------------------------
    def generate_synthetic_data(
        self,
        subject_truths: Dict[str, Dict[str, Dict[str, float]]],
        iteration: int,
        mode: str
    ) -> pd.DataFrame:
        """
        Constructs synthetic data using HDDM's internal forward
        simulation. Each subject x condition cell uses that
        subject's unique parameters (with hierarchical variability).
        """
        trial_rows = []

        for (subj, emo), group in self.empirical_df.groupby(
            ['subj_idx', 'emotion']
        ):
            n_trials = len(group)
            phys_params = subject_truths[subj][emo]

            # Forward simulation via hddm.generate
            sim_res = gen_rand_data(phys_params, size=n_trials)
            sim_df = (
                sim_res[0]
                if isinstance(sim_res, tuple)
                else sim_res
            )

            row_data = pd.DataFrame({
                'subj_idx': subj,
                'emotion': emo,
                'rt': sim_df['rt'].values,
                'response': sim_df['response'].values
            })

            # Tag true parameters for this cell
            for p in ['v', 'a', 't', 'z']:
                row_data[f'{p}_true'] = phys_params[p]

            trial_rows.append(row_data)

        return pd.concat(trial_rows, ignore_index=True)

    # -----------------------------------------------------------------
    # RECOVERY REFIT WITH EXPLICIT ARTIFACT PERSISTENCE
    # -----------------------------------------------------------------
    def execute_recovery_refit(
        self,
        synth_df: pd.DataFrame,
        iteration: int,
        mode: str
    ) -> Tuple[Optional[pd.DataFrame], bool, float,
               Dict[str, float]]:
        """
        Executes structural refitting of synthetic data with
        stratified convergence diagnostics and adaptive sampling.

        Phase 4 artifact persistence: after all adaptive
        cycles complete, explicitly saves accumulated InferenceData
        to .nc via model.to_infdata().to_netcdf(). This mirrors
        Step 2b's Phase 4 pattern and ensures disk artifacts
        reflect the FULL accumulated posterior, not just the last
        adaptive extension.

        Returns:
          summary_df: ArviZ summary (or None on failure).
          converged:  Whether all stratified criteria are satisfied.
          max_rhat:   Maximum R-hat among focal parameters.
          metrics:    Full diagnostic metrics dictionary.
        """
        synth_df = synth_df.copy()
        synth_df['subj_idx'] = synth_df['subj_idx'].astype(str)

        regressors = _build_regressors(
            self.target_model, CFG.baseline_condition
        )

        # Match Step 2b: all 4 parameters always included
        full_include = ['v', 'a', 't', 'z']

        model = hddm.HDDMRegressor(
            synth_df,
            regressors,
            include=full_include,
            is_group_model=True,
            group_only_regressors=CFG.group_only_regressors,
            keep_regressor_trace=CFG.keep_regressor_trace,
            informative=CFG.use_informative_priors,
            p_outlier=CFG.p_outlier
        )

        # Unique save path per iteration to avoid file conflicts
        save_prefix = str(
            PATHS['recovery']
            / f"recovery_{self.target_model}_{mode}_iter{iteration}"
        )

        # Phase 1: Initial sampling (runs to completion)
        infdata = model.sample(
            self.recovery_samples,
            burn=self.recovery_burn,
            thin=CFG.default_thin,
            chains=self.recovery_chains,
            return_infdata=True,
            loglike=False,
            ppc=False,
            save_name=save_prefix
        )

        # Phase 2: Stratified convergence evaluation
        converged, metrics, summary_df = (
            _evaluate_recovery_convergence(infdata, self.logger)
        )

        # Phase 3: Adaptive sampling loop
        total_samples = self.recovery_samples
        max_samples = self.recovery_samples * 2
        cycle = 0
        prev_metrics = None
        rhat_extension_count = 0

        while (not converged
               and cycle < RECOVERY_MAX_ADAPTIVE_CYCLES):
            cycle += 1

            # Compute extension size from focal ESS deficits
            deficit_ratios = []

            if metrics["f_bulk"] < RECOVERY_ESS_BULK_FOCAL:
                deficit_ratios.append(
                    RECOVERY_ESS_BULK_FOCAL
                    / max(metrics["f_bulk"], 1.0)
                )
            if metrics["f_tail"] < RECOVERY_ESS_TAIL_FOCAL:
                deficit_ratios.append(
                    RECOVERY_ESS_TAIL_FOCAL
                    / max(metrics["f_tail"], 1.0)
                )

            if deficit_ratios:
                # ESS-driven: scale by worst focal ESS deficit
                deficit_ratio = max(deficit_ratios)
                added_samples = max(
                    500,
                    int(total_samples
                        * (deficit_ratio - 1) * 1.25)
                )
            else:
                # R-hat-driven: moderate fixed extension
                deficit_ratio = 1.3
                added_samples = max(
                    500,
                    int(self.recovery_samples * 0.30)
                )
                rhat_extension_count += 1

            # R-hat plateau detection
            if (prev_metrics is not None
                    and rhat_extension_count > 0):
                prev_worst_rhat = max(
                    prev_metrics["f_rhat"],
                    prev_metrics["n_rhat"]
                )
                curr_worst_rhat = max(
                    metrics["f_rhat"],
                    metrics["n_rhat"]
                )
                rhat_improvement = (
                    prev_worst_rhat - curr_worst_rhat
                )

                self.logger.info(
                    f"    R-hat change: "
                    f"{prev_worst_rhat:.4f} -> "
                    f"{curr_worst_rhat:.4f} "
                    f"(improvement="
                    f"{rhat_improvement:+.4f})"
                )

                if rhat_improvement < RECOVERY_RHAT_IMPROVEMENT_MIN:
                    self.logger.warning(
                        f"    R-hat improvement insufficient "
                        f"(<{RECOVERY_RHAT_IMPROVEMENT_MIN}) "
                        f"after R-hat extension round "
                        f"{rhat_extension_count}. "
                        f"Terminating adaptive extension."
                    )
                    break

            if rhat_extension_count >= RECOVERY_MAX_RHAT_EXTENSIONS:
                self.logger.info(
                    f"    R-hat extension limit reached "
                    f"({RECOVERY_MAX_RHAT_EXTENSIONS} rounds). "
                    f"Terminating adaptive extension."
                )
                break

            prev_metrics = metrics.copy()

            # Sample ceiling enforcement
            if total_samples + added_samples > max_samples:
                added_samples = max_samples - total_samples
                if added_samples <= 0:
                    self.logger.warning(
                        f"    Sample ceiling "
                        f"({max_samples}) reached. "
                        f"Terminating adaptive cycles."
                    )
                    break

            self.logger.info(
                f"    [Adaptive Cycle "
                f"{cycle}/{RECOVERY_MAX_ADAPTIVE_CYCLES}] "
                f"Appending {added_samples} samples "
                f"(Deficit Ratio: {deficit_ratio:.2f})..."
            )

            model.sample(
                added_samples,
                burn=0,
                thin=CFG.default_thin,
                chains=self.recovery_chains,
                return_infdata=True,
                loglike=False,
                ppc=False,
                save_name=save_prefix
            )

            total_samples += added_samples

            # Regenerate InferenceData from accumulated trace
            try:
                infdata = model.to_infdata(
                    loglike=False, ppc=False
                )
            except Exception as e:
                self.logger.warning(
                    f"    model.to_infdata() failed: {e}. "
                    f"Diagnostics may reflect stale trace."
                )

            # Re-evaluate convergence
            converged, metrics, summary_df = (
                _evaluate_recovery_convergence(
                    infdata, self.logger
                )
            )

        # Log final adaptive outcome
        if cycle > 0:
            self.logger.info(
                f"    Adaptive sampling complete: "
                f"{total_samples} total samples, "
                f"{'CONVERGED' if converged else 'NOT CONVERGED'}"
            )

        # =============================================================
        # PHASE 4: EXPLICIT ARTIFACT PERSISTENCE
        # Mirrors Step 2b Phase 4. Ensures .nc on disk reflects
        # the FULL accumulated posterior (initial + all extensions),
        # not just the last model.sample() auto-save.
        # =============================================================
        try:
            infdata_final = model.to_infdata(
                loglike=False, ppc=False
            )
            nc_path = f"{save_prefix}.nc"
            infdata_final.to_netcdf(nc_path)
            self.logger.info(
                f"    Accumulated InferenceData saved: "
                f"{Path(nc_path).name} "
                f"({total_samples} total samples)"
            )
        except Exception as e:
            self.logger.warning(
                f"    Phase 4 .nc persistence failed: {e}. "
                f"On-disk .nc may reflect only last extension."
            )

        try:
            hddm_path = f"{save_prefix}.hddm"
            model.save(hddm_path)
            self.logger.info(
                f"    Model object saved: "
                f"{Path(hddm_path).name}"
            )
        except Exception as e:
            self.logger.warning(
                f"    Phase 4 .hddm persistence failed: {e}."
            )

        del model, infdata
        gc.collect()

        return (summary_df, converged,
                float(metrics["f_rhat"]), metrics)

    # -----------------------------------------------------------------
    # SINGLE-ITERATION RECOVERY METRICS
    # With scale-aligned comparison (physical scale)
    # -----------------------------------------------------------------
    def compute_iteration_metrics(
        self,
        group_truth: Dict[str, float],
        summary_df: pd.DataFrame,
        iteration: int,
        mode: str,
        converged: bool,
        max_rhat: float,
        metrics: Dict[str, float]
    ) -> List[Dict]:
        """
        Computes per-parameter recovery metrics for a single
        iteration: Bias, Squared Error, Coverage, HDI Width.

        Ground-truth and recovered values are compared on the physical
        parameter scale. For v/a/t the generative and reported scales
        coincide (identity link), so no conversion is applied; the z
        intercept is mapped from the logit scale back to a probability
        with the inverse link before comparison. Biases are thus in
        physical units (seconds for t, threshold units for a, probability
        for z).
        """
        records = []

        for param, gt_lp in group_truth.items():
            if param not in summary_df.index:
                continue

            rec_row = summary_df.loc[param]
            rec_mean = float(rec_row['mean'])
            hdi_low = float(rec_row['hdi_3%'])
            hdi_high = float(rec_row['hdi_97%'])

            family = (
                param.split('_')[0] if '_' in param else 'other'
            )
            is_intercept = 'Intercept' in param

            # Convert GT to comparison scale.
            # Intercepts: LP → physical (to match refit summary)
            # Contrasts & v: already in same scale, no conversion
            gt_compare = self._apply_link_and_clip(
                family, gt_lp
            ) if (is_intercept and family == 'z') else gt_lp

            bias = rec_mean - gt_compare
            is_covered = hdi_low <= gt_compare <= hdi_high

            param_type = (
                'Intercept'
                if is_intercept
                else 'ConditionEffect'
            )

            records.append({
                'pipeline_hash': self.active_hash,
                'mode': mode,
                'iteration': iteration,
                'Parameter': param,
                'Family': family,
                'Type': param_type,
                'Ground_Truth': gt_compare,
                'Recovered_Mean': rec_mean,
                'Bias': bias,
                'Abs_Bias': np.abs(bias),
                'Sq_Error': bias ** 2,
                'HDI_3': hdi_low,
                'HDI_97': hdi_high,
                'HDI_Width': hdi_high - hdi_low,
                'Coverage': int(is_covered),
                'Refit_Converged': converged,
                'Refit_Max_Rhat': max_rhat,
                'Refit_F_ESS_Bulk': metrics.get(
                    'f_bulk', np.nan
                ),
                'Refit_F_ESS_Tail': metrics.get(
                    'f_tail', np.nan
                ),
                'Refit_N_Rhat': metrics.get(
                    'n_rhat', np.nan
                )
            })

        return records

    # -----------------------------------------------------------------
    # SUBJECT-LEVEL INTERCEPT RECOVERY
    # -----------------------------------------------------------------
    def compute_subject_intercept_recovery(
        self,
        subject_truths: Dict[str, Dict[str, Dict[str, float]]],
        summary_df: pd.DataFrame,
        iteration: int,
        mode: str,
        converged: bool
    ) -> List[Dict]:
        """
        Evaluates recovery of the subject-level random intercepts that the
        hierarchical model estimates under group_only_regressors=True.

        Ground truth for subject s and family p is the physical-scale
        baseline-condition value subject_truths[s][baseline][p]; given the
        shared-deviation generation this is that subject's random intercept.
        The recovered counterpart is the posterior mean of the subject node
        ('{p}_Intercept_subj.{s}' for regression families, '{p}_subj.{s}'
        otherwise).

        Comparison is on the physical scale: exact for v/a/t (identity
        link); for z the posterior-summary convention used for the
        group-level z intercept is reused (no additional transform). The
        primary diagnostic is the across-subject correlation, computed in
        export_subject_recovery_summary, rather than interval coverage.
        """
        records = []
        baseline = CFG.baseline_condition

        for subj, by_emo in subject_truths.items():
            if baseline not in by_emo:
                continue
            for p in ['v', 'a', 't', 'z']:
                true_phys = by_emo[baseline].get(p)
                if true_phys is None:
                    continue

                # Resolve the subject node name (regression vs plain param)
                candidates = [
                    f"{p}_Intercept_subj.{subj}",
                    f"{p}_subj.{subj}",
                ]
                node = next(
                    (c for c in candidates if c in summary_df.index),
                    None
                )
                if node is None:
                    continue

                records.append({
                    'pipeline_hash': self.active_hash,
                    'mode': mode,
                    'iteration': iteration,
                    'subj_idx': subj,
                    'Family': p,
                    'True_Subject_Intercept': float(true_phys),
                    'Recovered_Subject_Mean': float(
                        summary_df.loc[node, 'mean']
                    ),
                    'Refit_Converged': converged,
                })

        return records

    # -----------------------------------------------------------------
    # INCREMENTAL PERSISTENCE
    # -----------------------------------------------------------------
    @staticmethod
    def _append_records_to_csv(
        records: List[Dict], csv_path: Path
    ):
        """
        Atomically appends records to a CSV file.
        Creates the file with header on first write.
        """
        df_new = pd.DataFrame(records)
        if csv_path.exists():
            df_new.to_csv(
                csv_path, mode='a', header=False, index=False
            )
        else:
            df_new.to_csv(csv_path, index=False)

    # -----------------------------------------------------------------
    # PUBLICATION-READY AGGREGATE VISUALIZATION
    # -----------------------------------------------------------------
    def render_aggregate_diagnostics(self, mode: str):
        """
        Constructs visualizations from ALL converged iterations:
        1. Identity scatter (Ground Truth vs Recovered Mean)
           with marginal distributions.
        2. Coverage rate bar chart per parameter family.
        3. Bias distribution violin per parameter family.
        """
        csv_path = (
            PATHS['recovery']
            / f"recovery_all_iterations_"
              f"{self.target_model}_{mode}.csv"
        )
        if not csv_path.exists():
            self.logger.warning(
                f"  [{mode}] No iteration data found. "
                f"Visualization skipped."
            )
            return

        df = pd.read_csv(csv_path)

        # Filter to converged iterations only
        if 'Refit_Converged' in df.columns:
            n_total = df['iteration'].nunique()
            df_conv = df[df['Refit_Converged'] == True]
            n_conv = df_conv['iteration'].nunique()
            self.logger.info(
                f"  [{mode}] Using {n_conv}/{n_total} "
                f"converged iterations for visualization."
            )
        else:
            df_conv = df

        if df_conv.empty:
            self.logger.warning(
                f"  [{mode}] No converged iterations. "
                f"Visualization skipped."
            )
            return

        title_prefix = (
            "Posterior-Anchored"
            if mode == 'posterior'
            else "Prior-Predictive"
        )

        pal = {
            'v': OKABE_ITO[0], 'a': OKABE_ITO[1],
            't': OKABE_ITO[2], 'z': OKABE_ITO[6]
        }

        # --- Plot 1: Identity Scatter (aggregated) ---
        families = df_conv['Family'].unique()
        n_fam = max(1, len(families))
        fig, axes = plt.subplots(
            1, n_fam, figsize=(5 * n_fam, 5), squeeze=False
        )
        axes = axes.flatten()

        for ax, fam in zip(axes, families):
            fam_df = df_conv[df_conv['Family'] == fam]
            color = pal.get(fam, '#333333')

            ax.scatter(
                fam_df['Ground_Truth'],
                fam_df['Recovered_Mean'],
                alpha=0.3, s=20, color=color, edgecolors='none'
            )

            all_vals = pd.concat([
                fam_df['Ground_Truth'],
                fam_df['Recovered_Mean']
            ])
            margin = max(
                0.05, (all_vals.max() - all_vals.min()) * 0.1
            )
            lims = [
                all_vals.min() - margin,
                all_vals.max() + margin
            ]
            ax.plot(lims, lims, 'k--', alpha=0.5, zorder=0)
            ax.set_xlim(lims)
            ax.set_ylim(lims)

            if len(fam_df) >= 3:
                r_val, _ = stats.pearsonr(
                    fam_df['Ground_Truth'],
                    fam_df['Recovered_Mean']
                )
                rmse = float(
                    np.sqrt(fam_df['Sq_Error'].mean())
                )
                ax.set_title(
                    f"{fam.upper()} (r={r_val:.2f}, "
                    f"RMSE={rmse:.3f})",
                    fontweight='bold'
                )
            else:
                ax.set_title(
                    f"{fam.upper()}", fontweight='bold'
                )

            ax.set_xlabel("Ground Truth")
            ax.set_ylabel("Recovered Mean")

        plt.suptitle(
            f"{title_prefix} Recovery (N="
            f"{df_conv['iteration'].nunique()} iterations): "
            f"{self.target_model.upper()}",
            y=1.05, fontweight='bold'
        )
        sns.despine(trim=True)
        plt.tight_layout()
        plt.savefig(
            PATHS['figures_supp']
            / f"recovery_scatter_aggregate_"
              f"{self.target_model}_{mode}.pdf",
            dpi=300, bbox_inches='tight'
        )
        plt.close(fig)

        # --- Plot 2: Coverage Rate per Family ---
        coverage_agg = df_conv.groupby('Family').agg(
            Coverage_Rate=('Coverage', 'mean'),
            N=('Coverage', 'count')
        ).reset_index()

        fig, ax = plt.subplots(figsize=(6, 4))
        bars = ax.bar(
            coverage_agg['Family'].str.upper(),
            coverage_agg['Coverage_Rate'],
            color=[
                pal.get(f, '#999999')
                for f in coverage_agg['Family']
            ],
            alpha=0.8, edgecolor='black', linewidth=0.5
        )
        ax.axhline(
            0.94, color='red', linestyle='--', linewidth=1.5,
            label='Nominal 94% Coverage'
        )
        ax.set_ylabel('Empirical Coverage Rate')
        ax.set_title(
            f"{title_prefix}: 94% HDI Coverage",
            fontweight='bold'
        )
        ax.set_ylim(0, 1.05)
        ax.legend(fontsize=10)

        # Annotate percentages
        for bar, rate in zip(
            bars, coverage_agg['Coverage_Rate']
        ):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.02,
                f"{rate:.0%}",
                ha='center', fontsize=11, fontweight='bold'
            )

        sns.despine(trim=True)
        plt.tight_layout()
        plt.savefig(
            PATHS['figures_supp']
            / f"recovery_coverage_bar_"
              f"{self.target_model}_{mode}.pdf",
            dpi=300, bbox_inches='tight'
        )
        plt.close(fig)

        # --- Plot 3: Bias Distribution per Family ---
        fig, ax = plt.subplots(figsize=(7, 5))
        sns.violinplot(
            x='Family', y='Bias', data=df_conv,
            palette=pal, inner='box', cut=0, ax=ax,
            alpha=0.6, order=sorted(families)
        )
        ax.axhline(
            0, color='black', linestyle='--',
            linewidth=1.5, alpha=0.7
        )
        ax.set_title(
            f"{title_prefix}: Bias Distribution",
            fontweight='bold'
        )
        ax.set_ylabel('Bias (Recovered - Truth)')
        ax.set_xlabel('Parameter Family')
        sns.despine(trim=True)
        plt.tight_layout()
        plt.savefig(
            PATHS['figures_supp']
            / f"recovery_bias_violin_"
              f"{self.target_model}_{mode}.pdf",
            dpi=300, bbox_inches='tight'
        )
        plt.close(fig)

        self.logger.info(
            f"  [{mode}] Aggregate visualizations exported."
        )

    # -----------------------------------------------------------------
    # AGGREGATE SUMMARY TABLE
    # -----------------------------------------------------------------
    def export_aggregate_summary(self, mode: str):
        """
        Computes and exports aggregate recovery metrics across
        all converged iterations.
        """
        csv_path = (
            PATHS['recovery']
            / f"recovery_all_iterations_"
              f"{self.target_model}_{mode}.csv"
        )
        if not csv_path.exists():
            return

        df = pd.read_csv(csv_path)
        if 'Refit_Converged' in df.columns:
            df = df[df['Refit_Converged'] == True]

        if df.empty:
            return

        agg = df.groupby(['Family', 'Type']).agg(
            N_Iterations=('iteration', 'nunique'),
            N_Params=('Parameter', 'nunique'),
            RMSE=('Sq_Error', lambda x: np.sqrt(x.mean())),
            Mean_Abs_Bias=('Abs_Bias', 'mean'),
            Mean_HDI_Width=('HDI_Width', 'mean'),
            Coverage_Rate=('Coverage', 'mean')
        ).reset_index()

        agg_path = (
            PATHS['tables_supp']
            / f"recovery_aggregate_summary_"
              f"{self.target_model}_{mode}.csv"
        )
        agg.to_csv(agg_path, index=False)

        # Global correlation
        if len(df) >= 3:
            r_val, p_val = stats.pearsonr(
                df['Ground_Truth'], df['Recovered_Mean']
            )
            rmse_global = float(
                np.sqrt(df['Sq_Error'].mean())
            )
            self.logger.info(
                f"  [{mode}] Global: r={r_val:.3f} "
                f"(p={p_val:.2e}), RMSE={rmse_global:.4f}"
            )

        self.logger.info(f"\n{agg.to_string(index=False)}")

    # -----------------------------------------------------------------
    # SUBJECT-LEVEL INTERCEPT RECOVERY SUMMARY + FIGURE
    # -----------------------------------------------------------------
    def export_subject_recovery_summary(self, mode: str):
        """
        Aggregates subject-level intercept recovery across all converged
        iterations: per-family across-subject Pearson correlation, OLS
        slope, and RMSE between the true and recovered subject intercepts
        (pooled over subjects and iterations). Emits a summary table and an
        identity scatter (true vs recovered) per parameter family.

        The across-subject correlation indexes whether between-subject
        differences in the baseline parameters are recovered — the
        complement to the group-level focal-parameter diagnostics.
        """
        subj_csv = (
            PATHS['recovery']
            / f"recovery_subject_pairs_"
              f"{self.target_model}_{mode}.csv"
        )
        if not subj_csv.exists():
            self.logger.warning(
                f"  [{mode}] No subject-level recovery data. "
                f"Subject summary skipped."
            )
            return

        df = pd.read_csv(subj_csv)
        if 'Refit_Converged' in df.columns:
            df = df[df['Refit_Converged'] == True]
        if df.empty:
            self.logger.warning(
                f"  [{mode}] No converged subject-level data."
            )
            return

        rows = []
        for fam, g in df.groupby('Family'):
            t = g['True_Subject_Intercept'].values
            r = g['Recovered_Subject_Mean'].values
            if len(t) < 3:
                continue
            rows.append({
                'Family': fam,
                'N_Pairs': int(len(g)),
                'N_Subjects': int(g['subj_idx'].nunique()),
                'N_Iterations': int(g['iteration'].nunique()),
                'Pearson_r': float(np.corrcoef(t, r)[0, 1]),
                'OLS_Slope': float(np.polyfit(t, r, 1)[0]),
                'RMSE': float(np.sqrt(np.mean((r - t) ** 2))),
            })
        out = pd.DataFrame(rows)
        out_path = (
            PATHS['tables_supp']
            / f"recovery_subject_intercepts_"
              f"{self.target_model}_{mode}.csv"
        )
        out.to_csv(out_path, index=False)
        self.logger.info(
            f"  [{mode}] Subject-level intercept recovery:\n"
            f"{out.to_string(index=False)}"
        )

        # Identity scatter per family (true vs recovered subject intercept)
        title_prefix = (
            "Posterior-Anchored"
            if mode == 'posterior'
            else "Prior-Predictive"
        )
        pal = {
            'v': OKABE_ITO[0], 'a': OKABE_ITO[1],
            't': OKABE_ITO[2], 'z': OKABE_ITO[6]
        }
        families = sorted(df['Family'].unique())
        n_fam = max(1, len(families))
        fig, axes = plt.subplots(
            1, n_fam, figsize=(5 * n_fam, 5), squeeze=False
        )
        axes = axes.flatten()
        for ax, fam in zip(axes, families):
            fam_df = df[df['Family'] == fam]
            color = pal.get(fam, '#333333')
            ax.scatter(
                fam_df['True_Subject_Intercept'],
                fam_df['Recovered_Subject_Mean'],
                alpha=0.3, s=20, color=color, edgecolors='none'
            )
            all_vals = pd.concat([
                fam_df['True_Subject_Intercept'],
                fam_df['Recovered_Subject_Mean']
            ])
            margin = max(
                0.05, (all_vals.max() - all_vals.min()) * 0.1
            )
            lims = [all_vals.min() - margin, all_vals.max() + margin]
            ax.plot(lims, lims, 'k--', alpha=0.5, zorder=0)
            ax.set_xlim(lims)
            ax.set_ylim(lims)
            if len(fam_df) >= 3:
                r_val = float(np.corrcoef(
                    fam_df['True_Subject_Intercept'],
                    fam_df['Recovered_Subject_Mean']
                )[0, 1])
                ax.set_title(
                    f"{fam.upper()} (r={r_val:.2f})",
                    fontweight='bold'
                )
            else:
                ax.set_title(f"{fam.upper()}", fontweight='bold')
            ax.set_xlabel("True Subject Intercept")
            ax.set_ylabel("Recovered Subject Mean")
        plt.suptitle(
            f"{title_prefix} Subject-Level Intercept Recovery: "
            f"{self.target_model.upper()}",
            y=1.05, fontweight='bold'
        )
        sns.despine(trim=True)
        plt.tight_layout()
        plt.savefig(
            PATHS['figures_supp']
            / f"recovery_subject_scatter_"
              f"{self.target_model}_{mode}.pdf",
            dpi=300, bbox_inches='tight'
        )
        plt.close(fig)
        self.logger.info(
            f"  [{mode}] Subject-level recovery figure exported."
        )

    # -----------------------------------------------------------------
    # MASTER ITERATION LOOP
    # -----------------------------------------------------------------
    def run_mode(self, mode: str):
        """
        Executes N iterations of recovery for a given mode.

        Safety mechanisms:
          - Memory headroom check before each iteration
          - Consecutive failure guard (MemoryError, numerical)
          - Incremental CSV persistence after each iteration
          - Checkpoint resume on restart
        """
        self.logger.info(f"\n{'*'*60}")
        self.logger.info(
            f"RECOVERY MODE: {mode.upper()} "
            f"({self.n_iterations} iterations)"
        )
        self.logger.info(f"{'*'*60}")

        csv_path = (
            PATHS['recovery']
            / f"recovery_all_iterations_"
              f"{self.target_model}_{mode}.csv"
        )
        subj_csv_path = (
            PATHS['recovery']
            / f"recovery_subject_pairs_"
              f"{self.target_model}_{mode}.csv"
        )

        # Check for existing results (resume support)
        # Robust CSV resume: handles corrupted files
        # from mid-write crashes (e.g., power loss during
        # _append_records_to_csv). If the CSV is unreadable,
        # it is backed up and a fresh run begins.
        completed_iters = set()
        if csv_path.exists():
            try:
                existing = pd.read_csv(csv_path)
                if 'iteration' in existing.columns:
                    completed_iters = set(
                        existing['iteration'].unique()
                    )
                    self.logger.info(
                        f"  Resume: {len(completed_iters)} "
                        f"iterations already completed."
                    )
            except Exception as e:
                backup = csv_path.with_suffix('.csv.bak')
                csv_path.rename(backup)
                self.logger.warning(
                    f"  CSV corrupted ({e}). "
                    f"Backed up to {backup.name}. "
                    f"Starting fresh."
                )

        consecutive_failures = 0

        for iteration in range(self.n_iterations):
            if iteration in completed_iters:
                continue

            self.logger.info(
                f"\n  --- Iteration {iteration + 1}/"
                f"{self.n_iterations} (mode={mode}) ---"
            )

            # Memory check
            try:
                check_memory_headroom(logger=self.logger)
            except MemoryError:
                self.logger.error(
                    "  Memory insufficient. "
                    "Terminating recovery."
                )
                break

            try:
                t_start = time.time()

                # Phase 1: Generate ground truth + synthetic data
                group_truth, subject_truths = (
                    self.establish_ground_truth(mode, iteration)
                )

                synth_df = self.generate_synthetic_data(
                    subject_truths, iteration, mode
                )

                # Phase 2: Refit with stratified convergence
                # and adaptive sampling (runs to completion)
                summary_df, converged, max_rhat, metrics = (
                    self.execute_recovery_refit(
                        synth_df, iteration, mode
                    )
                )

                # Phase 3: Compute metrics
                if summary_df is not None:
                    records = self.compute_iteration_metrics(
                        group_truth, summary_df,
                        iteration, mode,
                        converged, max_rhat,
                        metrics
                    )

                    # Incremental persistence
                    if records:
                        self._append_records_to_csv(
                            records, csv_path
                        )

                    # Subject-level intercept recovery (incremental)
                    subj_records = (
                        self.compute_subject_intercept_recovery(
                            subject_truths, summary_df,
                            iteration, mode, converged
                        )
                    )
                    if subj_records:
                        self._append_records_to_csv(
                            subj_records, subj_csv_path
                        )

                elapsed = (time.time() - t_start) / 60
                status = (
                    'CONVERGED' if converged
                    else f'NOT CONVERGED (R-hat={max_rhat:.3f})'
                )
                self.logger.info(
                    f"  Iteration {iteration + 1}: {status} "
                    f"({elapsed:.1f} min)"
                )

                consecutive_failures = 0

            except Exception as e:
                consecutive_failures += 1
                self.logger.warning(
                    f"  Iteration {iteration + 1}: EXCEPTION "
                    f"({type(e).__name__}: {e}). "
                    f"Consecutive failures: "
                    f"{consecutive_failures}/"
                    f"{MAX_CONSECUTIVE_FAILURES}"
                )
                self.logger.warning(traceback.format_exc())

            finally:
                gc.collect()

            # Consecutive failure guard
            if consecutive_failures >= MAX_CONSECUTIVE_FAILURES:
                self.logger.error(
                    f"  {MAX_CONSECUTIVE_FAILURES} consecutive "
                    f"failures reached. Terminating {mode} mode."
                )
                break

    # -----------------------------------------------------------------
    # FULL PIPELINE
    # -----------------------------------------------------------------
    def run_full_pipeline(self):
        """
        Orchestrates both posterior-anchored and prior-predictive
        recovery protocols with aggregate reporting for the single
        specified target model.
        """
        self.logger.info(f"\n{'#'*70}")
        self.logger.info(
            f"RECOVERY TARGET: [{self.target_model.upper()}]"
        )
        self.logger.info(f"{'#'*70}")

        for mode in ['posterior', 'prior']:
            self.run_mode(mode)
            self.export_aggregate_summary(mode)
            self.render_aggregate_diagnostics(mode)
            self.export_subject_recovery_summary(mode)

        self.logger.info(f"\n{'='*70}")
        self.logger.info(
            "PARAMETER RECOVERY COMPLETED"
        )
        self.logger.info(
            f"Model: [{self.target_model.upper()}]"
        )
        self.logger.info(f"{'='*70}")


# =============================================================================
# PIPELINE EXECUTION ENTRY POINT
# =============================================================================
# Usage:
#   engine = ParameterRecoveryEngine(target_model='va')
#   engine.run_full_pipeline()
#
# Extension (after initial 30 complete, add 10 more):
#   engine = ParameterRecoveryEngine(target_model='va', n_iterations=40)
#   engine.run_full_pipeline()
#   # Automatically skips iterations 0-29, runs only 30-39.
#
# Before FIRST run, ensure that:
#   1. Step 2a has been executed (hddm_config.py exists)
#   2. Step 2b has fitted the target model (hddm_va.nc exists)
#   3. Step 4 has produced the audit trail (final_model_selection_audit.csv)
#   4. OLD recovery CSV files have been DELETED or RENAMED to avoid
#      contamination from the pre-fix scale-mismatched results:
#        rm recovery_all_iterations_va_posterior.csv
#        rm recovery_all_iterations_va_prior.csv
#        rm recovery_subject_pairs_va_posterior.csv
#        rm recovery_subject_pairs_va_prior.csv
# =============================================================================

# ============================================================
# PARAMETER RECOVERY -- execution record (which models, and why)
# ------------------------------------------------------------
# Step 4's DIC funnel auto-selected `vaz` as winner, but `va` was
# the a-priori reported model (parsimony + consistency with E1).
# Recovery for `va` was therefore run FIRST and is COMPLETE
# (see recovery/ and tables_supp/recovery_aggregate_summary_va_*).
#
# To adjudicate `va` vs `vaz` rigorously we now ALSO run recovery
# for `vaz`, to test whether its z-by-emotion contrasts are
# identifiable/recoverable (vaz shows a v-z posterior correlation
# ~ -0.6). If vaz's z ConditionEffect fails to recover, that is the
# decisive evidence for reporting `va` over the DIC-preferred `vaz`.
# ============================================================

# (1) va recovery -- ALREADY COMPLETED; kept here for the record.
#     Re-enable only to regenerate va recovery (resume-safe: skips
#     iterations already on disk).
# engine_va = ParameterRecoveryEngine(target_model='va')
# engine_va.run_full_pipeline()

# (2) vaz recovery -- identifiability check for the DIC winner.
try:
    engine = ParameterRecoveryEngine(target_model='vaz')
    engine.run_full_pipeline()
except Exception as e:
    print(f"\nCRITICAL FAILURE: {e}")
    import traceback
    traceback.print_exc()